In [2]:
# ===================================================================
# === CELL A1: SETUP & CONFIG (GDRIVE LEGACY)
# Target: Notebook 0111a
# ===================================================================
# --- [1] Standard Python Libraries ---
import os
import re
import sqlite3
import time
import warnings

# --- [2] Data Manipulation & UI ---
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display  # <--- [FIX] Required for 'display()' calls in Codespaces

# --- [3] Google Cloud & GSheets Integration ---
import gspread
from google.oauth2 import service_account           # For 'service_account.Credentials'
from googleapiclient.discovery import build         # For 'build('drive', 'v3', ...)'

# --- [4] SQL Alchemy & Database Engines ---
from sqlalchemy import create_engine, text, inspect # For 'inspect(db_engine)' and views

# --- [5] Environment Configuration ---
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True) # Specific to your Phase 5 cleaning

print("✅ [A1] Master dependencies loaded. Parity Rule: Maintained.")

# --- [1] Configuración de Rutas y Secretos ---
SERVICE_ACCOUNT_FILE = '/workspaces/pienza/secrets/service-account.json'
project_root = '/workspaces/pienza/data/big_bang'
os.makedirs(project_root, exist_ok=True)

# La DB vive localmente en el workspace para máxima velocidad
db_file_path = os.path.join(project_root, 'pienzaba.db')

# --- [2] Autenticación con GSheets ---
SCOPES = ['https://www.googleapis.com/auth/spreadsheets', 'https://www.googleapis.com/auth/drive']
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
gc = gspread.authorize(creds)

print("✅ [A1] Bibliotecas importadas y Auth de GDrive lista.")
print(f"✅ [A1] El Big Bang Legacy nacerá en: {db_file_path}")

✅ [A1] Master dependencies loaded. Parity Rule: Maintained.
✅ [A1] Bibliotecas importadas y Auth de GDrive lista.
✅ [A1] El Big Bang Legacy nacerá en: /workspaces/pienza/data/big_bang/pienzaba.db


In [3]:
# ===================================================================
# === CELL A2: PHASE 1 - BUILD & POPULATE DATABASE (GDRIVE LEGACY)
# Target: Notebook 0111a
# ===================================================================
from googleapiclient.discovery import build

print(f"--- PHASE 1 STARTED: Building Legacy database '{db_file_path}' ---")

# --- Step 1: Clean slate ---
if os.path.exists(db_file_path):
    os.remove(db_file_path)
    print(f"Removed existing database file.")

# --- Step 2: Build Schema from Google Drive ---
try:
    # Localizar el archivo en Drive por nombre
    drive_service = build('drive', 'v3', credentials=creds)
    SQL_FILENAME = "schema_v251122.sql"
    query = f"name = '{SQL_FILENAME}' and trashed = false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    
    if not files:
        raise FileNotFoundError(f"No se encontró {SQL_FILENAME} en Drive.")
    
    # Descargar el contenido del SQL
    file_id = files[0]['id']
    request = drive_service.files().get_media(fileId=file_id)
    schema_script = request.execute().decode('utf-8')

    # Ejecutar en SQLite
    conn = sqlite3.connect(db_file_path)
    conn.executescript(schema_script)
    conn.commit()
    print("✅ Schema created successfully from GDrive SQL file.")
except Exception as e:
    print(f"❌ ERROR during schema creation: {e}")
    assert False, "Failed to build schema."
finally:
    if 'conn' in locals(): conn.close()

# --- Step 3: Populate All Lookup Tables ---
print(f"\n--- Populating all lookup tables... ---")

product_category_data = [(1, 'uberx'), (2, 'comfort'), (3, 'business_comfort'), (4, 'black'), (5, 'uber_planet'), (6, 'uber_pet'), (7, 'envíos_uber')]
offer_action_data = [(1, 'accepted'), (2, 'reject')]
reason_primary_data = [(1, 'dropoff_non_operational'), (2, 'dropoff_proxy'), (3, 'low_profitability'), (4, 'long_pickup_time'), (5, 'dropoff_strategic_mismatch'), (6, 'expected_value_gamble'), (7, 'system_logic_failure')]
heuristic_flag_data = [(1, 'deadhead_risk'), (2, 'long_ride_traffic_risk'), (3, 'obj_end_session'), (4, 'market_anomaly'), (5, 'friday_traffic_risk'), (6, 'dropoff_uncertain'), (7, 'trap'), (8, 'protest_anomaly')]
post_offer_status_data = [(1, 'continue_game'), (2, 'disconnect')]
driver_state_at_request_data = [(1, 'looking_for_rides'), (2, 'trip_in_progress')]
outcome_data = [(1, 'completed'), (2, 'rider_canceled'), (3, 'driver_canceled'), (4, 'unfulfilled'), (5, 'system_failure')]
interpolation_quality_data = [(1, 'unanchored'), (2, 'interpolated'), (3, 'extrapolated_end'), (4, 'extrapolated_start'), (5, 'interpolated_stationary')]
record_status_data = [(1, 'valid'), (2, 'invalid_non_offer')]

try:
    conn = sqlite3.connect(db_file_path)
    cur = conn.cursor()
    cur.executemany("INSERT INTO product_category (product_category_id, category_name) VALUES (?, ?)", product_category_data)
    cur.executemany("INSERT INTO offer_action (offer_action_id, offer_action_description) VALUES (?, ?)", offer_action_data)
    cur.executemany("INSERT INTO reason_primary (reason_primary_id, reason_primary_description) VALUES (?, ?)", reason_primary_data)
    cur.executemany("INSERT INTO heuristic_flag (heuristic_flag_id, heuristic_flag_description) VALUES (?, ?)", heuristic_flag_data)
    cur.executemany("INSERT INTO post_offer_status (post_offer_status_id, post_offer_status_description) VALUES (?, ?)", post_offer_status_data)
    cur.executemany("INSERT INTO driver_state_at_request (driver_state_at_request_id, driver_state_at_request_description) VALUES (?, ?)", driver_state_at_request_data)
    cur.executemany("INSERT INTO outcome (outcome_id, outcome_description) VALUES (?, ?)", outcome_data)
    cur.executemany("INSERT INTO interpolation_quality (interpolation_quality_id, interpolation_quality_description) VALUES (?, ?)", interpolation_quality_data)
    cur.executemany("INSERT INTO record_status (record_status_id, record_status_description) VALUES (?, ?)", record_status_data)
    conn.commit()
    print("✅ Success: All lookup tables have been populated.")
except sqlite3.Error as e:
    print(f"\n❌ SQL error: {e}")
    conn.rollback()
finally:
    if 'conn' in locals(): conn.close()

print("\n--- PHASE 1 COMPLETE (LEGACY) ---")

--- PHASE 1 STARTED: Building Legacy database '/workspaces/pienza/data/big_bang/pienzaba.db' ---
Removed existing database file.
✅ Schema created successfully from GDrive SQL file.

--- Populating all lookup tables... ---
✅ Success: All lookup tables have been populated.

--- PHASE 1 COMPLETE (LEGACY) ---


In [4]:
# ===================================================================
# === CELL A3: AUDIT SCHEMA (GDRIVE LEGACY)
# Target: Notebook 0111a
# ===================================================================
import sqlite3

# Connect to the database created in the previous step
conn = sqlite3.connect(db_file_path)
cursor = conn.cursor()

# --- Audit 'offers' table ---
print("--- Auditing 'offers' table schema (LEGACY) ---")
offers_query = "PRAGMA table_info(offers);"
cursor.execute(offers_query)
offers_schema = cursor.fetchall()
for column in offers_schema:
    print(column)

# --- Audit 'offer_action' table ---
print("\n--- Auditing 'offer_action' table schema (LEGACY) ---")
action_query = "PRAGMA table_info(offer_action);"
cursor.execute(action_query)
action_schema = cursor.fetchall()
for column in action_schema:
    print(column)

# Close the connection
conn.close()

--- Auditing 'offers' table schema (LEGACY) ---
(0, 'offer_id', 'varchar(64)', 1, None, 1)
(1, 'session_id', 'INTEGER', 1, None, 0)
(2, 'ocr_id', 'INTEGER', 0, None, 0)
(3, 'image_content_hash', 'varchar(64)', 0, None, 0)
(4, 'offer_timestamp', 'datetime', 1, None, 0)
(5, 'upfront_fare', 'decimal(10,2)', 1, None, 0)
(6, 'time_to_pickup_sec', 'INTEGER', 1, None, 0)
(7, 'dist_to_pickup_km', 'decimal(5,2)', 1, None, 0)
(8, 'est_trip_time_sec', 'INTEGER', 1, None, 0)
(9, 'est_trip_dist_km', 'decimal(5,2)', 1, None, 0)
(10, 'pickup_address', 'TEXT', 1, None, 0)
(11, 'dropoff_address', 'TEXT', 1, None, 0)
(12, 'pickup_lat', 'decimal(9,6)', 0, None, 0)
(13, 'pickup_lon', 'decimal(9,6)', 0, None, 0)
(14, 'dropoff_lat', 'decimal(9,6)', 0, None, 0)
(15, 'dropoff_lon', 'decimal(9,6)', 0, None, 0)
(16, 'is_surge', 'boolean', 1, None, 0)
(17, 'surge_amount', 'decimal(10,2)', 0, None, 0)
(18, 'is_turbo_plus', 'boolean', 1, None, 0)
(19, 'turbo_plus_amount', 'decimal(10,2)', 0, None, 0)
(20, 'is_rese

In [5]:
# ====================================================================================
# === CELL A4: PHASE 2 - STAGE RAW OCR DATA (GDRIVE LEGACY)
# Target: Notebook 0111a
# ===================================================================
print(f"--- PHASE 2 STARTED: Staging Raw OCR data from GSheet (Mirroring Source) ---")

import pandas as pd
import os

# Definiciones locales a la fase
SHEET_TAB_NAME = "raw_requests_messy"
RAW_OCR_SHEET_NAME = "raw_Offers" # Asegurando que coincida con el nombre en Drive
staged_ocr_file = os.path.join(project_root, "staged_raw_ocr_legacy.parquet")

try:
    # Abrir el workbook por nombre (usando el objeto gc de A1)
    workbook = gc.open(RAW_OCR_SHEET_NAME)
    sheet = workbook.worksheet(SHEET_TAB_NAME)
    print(f"Successfully opened worksheet '{SHEET_TAB_NAME}'.")

    # --- [1] ROBUST EXTRACTION ---
    all_values = sheet.get_all_values()
    data_rows = all_values[1:] # Saltamos el header original de la hoja

    df = pd.DataFrame(data_rows)
    # Rebanar columnas A-K (0 a 11)
    df = df.iloc[:, 0:11]

    # Aplicar encabezados canónicos
    canonical_headers = [
        'ocr_id', 'image_filename', 'time_taken', 'ride_type', 'upfront_fare',
        'pickup_details', 'pickup_address', 'trip_details', 'dropoff_address',
        'rider_rating', 'special_note'
    ]
    df.columns = canonical_headers
    print("✅ Data extracted and sliced to columns A-K using canonical headers.")

    # --- [2] DATA CLEANSING (Minimal) ---
    df.dropna(subset=['ocr_id', 'image_filename'], how='any', inplace=True)
    df = df[(df['ocr_id'] != '') & (df['image_filename'] != '')].copy()
    print(f"Extracted and cleaned {len(df)} valid rows from the sheet.")

    # --- [3] NO TRANSFORMATION ---
    df_staged = df
    print("✅ Skipping transformation. Staging raw text data to preserve source integrity.")

    # --- [4] LOAD to Staging File ---
    df_staged.to_parquet(staged_ocr_file, index=False)
    if os.path.exists(staged_ocr_file):
        print(f"✅ SUCCESS! Staged {len(df_staged)} raw records to: {staged_ocr_file}")
    else:
        print(f"❌ FAILED: Staging file was not created.")

except Exception as e:
    print(f"❌ An error occurred during OCR data staging: {e}")

print("\n--- PHASE 2 COMPLETE (A) ---")

--- PHASE 2 STARTED: Staging Raw OCR data from GSheet (Mirroring Source) ---


Successfully opened worksheet 'raw_requests_messy'.
✅ Data extracted and sliced to columns A-K using canonical headers.
Extracted and cleaned 4765 valid rows from the sheet.
✅ Skipping transformation. Staging raw text data to preserve source integrity.
✅ SUCCESS! Staged 4765 raw records to: /workspaces/pienza/data/big_bang/staged_raw_ocr_legacy.parquet

--- PHASE 2 COMPLETE (A) ---


In [6]:
# =================================================================================
# === CELL A5: PHASE 3 - LOAD STAGED OCR DATA TO DB (GDRIVE LEGACY)
# Target: Notebook 0111a
# =================================================================================
print("--- PHASE 3 STARTED: Loading staged OCR data into 'raw_offers_ocr' table ---")

import pandas as pd
import sqlite3
import os
import numpy as np

# El path del archivo generado en la celda A4
source_parquet_file = os.path.join(project_root, "staged_raw_ocr_legacy.parquet")
table_name = 'raw_offers_ocr'

try:
    # --- [1] EXTRACT from staged file ---
    df_staged = pd.read_parquet(source_parquet_file)
    print(f"Extracted {len(df_staged)} raw records from staged Parquet file.")

    # --- [2] TRANSFORM (Nullification) ---
    print("Transforming string 'NULL's into true NULL values...")
    # Reemplazamos el lodo 'NULL' por np.nan para que SQLite lo entienda como NULL real
    df_to_load = df_staged.replace('NULL', np.nan)
    print("✅ Nullification complete.")

    # --- [3] LOAD ---
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"✅ Success: Loaded {len(df_to_load)} cleaned records into '{table_name}'.")

    # --- [4] VALIDATE ---
    with sqlite3.connect(db_file_path) as conn:
        count_df = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table_name}", conn)
        total_rows = count_df['count'][0]

    if total_rows == len(df_to_load):
        print(f"✅ Validation successful: DB table contains {total_rows} records.")
    else:
        print(f"❌ VALIDATION FAILED: Row count mismatch.")

except Exception as e:
    print(f"❌ An error occurred during the loading process: {e}")

print("\n--- PHASE 3 COMPLETE (A) ---")

--- PHASE 3 STARTED: Loading staged OCR data into 'raw_offers_ocr' table ---
Extracted 4765 raw records from staged Parquet file.
Transforming string 'NULL's into true NULL values...
✅ Nullification complete.
✅ Success: Loaded 4765 cleaned records into 'raw_offers_ocr'.
✅ Validation successful: DB table contains 4765 records.

--- PHASE 3 COMPLETE (A) ---


In [7]:
# =========================================================================================
# === CELL A6: PHASE 4 - ETL FOR THE POLISHED 'offers' TABLE (GDRIVE LEGACY)
# Target: Notebook 0111a
# =========================================================================================
print("--- PHASE 4 STARTED: Polishing and Loading data for the 'offers' table (A) ---")

import pandas as pd
import sqlite3
import os
import gspread
from tqdm import tqdm
import re
import numpy as np

# --- [1] EXTRACT ---
print("\n--- [E] EXTRACTING all necessary data sources from GDrive ---")
try:
    DIAMOND_SHEET_NAME = "raw_Offers"
    DIAMOND_TAB_NAME = "diamond_offers"
    
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    # get_all_records() preserva la estructura de diccionario de GSheet
    df_polished_raw = pd.DataFrame(sheet.get_all_records())
    print(f"✅ Extracted {len(df_polished_raw)} polished rows from GSheet.")

    with sqlite3.connect(db_file_path) as conn:
        df_ocr_link = pd.read_sql_query("SELECT ocr_id FROM raw_offers_ocr", conn)
    print(f"✅ Extracted {len(df_ocr_link)} rows from 'raw_offers_ocr' for linking.")

    with sqlite3.connect(db_file_path) as conn:
        lookup_table_names = [
            "product_category", "offer_action", "reason_primary", "heuristic_flag",
            "post_offer_status", "driver_state_at_request", "outcome",
            "interpolation_quality", "record_status"
        ]
        lookup_dfs = {}
        for table in tqdm(lookup_table_names, desc="Extracting lookups"):
            lookup_dfs[table] = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    print(f"✅ Extracted all {len(lookup_dfs)} lookup tables.")

except Exception as e:
    print(f"❌ ERROR during extraction: {e}")
    assert False, "Extraction failed, halting execution."

# --- [2] TRANSFORM (PART 1) ---
print("\n--- [T] TRANSFORMING and enriching 'offer' data ---")

if 'ocr_fk' in df_polished_raw.columns:
    df_polished_raw.drop(columns=['ocr_fk'], inplace=True)
df_polished_raw['join_key'] = pd.to_numeric(df_polished_raw['offer_id'].str.replace('OF', ''), errors='coerce')
df_ocr_link['join_key'] = pd.to_numeric(df_ocr_link['ocr_id'].str.replace('OCR_', ''), errors='coerce')
df_ocr_link.rename(columns={'ocr_id': 'ocr_fk'}, inplace=True)
df_transformed = pd.merge(df_polished_raw, df_ocr_link[['join_key', 'ocr_fk']], on='join_key', how='left')
df_transformed.drop(columns=['join_key'], inplace=True)
print("✅ ocr_fk link established.")

df_transformed.replace({'': None, 'N/A': None, 'nan': None, 'NULL': None}, inplace=True)

numeric_cols = [
    'upfront_fare', 'time_to_pickup_sec', 'dist_to_pickup_km', 'est_trip_time_sec',
    'est_trip_dist_km', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
    'surge_amount', 'turbo_plus_amount', 'reservation_amount', 'priority_amount',
    'rider_star_rating', 'rider_trip_count', 'time_in_session_sec', 'session_progress_ratio',
    'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing', 'inferred_agent_speed_mps'
]
for col in numeric_cols:
    if col in df_transformed.columns:
        df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
        df_transformed[col] = df_transformed[col].astype('float64')

df_transformed['offer_timestamp'] = pd.to_datetime(
    df_transformed['offer_timestamp'], format='%Y:%m:%d %H:%M:%S', errors='coerce'
).dt.strftime('%Y-%m-%d %H:%M:%S')

def three_state_bool_converter(value):
    if isinstance(value, str):
        val_lower = value.lower().strip()
        if val_lower == 'true': return True
        elif val_lower == 'false': return False
        else: return None
    elif isinstance(value, bool): return value
    else: return None

boolean_cols = [col for col in df_transformed.columns if col.startswith('is_')]
for col in boolean_cols:
    if col in df_transformed.columns:
        df_transformed[col] = df_transformed[col].apply(three_state_bool_converter)
        df_transformed[col] = df_transformed[col].astype('boolean')

merge_map_1_to_1 = {
    'product_category': ('product_category_fk', 'category_name'),
    'offer_action': ('offer_action_fk', 'offer_action_description'),
    'reason_primary': ('reason_primary_fk', 'reason_primary_description'),
    'post_offer_status': ('post_offer_status_fk', 'post_offer_status_description'),
    'driver_state_at_request': ('driver_state_at_request_fk', 'driver_state_at_request_description'),
    'outcome': ('outcome_fk', 'outcome_description'),
    'interpolation_quality': ('interpolation_quality_fk', 'interpolation_quality_description'),
    'record_status': ('record_status_fk', 'record_status_description')
}
for lookup_table, (source_fk_col, desc_col) in merge_map_1_to_1.items():
    if source_fk_col in df_transformed.columns:
        df_transformed[source_fk_col] = df_transformed[source_fk_col].astype(str).str.strip().str.lower().str.replace(' ', '_')
        if source_fk_col == 'product_category_fk':
            df_transformed[source_fk_col] = df_transformed[source_fk_col].str.replace('artículo', 'envíos_uber', regex=False)
        df_lookup = lookup_dfs[lookup_table]
        id_col = f"{lookup_table}_id"
        df_transformed = pd.merge(df_transformed, df_lookup[[id_col, desc_col]], left_on=source_fk_col, right_on=desc_col, how='left')
        df_transformed.drop(columns=[source_fk_col, desc_col], inplace=True)
        df_transformed.rename(columns={id_col: source_fk_col}, inplace=True)

# --- [BLOCK 2: M:M Heuristic] ---
df_heuristic_lookup = lookup_dfs['heuristic_flag']
df_flags_raw = df_transformed[['offer_id', 'heuristic_flag']].dropna(subset=['heuristic_flag'])
df_flags_raw['flag_list'] = df_flags_raw['heuristic_flag'].str.split(', ')
df_exploded = df_flags_raw.explode('flag_list')
df_exploded['flag_list'] = df_exploded['flag_list'].str.strip()
df_join_table_prep = pd.merge(df_exploded, df_heuristic_lookup, left_on='flag_list', right_on='heuristic_flag_description', how='inner')
df_heuristic_flag_offers_to_load = df_join_table_prep[['offer_id', 'heuristic_flag_id']].copy()
df_heuristic_flag_offers_to_load.rename(columns={'offer_id': 'offers_offer_id', 'heuristic_flag_id': 'heuristic_flag_heuristic_flag_id'}, inplace=True)

# --- [BLOCK 3: Load] ---
final_offer_columns = [
    'offer_id', 'session_fk', 'ocr_fk', 'image_content_hash', 'offer_timestamp', 'upfront_fare',
    'time_to_pickup_sec', 'dist_to_pickup_km', 'est_trip_time_sec', 'est_trip_dist_km',
    'pickup_address', 'dropoff_address', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
    'is_surge', 'surge_amount', 'is_turbo_plus', 'turbo_plus_amount', 'is_reservation',
    'reservation_amount', 'is_priority', 'priority_amount', 'is_exclusive', 'is_vip',
    'is_identity_verified', 'is_long_trip', 'is_multiple_destinations', 'is_teens',
    'rider_star_rating', 'rider_trip_count', 'time_in_session_sec', 'session_progress_ratio',
    'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing',
    'inferred_agent_speed_mps', 'is_imputed', 'special_note_raw', 'comment_1', 'comment_2',
    'product_category_fk', 'offer_action_fk', 'reason_primary_fk', 'post_offer_status_fk',
    'driver_state_at_request_fk', 'outcome_fk', 'interpolation_quality_fk', 'record_status_fk'
]
existing_final_columns = [col for col in final_offer_columns if col in df_transformed.columns]
df_offers_to_load = df_transformed[existing_final_columns]

with sqlite3.connect(db_file_path) as conn:
    df_offers_to_load.to_sql("offers", conn, if_exists='replace', index=False)
    df_heuristic_flag_offers_to_load.to_sql("heuristic_flag_offers", conn, if_exists='replace', index=False)

print(f"\n✅ PHASE 4 COMPLETE (A): Main 'offers' and M:M join tables loaded into Legacy DB.")

--- PHASE 4 STARTED: Polishing and Loading data for the 'offers' table (A) ---

--- [E] EXTRACTING all necessary data sources from GDrive ---


✅ Extracted 4765 polished rows from GSheet.
✅ Extracted 4765 rows from 'raw_offers_ocr' for linking.


Extracting lookups: 100%|██████████| 9/9 [00:00<00:00, 1806.59it/s]

✅ Extracted all 9 lookup tables.

--- [T] TRANSFORMING and enriching 'offer' data ---
✅ ocr_fk link established.



✅ PHASE 4 COMPLETE (A): Main 'offers' and M:M join tables loaded into Legacy DB.


In [8]:
# =================================================================================
# === CELL A7: THREE-STATE BOOLEAN PATCH (GDRIVE LEGACY)
# Target: Notebook 0111a
# =================================================================================
print("--- Starting the 'Three-State Boolean' Post-Processing Patch (A) ---")

import pandas as pd
import sqlite3
import os

try:
    # --- [1] Get the ground truth for NULL booleans from the GSheet ---
    print("Extracting source data to identify 'Unknown' booleans from Drive...")
    DIAMOND_SHEET_NAME = "raw_Offers"
    DIAMOND_TAB_NAME = "diamond_offers"
    
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    
    # Estandarizar nombres de columnas para el mapeo
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')
    boolean_cols = [col for col in df_source.columns if col.startswith('is_')]

    # Identificar qué IDs necesitan ser nulificados por columna
    ids_to_null = {col: [] for col in boolean_cols}
    for col in boolean_cols:
        # En GSheet, buscamos lo que no sea explícitamente booleano o texto TRUE/FALSE
        mask = ~df_source[col].isin(['TRUE', 'FALSE', True, False])
        ids_to_null[col] = df_source[mask]['offer_id'].tolist()

    print("✅ Successfully identified offers with 'Unknown' boolean states.")

    # --- [2] Execute the Surgical UPDATEs on the Database ---
    print(f"\nConnecting to {os.path.basename(db_file_path)} for surgical UPDATEs...")
    total_updates = 0
    with sqlite3.connect(db_file_path) as conn:
        cursor = conn.cursor()
        for col, ids in ids_to_null.items():
            if ids:
                placeholders = ', '.join(['?'] * len(ids))
                update_sql = f"UPDATE offers SET {col} = NULL WHERE offer_id IN ({placeholders});"
                cursor.execute(update_sql, ids)
                total_updates += cursor.rowcount
                print(f"  - Set {cursor.rowcount} records to NULL for column '{col}'.")

    print(f"\n✅ Patch complete. A total of {total_updates} fields were updated to NULL.")
    print("The 'offers' table now correctly represents the three-state boolean logic.")

except Exception as e:
    print(f"❌ ERROR during post-processing patch: {e}")

print("\n--- PATCH COMPLETE (A) ---")

--- Starting the 'Three-State Boolean' Post-Processing Patch (A) ---
Extracting source data to identify 'Unknown' booleans from Drive...
✅ Successfully identified offers with 'Unknown' boolean states.

Connecting to pienzaba.db for surgical UPDATEs...

✅ Patch complete. A total of 0 fields were updated to NULL.
The 'offers' table now correctly represents the three-state boolean logic.

--- PATCH COMPLETE (A) ---


In [9]:
# ==============================================================================
# === CELL A8: DIAGNOSTIC - BOOLEAN AUDIT (GDRIVE LEGACY)
# Target: Notebook 0111a
# ==============================================================================
print(f"--- Auditing Boolean Flags in Legacy DB: {os.path.basename(db_file_path)} ---")

import pandas as pd
import sqlite3

query_sql = """
SELECT 'is_surge' AS flag_name, SUM(is_surge) AS count_of_trues FROM offers
UNION ALL SELECT 'is_turbo_plus', SUM(is_turbo_plus) FROM offers
UNION ALL SELECT 'is_reservation', SUM(is_reservation) FROM offers
UNION ALL SELECT 'is_priority', SUM(is_priority) FROM offers
UNION ALL SELECT 'is_exclusive', SUM(is_exclusive) FROM offers
UNION ALL SELECT 'is_vip', SUM(is_vip) FROM offers
UNION ALL SELECT 'is_identity_verified', SUM(is_identity_verified) FROM offers
UNION ALL SELECT 'is_long_trip', SUM(is_long_trip) FROM offers
UNION ALL SELECT 'is_multiple_destinations', SUM(is_multiple_destinations) FROM offers
UNION ALL SELECT 'is_teens', SUM(is_teens) FROM offers
UNION ALL SELECT 'is_imputed', SUM(is_imputed) FROM offers;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_audit_a = pd.read_sql_query(query_sql, conn)
    
    print("\n✅ [A8] Audit query executed successfully.")
    print(df_audit_a)

except Exception as e:
    print(f"❌ [A8] ERROR during audit: {e}")

--- Auditing Boolean Flags in Legacy DB: pienzaba.db ---

✅ [A8] Audit query executed successfully.
                   flag_name  count_of_trues
0                   is_surge            1509
1              is_turbo_plus            1336
2             is_reservation             168
3                is_priority             293
4               is_exclusive            2642
5                     is_vip            1485
6       is_identity_verified            1983
7               is_long_trip             304
8   is_multiple_destinations              62
9                   is_teens              18
10                is_imputed              19


In [10]:
# ==============================================================================
# === CELL A9: DIAGNOSTIC - HEURISTIC FLAG ANOMALIES (GDRIVE LEGACY)
# Target: Notebook 0111a
# ==============================================================================
print("--- Hunting for the missing heuristic flag records in Legacy (A) ---")

import pandas as pd

# 1. Obtener el set de descripciones válidas desde el catálogo ya cargado
valid_flags = set(lookup_dfs['heuristic_flag']['heuristic_flag_description'])

# 2. Preparar el lodo de flags (asumiendo que df_transformed sigue en memoria)
df_flags_raw = df_transformed[['offer_id', 'heuristic_flag']].dropna(subset=['heuristic_flag'])
df_flags_raw['flag_list'] = df_flags_raw['heuristic_flag'].str.split(', ')
df_exploded = df_flags_raw.explode('flag_list')
df_exploded['flag_list'] = df_exploded['flag_list'].str.strip()

# 3. EL ANTI-JOIN: Buscar anomalías
df_anomalies_a = df_exploded[~df_exploded['flag_list'].isin(valid_flags)]

print(f"\n✅ [A9] Anomaly hunt complete. Found {len(df_anomalies_a)} invalid flag entries.")
if not df_anomalies_a.empty:
    print("These are the specific offers and flag strings that failed to match in GDrive:")
    print(df_anomalies_a[['offer_id', 'heuristic_flag', 'flag_list']])

--- Hunting for the missing heuristic flag records in Legacy (A) ---

✅ [A9] Anomaly hunt complete. Found 0 invalid flag entries.


In [11]:
# =================================================================================
# === CELL A10: CREATE THE MASTER ANALYTICAL VIEW (GDRIVE LEGACY)
# Target: Notebook 0111a
# =================================================================================
print(f"--- Creating the master analytical view in Legacy DB: {os.path.basename(db_file_path)} ---")

import pandas as pd
import sqlite3

# Master SQL Query: Reconciliación total de la tabla 'offers' con sus catálogos
view_sql = """
CREATE VIEW v_reconciled_offer AS
SELECT
    o.*, 

    -- Alias de descripciones para nombres limpios y únicos
    pc.category_name,
    oa.offer_action_description AS offer_action,
    rp.reason_primary_description AS reason_primary,
    pos.post_offer_status_description AS post_offer_status,
    ds.driver_state_at_request_description AS driver_state_at_request,
    ot.outcome_description AS outcome,
    iq.interpolation_quality_description AS interpolation_quality,
    rs.record_status_description AS record_status
FROM
    offers o
LEFT JOIN product_category pc        ON o.product_category_fk = pc.product_category_id
LEFT JOIN offer_action oa             ON o.offer_action_fk = oa.offer_action_id
LEFT JOIN reason_primary rp           ON o.reason_primary_fk = rp.reason_primary_id
LEFT JOIN post_offer_status pos       ON o.post_offer_status_fk = pos.post_offer_status_id
LEFT JOIN driver_state_at_request ds  ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
LEFT JOIN outcome ot                  ON o.outcome_fk = ot.outcome_id
LEFT JOIN interpolation_quality iq    ON o.interpolation_quality_fk = iq.interpolation_quality_id
LEFT JOIN record_status rs            ON o.record_status_fk = rs.record_status_id;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_reconciled_offer;")
        conn.execute(view_sql)
    print("✅ Success: 'v_reconciled_offer' VIEW created successfully in Legacy DB.")

    # --- Verification Step ---
    print("\n--- Final Verification: Querying the VIEW ---")
    with sqlite3.connect(db_file_path) as conn:
        df_view = pd.read_sql_query("""
            SELECT 
                offer_id, 
                offer_action, 
                reason_primary, 
                record_status 
            FROM 
                v_reconciled_offer 
            LIMIT 10;
        """, conn)
        print("Successfully queried the view. Displaying results:")
        print(df_view)

except Exception as e:
    print(f"❌ ERROR in Legacy View Creation: {e}")

print("\n--- PHASE 4 COMPLETE (A) ---")

--- Creating the master analytical view in Legacy DB: pienzaba.db ---
✅ Success: 'v_reconciled_offer' VIEW created successfully in Legacy DB.

--- Final Verification: Querying the VIEW ---
Successfully queried the view. Displaying results:
  offer_id offer_action           reason_primary record_status
0  OF00001       reject  dropoff_non_operational         valid
1  OF00002       reject  dropoff_non_operational         valid
2  OF00003     accepted                     None         valid
3  OF00004       reject        low_profitability         valid
4  OF00005       reject  dropoff_non_operational         valid
5  OF00006       reject  dropoff_non_operational         valid
6  OF00007       reject        low_profitability         valid
7  OF00008       reject  dropoff_non_operational         valid
8  OF00009       reject  dropoff_non_operational         valid
9  OF00010       reject        low_profitability         valid

--- PHASE 4 COMPLETE (A) ---


In [12]:
# =================================================================================
# === CELL A11: CREATE THE MASTER ANALYTICAL VIEW (GDRIVE LEGACY)
# Target: Notebook 0111a
# =================================================================================
print(f"--- Creating the master analytical view in Legacy DB: {os.path.basename(db_file_path)} ---")

import pandas as pd
import sqlite3
import os

# Query SQL para unir las tablas de catálogo con la tabla base 'offers'
view_sql = """
CREATE VIEW v_reconciled_offer AS
SELECT
    o.*,
    pc.category_name,
    oa.offer_action_description AS offer_action,
    rp.reason_primary_description AS reason_primary,
    pos.post_offer_status_description AS post_offer_status,
    ds.driver_state_at_request_description AS driver_state_at_request,
    ot.outcome_description AS outcome,
    iq.interpolation_quality_description AS interpolation_quality,
    rs.record_status_description AS record_status
FROM
    offers o
LEFT JOIN product_category pc        ON o.product_category_fk = pc.product_category_id
LEFT JOIN offer_action oa             ON o.offer_action_fk = oa.offer_action_id
LEFT JOIN reason_primary rp           ON o.reason_primary_fk = rp.reason_primary_id
LEFT JOIN post_offer_status pos       ON o.post_offer_status_fk = pos.post_offer_status_id
LEFT JOIN driver_state_at_request ds  ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
LEFT JOIN outcome ot                  ON o.outcome_fk = ot.outcome_id
LEFT JOIN interpolation_quality iq    ON o.interpolation_quality_fk = iq.interpolation_quality_id
LEFT JOIN record_status rs            ON o.record_status_fk = rs.record_status_id;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_reconciled_offer;")
        conn.execute(view_sql)
    print("✅ Success: 'v_reconciled_offer' VIEW created successfully in Legacy DB.")

except Exception as e:
    print(f"❌ ERROR creating view in Legacy DB: {e}")

--- Creating the master analytical view in Legacy DB: pienzaba.db ---
✅ Success: 'v_reconciled_offer' VIEW created successfully in Legacy DB.


In [13]:
# ==============================================================================
# === CELL A12: QUERY 1 (DIAGNOSTIC MODE) - GDRIVE LEGACY
# Target: Notebook 0111a
# ==============================================================================
import pandas as pd
import sqlite3
import os

print(f"\n--- Querying the master view in Legacy DB: {os.path.basename(db_file_path)} ---")

# Consulta diagnóstica con ordenamiento explícito para paridad visual
query = """
SELECT
    offer_id,
    offer_timestamp,
    upfront_fare,
    category_name,
    offer_action,
    reason_primary
FROM
    v_reconciled_offer
ORDER BY
    offer_timestamp DESC
LIMIT 20;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_sample = pd.read_sql_query(query, conn)

    print("\n✅ SUCCESS: Diagnostic query executed in Legacy DB.")
    print("Displaying raw data from the top 20 most recent offers in the VIEW:")
    print(df_sample)

except Exception as e:
    print(f"❌ ERROR: Query failed in Legacy. Reason: {e}")


--- Querying the master view in Legacy DB: pienzaba.db ---

✅ SUCCESS: Diagnostic query executed in Legacy DB.
Displaying raw data from the top 20 most recent offers in the VIEW:
   offer_id      offer_timestamp  upfront_fare     category_name offer_action  \
0   OF04765  2025-10-01 09:56:26        130.33           comfort       reject   
1   OF04764  2025-10-01 09:54:25         79.66       uber_planet       reject   
2   OF04763  2025-10-01 09:51:29         44.23           comfort       reject   
3   OF04762  2025-10-01 09:51:00         76.27             uberx       reject   
4   OF04761  2025-10-01 09:06:38        135.37           comfort     accepted   
5   OF04760  2025-10-01 08:35:48        165.68  business_comfort     accepted   
6   OF04759  2025-10-01 08:35:27        233.71  business_comfort       reject   
7   OF04758  2025-10-01 08:35:03        256.12             uberx       reject   
8   OF04757  2025-10-01 08:34:58        149.25             uberx       reject   
9   OF0475

In [14]:
# ==============================================================================
# === CELL A13: DATABASE DIAGNOSTIC - LIST ROWS WITH NULLS (GDRIVE LEGACY)
# Target: Notebook 0111a
# ==============================================================================
print(f"--- Auditing 'category_name' NULLs in Legacy DB: {os.path.basename(db_file_path)} ---")

import pandas as pd
import sqlite3
import os

# Seleccionamos la llave original para inspeccionar el lodo problemático
query_sql = """
SELECT
    offer_id,
    offer_timestamp,
    product_category_fk  -- Esta es la columna de origen a inspeccionar
FROM
    v_reconciled_offer
WHERE
    category_name IS NULL;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_nulls = pd.read_sql_query(query_sql, conn)

        print(f"\n✅ DIAGNOSTIC COMPLETE (A): Found {len(df_nulls)} records with a NULL category name.")

        if not df_nulls.empty:
            print("Displaying the raw 'product_category_fk' values for these records in Legacy:")
            print(df_nulls)

            unique_problem_values = df_nulls['product_category_fk'].unique()
            print("\n--- Summary of Unique Problematic Values ---")
            print("The following values in 'product_category_fk' could not be matched:")
            print(unique_problem_values)
        else:
            print("✅ [A] No records found with a NULL category name. The link is 100% complete.")

except Exception as e:
    print(f"❌ ERROR auditing the Legacy view: {e}")

--- Auditing 'category_name' NULLs in Legacy DB: pienzaba.db ---

✅ DIAGNOSTIC COMPLETE (A): Found 0 records with a NULL category name.
✅ [A] No records found with a NULL category name. The link is 100% complete.


In [15]:
# ==============================================================================
# === CELL A14: DIAGNOSTIC - FK VALIDATION (GDRIVE LEGACY)
# Target: Notebook 0111a
# ==============================================================================
print("--- Validating 'product_category_fk' in Legacy DB ---")

import pandas as pd
import sqlite3
import os

try:
    with sqlite3.connect(db_file_path) as conn:
        # Consulta 1: IDs distintos en la tabla de hechos
        print("\n--- [1] Distinct values found in 'offers.product_category_fk' ---")
        df_keys = pd.read_sql_query("SELECT DISTINCT product_category_fk FROM offers WHERE product_category_fk IS NOT NULL;", conn)
        print(df_keys)

        # Consulta 2: IDs válidos en el catálogo
        print("\n--- [2] All values in the 'product_category' Lookup Table ---")
        df_lookup = pd.read_sql_query("SELECT * FROM product_category;", conn)
        print(df_lookup)

    print("\n--- [3] VERIFICATION ---")
    key_set = set(df_keys['product_category_fk'])
    lookup_set = set(df_lookup['product_category_id'])

    if key_set.issubset(lookup_set):
        print("SUCCESS: All foreign key values in 'offers' are valid and exist in the lookup table.")
    else:
        print(f"WARNING: Found invalid foreign key values: {key_set - lookup_set}")

except Exception as e:
    print(f"ERROR: {e}")

--- Validating 'product_category_fk' in Legacy DB ---

--- [1] Distinct values found in 'offers.product_category_fk' ---
   product_category_fk
0                    1
1                    6
2                    2
3                    5
4                    3
5                    7
6                    4

--- [2] All values in the 'product_category' Lookup Table ---
   product_category_id     category_name
0                    1             uberx
1                    2           comfort
2                    3  business_comfort
3                    4             black
4                    5       uber_planet
5                    6          uber_pet
6                    7       envíos_uber

--- [3] VERIFICATION ---
SUCCESS: All foreign key values in 'offers' are valid and exist in the lookup table.


In [16]:
# ==============================================================================
# === CELL A15: TIMESTAMP COLUMN AUDIT - GDRIVE LEGACY
# Target: Notebook 0111a
# ==============================================================================
print("--- Performing a deep-dive audit of the 'offer_timestamp' column (Legacy A) ---")

import pandas as pd
import sqlite3
import os

try:
    with sqlite3.connect(db_file_path) as conn:
        # --- [1] Schema Audit ---
        print("\n--- [1] Schema Audit ---")
        df_schema = pd.read_sql_query("PRAGMA table_info(offers);", conn)
        timestamp_schema = df_schema[df_schema['name'] == 'offer_timestamp']
        print(timestamp_schema)

        # --- [2] Null Value Audit ---
        print("\n--- [2] Null Value Audit ---")
        null_count_query = "SELECT COUNT(*) AS null_count FROM offers WHERE offer_timestamp IS NULL;"
        df_null_count = pd.read_sql_query(null_count_query, conn)
        null_count = df_null_count['null_count'].iloc[0]
        print(f"Number of rows with a NULL 'offer_timestamp': {null_count}")

        # --- [3] Format & Range Audit ---
        print("\n--- [3] Format & Range Audit (Sample of distinct values) ---")
        format_query = """
        SELECT
            MIN(offer_timestamp) AS earliest_timestamp,
            MAX(offer_timestamp) AS latest_timestamp
        FROM offers
        WHERE offer_timestamp IS NOT NULL;
        """
        df_format = pd.read_sql_query(format_query, conn)
        print(df_format)

        total_rows_query = "SELECT COUNT(*) as total FROM offers;"
        total_rows = pd.read_sql_query(total_rows_query, conn)['total'].iloc[0]

        # --- [4] Final Diagnosis ---
        print("\n--- [4] Final Diagnosis ---")
        if null_count == total_rows:
            print("CRITICAL FINDING: All rows have a NULL 'offer_timestamp'. The ETL conversion is failing.")
        elif null_count > 0:
            print(f"WARNING: {null_count} out of {total_rows} rows have a NULL timestamp. Verify if those are the imputed rows.")
        else:
            print("SUCCESS: The 'offer_timestamp' column is fully populated with no NULL values.")
            first_ts = pd.read_sql_query("SELECT offer_timestamp FROM offers WHERE offer_timestamp IS NOT NULL LIMIT 1;", conn).iloc[0,0]
            print(f"Sample format check: '{first_ts}'. Correct 'YYYY-MM-DD HH:MM:SS' format.")

except Exception as e:
    print(f"ERROR during audit: {e}")

--- Performing a deep-dive audit of the 'offer_timestamp' column (Legacy A) ---

--- [1] Schema Audit ---
   cid             name  type  notnull dflt_value  pk
4    4  offer_timestamp  TEXT        0       None   0

--- [2] Null Value Audit ---
Number of rows with a NULL 'offer_timestamp': 0

--- [3] Format & Range Audit (Sample of distinct values) ---
    earliest_timestamp     latest_timestamp
0  2025-08-22 06:44:33  2025-10-01 09:56:26

--- [4] Final Diagnosis ---
SUCCESS: The 'offer_timestamp' column is fully populated with no NULL values.
Sample format check: '2025-08-22 06:44:33'. Correct 'YYYY-MM-DD HH:MM:SS' format.


In [17]:
# ==============================================================================
# === CELL A16: QUERY 4 - AVERAGE FARE BY DAY OF WEEK (GDRIVE LEGACY)
# Target: Notebook 0111a
# ==============================================================================
print("--- Calculando la tarifa promedio (upfront_fare) por día de la semana (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# Consulta utilizando strftime para extraer el día y CASE para el nombre
query_sql = """
SELECT
    CASE strftime('%w', offer_timestamp)
        WHEN '0' THEN '0 - Domingo'
        WHEN '1' THEN '1 - Lunes'
        WHEN '2' THEN '2 - Martes'
        WHEN '3' THEN '3 - Miércoles'
        WHEN '4' THEN '4 - Jueves'
        WHEN '5' THEN '5 - Viernes'
        WHEN '6' THEN '6 - Sábado'
    END AS day_of_week,
    AVG(upfront_fare) AS average_fare
FROM
    v_reconciled_offer
WHERE
    upfront_fare IS NOT NULL
GROUP BY
    day_of_week
ORDER BY
    strftime('%w', offer_timestamp);
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_avg_fare = pd.read_sql_query(query_sql, conn)

    print("SUCCESS: La consulta se ejecutó correctamente en Legacy DB.")
    print("Mostrando la tarifa promedio por día de la semana:")
    
    # Formateo limpio para la visualización
    display(df_avg_fare.style.format({'average_fare': '{:,.2f}'}))

except Exception as e:
    print(f"ERROR: La consulta falló en Legacy. Razón: {e}")

--- Calculando la tarifa promedio (upfront_fare) por día de la semana (Legacy A) ---
SUCCESS: La consulta se ejecutó correctamente en Legacy DB.
Mostrando la tarifa promedio por día de la semana:


,day_of_week,average_fare
0,0 - Domingo,106.12
1,1 - Lunes,123.02
2,2 - Martes,124.36
3,3 - Miércoles,133.91
4,4 - Jueves,138.77
5,5 - Viernes,141.80
6,6 - Sábado,124.83


In [18]:
# ==============================================================================
# === CELL A17: QUERY 5 (DIAGNOSTIC) - GDRIVE LEGACY
# Target: Notebook 0111a
# ==============================================================================
print("\n--- Calculando la tarifa promedio (upfront_fare) por hora del día (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# Esta consulta utiliza strftime('%H', ...) para extraer la hora militar (00-23)
query = """
SELECT
    strftime('%H', offer_timestamp) as hour_of_day,
    COUNT(offer_id) as number_of_offers,
    AVG(upfront_fare) as average_fare
FROM
    v_reconciled_offer
WHERE
    upfront_fare IS NOT NULL
GROUP BY
    hour_of_day
ORDER BY
    hour_of_day;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_hod_agg = pd.read_sql_query(query, conn)

    print("SUCCESS: La consulta horaria se ejecutó correctamente en Legacy DB.")
    print("Average Fare by Hour of Day:")
    display(df_hod_agg)

    # --- Preparación para Visualización ---
    df_hod_agg['hour_of_day'] = pd.to_numeric(df_hod_agg['hour_of_day'])
    df_hod_agg = df_hod_agg.set_index('hour_of_day')

except Exception as e:
    print(f"ERROR: Fallo en la consulta horaria de Legacy. Razón: {e}")


--- Calculando la tarifa promedio (upfront_fare) por hora del día (Legacy A) ---
SUCCESS: La consulta horaria se ejecutó correctamente en Legacy DB.
Average Fare by Hour of Day:


,hour_of_day,number_of_offers,average_fare
0,05,72,119.513611
1,06,797,133.209297
2,07,595,129.524924
3,08,414,122.646739
4,09,157,91.632229
5,10,68,92.844706
6,11,106,102.010755
7,12,112,104.132946
8,13,511,103.427397
9,14,496,145.397923


In [19]:
# ==============================================================================
# === CELL A18: QUERY 5 - LA HIPÓTESIS CENTRAL DE PIENZA (GDRIVE LEGACY)
# Target: Notebook 0111a
# ==============================================================================
import pandas as pd
import sqlite3
import os

# Configuracion de rutas
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/pienzaba.db')
    print("INFO: 'db_file_path' no encontrado. Usando ruta por defecto.")

print("\n--- Calculando la tarifa promedio para ofertas Aceptadas vs. Rechazadas (Legacy A) ---")

# Esta consulta agrupa todas las ofertas por su estado (aceptada/rechazada)
# y calcula la tarifa promedio para cada grupo.
query_sql = """
SELECT
    offer_action,
    AVG(upfront_fare) AS average_fare,
    COUNT(*) AS total_offers
FROM
    v_reconciled_offer
WHERE
    offer_action IN ('accepted', 'reject')
    AND upfront_fare IS NOT NULL
GROUP BY
    offer_action
ORDER BY
    average_fare DESC;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_hypothesis = pd.read_sql_query(query_sql, conn)

    print("SUCCESS: La consulta de la hipotesis central se ejecuto correctamente en Legacy DB.")
    print("Mostrando los resultados:")

    # Aplicamos un formato elegante para la moneda y los números enteros
    display(df_hypothesis.style.format({
        'average_fare': 'MXN {:,.2f}',
        'total_offers': '{:,}'
    }).hide(axis="index"))

except Exception as e:
    print(f"ERROR: La consulta fallo en Legacy. Razon: {e}")


--- Calculando la tarifa promedio para ofertas Aceptadas vs. Rechazadas (Legacy A) ---
SUCCESS: La consulta de la hipotesis central se ejecuto correctamente en Legacy DB.
Mostrando los resultados:


offer_action,average_fare,total_offers
accepted,MXN 147.59,346
reject,MXN 129.88,"4,419"


In [20]:
# ===================================================================
# === CELL A19: PHASE 5 - STAGE AND CLEAN GTS-4 DATA (GDRIVE LEGACY)
# Target: Notebook 0111a (Local Workspace Environment)
# ===================================================================
print("--- PHASE 5 STARTED: Staging and Cleaning GTS-4 Trip Events (Legacy A) ---")

import pandas as pd
import os
import gspread
pd.set_option('future.no_silent_downcasting', True)

# --- Extraction from GDrive ---
try:
    # Assuming gc is already initialized in a previous cell via service_account
    # or local oauth. We remove all google.colab references.
    
    spreadsheet = gc.open("GTS-4")
    worksheet = spreadsheet.worksheet("trip_events")
    
    # Load all records into DataFrame
    df_raw_gts = pd.DataFrame(worksheet.get_all_records())
    print(f"Extracted {len(df_raw_gts)} records from GSheet 'trip_events'.")

    # --- Transformation and PK Creation ---
    # Rename the legacy trip identifier for clarity and precision
    if 'event_id_legacy' in df_raw_gts.columns:
        df_raw_gts.rename(columns={'event_id_legacy': 'trip_id_legacy'}, inplace=True)

    # Create the new, unique, and non-negotiable Primary Key (event_id)
    df_raw_gts.reset_index(inplace=True)
    df_raw_gts.rename(columns={'index': 'event_id'}, inplace=True)
    df_raw_gts['event_id'] = df_raw_gts['event_id'] + 1 # Start PKs at 1

    # --- Data Purity & Schema Enforcement ---
    df_staged = df_raw_gts.copy()

    # 1. Handle Missing Timestamps (CRITICAL)
    df_staged['event_timestamp'] = pd.to_datetime(df_staged['event_timestamp'], errors='coerce')
    df_staged.dropna(subset=['event_timestamp'], inplace=True)

    # 2. Coerce Numeric Types
    numeric_cols = ['event_lat', 'event_lon', 'upfront_fare', 'realized_fare']
    for col in numeric_cols:
        if col in df_staged.columns:
            df_staged[col] = pd.to_numeric(df_staged[col], errors='coerce')

    # 3. Coerce Boolean Type
    boolean_map = {'TRUE': True, 'FALSE': False, 'True': True, 'False': False, True: True, False: False}
    if 'is_imputed' in df_staged.columns:
        df_staged['is_imputed'] = df_staged['is_imputed'].map(boolean_map).fillna(False)

    # --- Save to Staging ---
    # Using the local workspace path established in previous SOPs
    staged_file = os.path.join(project_root, "staged_gts4_legacy.parquet")
    df_staged.to_parquet(staged_file, index=False)
    
    print(f"SUCCESS: Staged {len(df_staged)} cleaned records to: {staged_file}")
    print("\n--- Final Health Check (Legacy) ---")
    df_staged.info()

except Exception as e:
    print(f"ERROR during GTS-4 staging in Legacy: {e}")

print("\n--- PHASE 5 COMPLETE (A) ---")

--- PHASE 5 STARTED: Staging and Cleaning GTS-4 Trip Events (Legacy A) ---
Extracted 1031 records from GSheet 'trip_events'.
SUCCESS: Staged 1031 cleaned records to: /workspaces/pienza/data/big_bang/staged_gts4_legacy.parquet

--- Final Health Check (Legacy) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event_id           1031 non-null   int64         
 1   trip_id_legacy     1031 non-null   object        
 2   event_timestamp    1031 non-null   datetime64[ns]
 3   event_types_id_fk  1031 non-null   object        
 4   event_lat          966 non-null    float64       
 5   event_lon          966 non-null    float64       
 6   event_address      1031 non-null   object        
 7   upfront_fare       259 non-null    float64       
 8   realized_fare      249 non-null    float64       
 9   is_imputed         1

In [21]:
# ===================================================================
# === CELL A20: PHASE 2 - TRANSFORM & NORMALIZE (event_types)
# Target: Notebook 0111a (Legacy)
# ===================================================================
print("\n--- PHASE 2: TRANSFORMING & NORMALIZING event_types (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# 1. Isolate unique event descriptions from the staged data
# Using the df_staged created in Cell A19
unique_event_descriptions = df_staged['event_types_id_fk'].unique()

# 2. Create the base lookup DataFrame
df_event_types = pd.DataFrame({'description': unique_event_descriptions})

# 3. Engineer the 'event_code' from the description
# Parsing the short code (e.g., 'T1') from strings like "T1: Ride Accepted"
df_event_types['event_code'] = df_event_types['description'].str.split(':').str[0]

# 4. Reorder the table based on the logical flow of a trip
event_order = ['T0', 'T1', 'T2', 'T3', 'T4']
df_event_types['event_code'] = pd.Categorical(df_event_types['event_code'], categories=event_order, ordered=True)
df_event_types = df_event_types.sort_values('event_code')

# 5. Create the Primary Key after sorting to ensure ID sequence matches logical flow
df_event_types.reset_index(drop=True, inplace=True)
df_event_types.reset_index(inplace=True)
df_event_types.rename(columns={'index': 'event_type_id'}, inplace=True)
df_event_types['event_type_id'] = df_event_types['event_type_id'] + 1 

# 6. Final Schema Alignment
final_columns = ['event_type_id', 'event_code', 'description']
df_event_types = df_event_types[final_columns]

# 7. Load into SQLite using a context manager
try:
    with sqlite3.connect(db_file_path) as conn:
        df_event_types.to_sql('event_types', conn, if_exists='replace', index=False)
    print("SUCCESS: 'event_types' table has been created and populated in Legacy DB.")
    print(f"Total event types loaded: {len(df_event_types)}")
except Exception as e:
    print(f"ERROR: Failed to load 'event_types' table in Legacy. Reason: {e}")

# Display the final lookup table for verification
print("\n--- Final event_types Lookup Table (Legacy) ---")
print(df_event_types)


--- PHASE 2: TRANSFORMING & NORMALIZING event_types (Legacy A) ---
SUCCESS: 'event_types' table has been created and populated in Legacy DB.
Total event types loaded: 5

--- Final event_types Lookup Table (Legacy) ---
   event_type_id event_code                           description
0              1         T0                 T0: Looking for rides
1              2         T1  T1: Ride Accepted, Driving to Pickup
2              3         T2             T2: Waiting for passenger
3              4         T3                      T3: Ride Started
4              5         T4                    T4: Ride completed


In [22]:
# ===================================================================
# === CELL A21: PHASE 3 - ENRICH & LOAD (trip_events)
# Target: Notebook 0111a (Legacy)
# ===================================================================
print("\n--- PHASE 3: ENRICHING & LOADING trip_events (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# 1. Enrich the main DataFrame by joining with the lookup table
# Joining staged data with the lookup table created in the previous cell
df_trip_events_prep = pd.merge(
    df_staged,
    df_event_types[['description', 'event_type_id']], 
    left_on='event_types_id_fk', 
    right_on='description',
    how='left'
)

# 2. Architectural Correction: Drop redundant text and rename numeric ID
df_trip_events_prep.drop(columns=['event_types_id_fk', 'description'], inplace=True)
df_trip_events_prep.rename(columns={'event_type_id': 'event_types_id_fk'}, inplace=True)

# 3. Final Schema Alignment
final_columns = [
    'event_id',
    'event_timestamp',
    'event_lat',
    'event_lon',
    'event_address',
    'upfront_fare',
    'realized_fare',
    'source',
    'is_imputed',
    'comment',
    'trip_id_legacy',
    'event_types_id_fk' 
]

existing_final_columns = [col for col in final_columns if col in df_trip_events_prep.columns]
df_trip_events = df_trip_events_prep[existing_final_columns]

# 4. Load the final, enriched DataFrame into the Legacy SQLite database
try:
    with sqlite3.connect(db_file_path) as conn:
        df_trip_events.to_sql('trip_events', conn, if_exists='replace', index=False)
    print("SUCCESS: 'trip_events' table has been created and populated in Legacy DB.")
    print(f"Total events loaded: {len(df_trip_events)}")
except Exception as e:
    print(f"ERROR: Failed to load 'trip_events' table in Legacy. Reason: {e}")

# --- Final verification ---
print("\n--- Verifying the first 5 rows of the loaded trip_events data ---")
display(df_trip_events.head())

print("\n--- Verifying the schema of the loaded trip_events data ---")
df_trip_events.info()


--- PHASE 3: ENRICHING & LOADING trip_events (Legacy A) ---
SUCCESS: 'trip_events' table has been created and populated in Legacy DB.
Total events loaded: 1031

--- Verifying the first 5 rows of the loaded trip_events data ---


,event_id,event_timestamp,event_lat,event_lon,event_address,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
0,1,2025-08-22 06:48:00,NaN,NaN,N/A,136.53,NaN,GTS-1,False,N/A,250822-01,2
1,2,2025-08-22 07:00:00,19.469418,-99.164297,"Colonia Ampliación Del Gas, Mexico City, Azcap...",NaN,NaN,GTS-1,False,N/A,250822-01,4
2,3,2025-08-22 07:22:05,19.435358,-99.182785,"Avenida Homero, Polanco 1ª Sección, Mexico Cit...",NaN,114.49,GTS-1,False,N/A,250822-01,5
3,4,2025-08-22 07:28:00,NaN,NaN,N/A,162.00,NaN,GTS-1,False,N/A,250822-02,2
4,5,2025-08-22 07:38:00,19.453696,-99.208148,"35, Callejón San Joaquín, Argentina Antigua, M...",NaN,NaN,GTS-1,False,N/A,250822-02,4



--- Verifying the schema of the loaded trip_events data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event_id           1031 non-null   int64         
 1   event_timestamp    1031 non-null   datetime64[ns]
 2   event_lat          966 non-null    float64       
 3   event_lon          966 non-null    float64       
 4   event_address      1031 non-null   object        
 5   upfront_fare       259 non-null    float64       
 6   realized_fare      249 non-null    float64       
 7   source             1031 non-null   object        
 8   is_imputed         1031 non-null   object        
 9   comment            1031 non-null   object        
 10  trip_id_legacy     1031 non-null   object        
 11  event_types_id_fk  1031 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(2), object(5)
mem

In [23]:
# ===================================================================
# === CELL A22: QUERY 1 - VERIFY DATA PRESENCE (GDRIVE LEGACY)
# Target: Notebook 0111a (Legacy)
# ===================================================================
import pandas as pd
import sqlite3
import os

# Ensuring the path reflects the Pienza rebranding
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

query_1 = "SELECT * FROM trip_events LIMIT 20;"

print("--- Running Query 1: Data presence check in Legacy DB ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        df_query_1_results = pd.read_sql_query(query_1, conn)

    print("SUCCESS: Query executed. Displaying first 20 rows from the Legacy database:")
    display(df_query_1_results)

except Exception as e:
    print(f"ERROR: Query 1 failed in Legacy. Reason: {e}")

--- Running Query 1: Data presence check in Legacy DB ---
SUCCESS: Query executed. Displaying first 20 rows from the Legacy database:


,event_id,event_timestamp,event_lat,event_lon,event_address,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
0,1,2025-08-22 06:48:00,NaN,NaN,N/A,136.53,NaN,GTS-1,0,N/A,250822-01,2
1,2,2025-08-22 07:00:00,19.469418,-99.164297,"Colonia Ampliación Del Gas, Mexico City, Azcap...",NaN,NaN,GTS-1,0,N/A,250822-01,4
2,3,2025-08-22 07:22:05,19.435358,-99.182785,"Avenida Homero, Polanco 1ª Sección, Mexico Cit...",NaN,114.49,GTS-1,0,N/A,250822-01,5
3,4,2025-08-22 07:28:00,NaN,NaN,N/A,162.00,NaN,GTS-1,0,N/A,250822-02,2
4,5,2025-08-22 07:38:00,19.453696,-99.208148,"35, Callejón San Joaquín, Argentina Antigua, M...",NaN,NaN,GTS-1,0,N/A,250822-02,4
5,6,2025-08-22 08:10:43,19.386737,-99.253680,"Calle Paseo de los Tamarindos, Bosques de las ...",NaN,135.72,GTS-1,0,N/A,250822-02,5
6,7,2025-08-22 08:12:00,NaN,NaN,N/A,64.31,NaN,GTS-1,0,N/A,250822-03,2
7,8,2025-08-22 08:17:00,19.373797,-99.259800,"Calle Roberto Medellín, Santa Fe Peña Blanca, ...",NaN,NaN,GTS-1,0,N/A,250822-03,4
8,9,2025-08-22 08:28:17,19.404960,-99.241322,"Calle Bosque de Duraznos, Bosques de las Lomas...",NaN,57.53,GTS-1,0,N/A,250822-03,5
9,10,2025-08-22 08:38:00,NaN,NaN,N/A,146.00,NaN,GTS-1,0,N/A,250822-04,2


In [24]:
# ===================================================================
# === CELL A23: QUERY - LAST 20 RECORDS (GDRIVE LEGACY)
# Target: Notebook 0111a (Legacy)
# ===================================================================
import pandas as pd
import sqlite3
import os

# Using the canonical path established in the workspace configuration
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# SQL query to retrieve the tail of the table
query_last_20 = """
SELECT *
FROM trip_events
ORDER BY event_id DESC
LIMIT 20;
"""

print("--- Running Query: Data tail check in Legacy DB ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        df_query_last_20_results = pd.read_sql_query(query_last_20, conn)

    print("SUCCESS: Query executed. Displaying the last 20 rows from the Legacy database:")
    # Re-sorting for a more intuitive ascending chronological display
    display(df_query_last_20_results.sort_values('event_id'))

except Exception as e:
    print(f"ERROR: Query failed in Legacy. Reason: {e}")

--- Running Query: Data tail check in Legacy DB ---
SUCCESS: Query executed. Displaying the last 20 rows from the Legacy database:


,event_id,event_timestamp,event_lat,event_lon,event_address,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
19,1012,2025-10-01 07:10:17,19.444644,-99.200335,"Torre Río 436, 436, Avenida Río San Joaquín, N...",NaN,NaN,GTS-4,0,N/A,251001-03,3
18,1013,2025-10-01 07:11:27,19.444644,-99.200335,"Torre Río 436, 436, Avenida Río San Joaquín, N...",NaN,NaN,GTS-4,0,N/A,251001-03,4
17,1014,2025-10-01 07:43:01,19.388093,-99.251259,"Sambalca Enterprise, Calle Paseo de los Tamari...",NaN,156.07,GTS-4,0,N/A,251001-03,5
16,1015,2025-10-01 07:43:53,19.390382,-99.249264,"14, Paseo de los Laureles, Bosques de las Loma...",133.47,NaN,GTS-4,0,N/A,251001-04,2
15,1016,2025-10-01 07:45:39,19.391369,-99.252085,"Calle Bosque de los Tabachines, Bosques de las...",NaN,NaN,GTS-4,0,N/A,251001-04,3
14,1017,2025-10-01 07:47:21,19.391383,-99.252093,"Calle Bosque de los Tabachines, Bosques de las...",NaN,NaN,GTS-4,0,N/A,251001-04,4
13,1018,2025-10-01 07:53:08,19.381855,-99.267378,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",NaN,120.01,GTS-4,0,N/A,251001-04,5
12,1019,2025-10-01 07:53:16,19.381855,-99.267379,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",NaN,NaN,GTS-4,0,N/A,251001-05,1
11,1020,2025-10-01 07:57:49,19.392293,-99.251349,"Calle Bosque de los Tabachines, Bosques de las...",120.57,NaN,GTS-4,0,N/A,251001-05,2
10,1021,2025-10-01 08:04:21,19.389307,-99.259022,"Privada Calle Ballonetas, Colonia Lomas del Ch...",NaN,NaN,GTS-4,0,N/A,251001-05,3


In [25]:
# ==============================================================================
# === CELL A24: DIAGNOSTIC AUDIT - JOIN INTEGRITY (GDRIVE LEGACY)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
import pandas as pd
import sqlite3
import os

# Self-Contained Configuration
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Auditing the JOIN condition between 'trip_events' and 'event_types' (Legacy A) ---")
print("Target Condition: trip_events.event_types_id_fk = event_types.event_type_id")

try:
    with sqlite3.connect(db_file_path) as conn:
        # Audit 1: Fact Table Keys
        print("\n--- [1] Unique values in 'trip_events.event_types_id_fk' ---")
        query1 = "SELECT DISTINCT event_types_id_fk FROM trip_events;"
        df_from = pd.read_sql_query(query1, conn)
        display(df_from)

        # Audit 2: Lookup Table Keys
        print("\n--- [2] All values in the 'event_types' Lookup Table ---")
        query2 = "SELECT event_type_id, event_code, description FROM event_types;"
        df_to = pd.read_sql_query(query2, conn)
        display(df_to)

except Exception as e:
    print(f"ERROR during diagnostic audit in Legacy: {e}")

--- Auditing the JOIN condition between 'trip_events' and 'event_types' (Legacy A) ---
Target Condition: trip_events.event_types_id_fk = event_types.event_type_id

--- [1] Unique values in 'trip_events.event_types_id_fk' ---


,event_types_id_fk
0,2
1,4
2,5
3,1
4,3



--- [2] All values in the 'event_types' Lookup Table ---


,event_type_id,event_code,description
0,1,T0,T0: Looking for rides
1,2,T1,"T1: Ride Accepted, Driving to Pickup"
2,3,T2,T2: Waiting for passenger
3,4,T3,T3: Ride Started
4,5,T4,T4: Ride completed


In [26]:
# ===================================================================
# === CELL A25: FINAL ANALYTICAL QUERY (v3.0) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# ===================================================================
import pandas as pd
import sqlite3
import os

print("--- INITIALIZING ANALYTICAL QUERY: TRIP FINANCIALS ---")

# Canonical path configuration
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

if not os.path.exists(db_file_path):
    print("CRITICAL FAILURE: The database file was not found at the specified path.")
else:
    print(f"Connecting to database at: {db_file_path}")

    # Aggregating fares into trip-level records
    sql_query_v3 = """
    WITH trip_financials AS (
        SELECT
            trip_id_legacy,
            MAX(CASE WHEN et.event_code = 'T1' THEN te.upfront_fare ELSE NULL END) AS trip_upfront_fare,
            MAX(CASE WHEN et.event_code = 'T4' THEN te.realized_fare ELSE NULL END) AS trip_real_fare
        FROM
            trip_events te
        JOIN
            event_types et ON te.event_types_id_fk = et.event_type_id
        WHERE
            te.trip_id_legacy IS NOT NULL
        GROUP BY
            te.trip_id_legacy
    )
    SELECT 
        trip_id_legacy,
        trip_upfront_fare,
        trip_real_fare
    FROM 
        trip_financials;
    """

    try:
        with sqlite3.connect(db_file_path) as conn:
            df_results_v3 = pd.read_sql_query(sql_query_v3, conn)

        print("\nSUCCESS: CTE financial query executed in Legacy DB.")
        print("Displaying aggregated trip financials:")
        display(df_results_v3.head(20))

    except Exception as e:
        print(f"\nERROR: Query failed in Legacy. Reason: {e}")

--- INITIALIZING ANALYTICAL QUERY: TRIP FINANCIALS ---
Connecting to database at: /workspaces/pienza/data/big_bang/pienzaba.db

SUCCESS: CTE financial query executed in Legacy DB.
Displaying aggregated trip financials:


,trip_id_legacy,trip_upfront_fare,trip_real_fare
0,250822-01,136.53,114.49
1,250822-02,162.00,135.72
2,250822-03,64.31,57.53
3,250822-04,146.00,130.60
4,250822-05,60.00,49.93
5,250822-06,108.67,86.91
6,250822-07,170.00,129.02
7,250822-08,173.44,131.25
8,250822-09,89.00,76.50
9,250822-10,202.00,155.24


In [27]:
# ===================================================================
# === CELL A26: FINAL ANALYTICAL CELL (v4.1) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# ===================================================================
import pandas as pd
import sqlite3
import os

# Updated to reflect the Pienza canonical path and nomenclature
project_root = '/workspaces/pienza'
db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# Master query pivoting timestamps and fares by event code
query_the_money = """
WITH trip_summary_raw AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN et.event_code = 'T0' THEN te.event_timestamp ELSE NULL END) AS t0_timestamp,
        MAX(CASE WHEN et.event_code = 'T1' THEN te.event_timestamp ELSE NULL END) AS t1_timestamp,
        MAX(CASE WHEN et.event_code = 'T2' THEN te.event_timestamp ELSE NULL END) AS t2_timestamp,
        MAX(CASE WHEN et.event_code = 'T3' THEN te.event_timestamp ELSE NULL END) AS t3_timestamp,
        MAX(CASE WHEN et.event_code = 'T4' THEN te.event_timestamp ELSE NULL END) AS t4_timestamp,
        MAX(CASE WHEN et.event_code = 'T1' THEN te.upfront_fare ELSE NULL END) AS t1_upfront_fare,
        MAX(CASE WHEN et.event_code = 'T4' THEN te.realized_fare ELSE NULL END) AS t4_realized_fare
    FROM
        trip_events te
    JOIN
        event_types et ON te.event_types_id_fk = et.event_type_id
    GROUP BY
        te.trip_id_legacy
)
SELECT * FROM trip_summary_raw;
"""

print(f"--- INITIALIZING: Targeting legacy database at '{db_file_path}' ---")

if not os.path.exists(db_file_path):
    print("CRITICAL FAILURE: The database file does NOT exist.")
else:
    print("Pre-flight check PASSED. Database file found.")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_money_table = pd.read_sql_query(query_the_money, conn)

        print("SUCCESS: Query executed. Generating Pienza Money Table.")

        # Data Type Conversion for analytical readiness
        timestamp_cols = ['t0_timestamp', 't1_timestamp', 't2_timestamp', 't3_timestamp', 't4_timestamp']
        for col in timestamp_cols:
            df_money_table[col] = pd.to_datetime(df_money_table[col], errors='coerce')

        display(df_money_table)

    except Exception as e:
        print(f"ERROR: Query failed in Legacy. Reason: {e}")

--- INITIALIZING: Targeting legacy database at '/workspaces/pienza/data/big_bang/pienzaba.db' ---
Pre-flight check PASSED. Database file found.
SUCCESS: Query executed. Generating Pienza Money Table.


,trip_id_legacy,t0_timestamp,t1_timestamp,t2_timestamp,t3_timestamp,t4_timestamp,t1_upfront_fare,t4_realized_fare
0,250822-01,NaT,2025-08-22 06:48:00,NaT,2025-08-22 07:00:00,2025-08-22 07:22:05,136.53,114.49
1,250822-02,NaT,2025-08-22 07:28:00,NaT,2025-08-22 07:38:00,2025-08-22 08:10:43,162.00,135.72
2,250822-03,NaT,2025-08-22 08:12:00,NaT,2025-08-22 08:17:00,2025-08-22 08:28:17,64.31,57.53
3,250822-04,NaT,2025-08-22 08:38:00,NaT,2025-08-22 08:49:00,2025-08-22 09:17:10,146.00,130.60
4,250822-05,NaT,2025-08-22 09:16:00,NaT,2025-08-22 09:20:00,2025-08-22 09:29:01,60.00,49.93
...,...,...,...,...,...,...,...,...
254,251001-03,NaT,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01,182.17,156.07
255,251001-04,NaT,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08,133.47,120.01
256,251001-05,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35,120.57,106.95
257,251001-06,2025-10-01 08:34:44,2025-10-01 08:36:35,NaT,2025-10-01 08:44:11,2025-10-01 09:13:29,165.68,148.14


In [28]:
# ==============================================================================
# === CELL A27: SCHEMA AUDIT - TRIP EVENTS (GDRIVE LEGACY)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
import pandas as pd
import sqlite3
import os

print("--- Auditing the Canonical Schema for 'trip_events' (Legacy A) ---")

# PRAGMA is the definitive source for table structure in SQLite
query_sql = "PRAGMA table_info(trip_events);"

try:
    with sqlite3.connect(db_file_path) as conn:
        # Use pandas for clean, tabular metadata output
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Schema audit completed for Legacy DB.")
    print("Below is the definitive and canonical list of columns in the database:")
    display(df_schema)

except Exception as e:
    print(f"ERROR: Schema audit failed in Legacy. Reason: {e}")

--- Auditing the Canonical Schema for 'trip_events' (Legacy A) ---

SUCCESS: Schema audit completed for Legacy DB.
Below is the definitive and canonical list of columns in the database:


,cid,name,type,notnull,dflt_value,pk
0,0,event_id,INTEGER,0,None,0
1,1,event_timestamp,TIMESTAMP,0,None,0
2,2,event_lat,REAL,0,None,0
3,3,event_lon,REAL,0,None,0
4,4,event_address,TEXT,0,None,0
5,5,upfront_fare,REAL,0,None,0
6,6,realized_fare,REAL,0,None,0
7,7,source,TEXT,0,None,0
8,8,is_imputed,INTEGER,0,None,0
9,9,comment,TEXT,0,None,0


In [29]:
# ==============================================================================
# === CELL A28: TASK 1 (Self-Contained) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# ==============================================================================
import pandas as pd
import sqlite3
import os

# Configuration for the Pienza canonical path
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Architecting the 'v_trip_funnel_metrics' analytical view (Legacy A) ---")

view_sql = """
CREATE VIEW v_trip_funnel_metrics AS
WITH PivotedEvents AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_completed,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,
    (julianday(p.t2_arrived) - julianday(p.t1_accepted)) * 86400.0 AS duration_to_pickup_sec,
    (julianday(p.t3_started) - julianday(p.t2_arrived)) * 86400.0 AS duration_waiting_sec,
    (julianday(p.t4_completed) - julianday(p.t3_started)) * 86400.0 AS duration_trip_sec,
    (julianday(p.t4_completed) - julianday(p.t1_accepted)) * 86400.0 AS duration_total_engagement_sec,
    p.t0_looking,
    p.t1_accepted,
    p.t2_arrived,
    p.t3_started,
    p.t4_completed
FROM
    PivotedEvents p;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_metrics;")
        conn.execute(view_sql)
    print("SUCCESS: View 'v_trip_funnel_metrics' created successfully in Legacy DB.")

    # --- Verification Step (CORRECTED) ---
    print("\n--- Displaying LAST 20 rows of the new view for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_metrics
        ORDER BY t1_accepted DESC
        LIMIT 20;
        """
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"ERROR creating view in Legacy: {e}")

--- Architecting the 'v_trip_funnel_metrics' analytical view (Legacy A) ---
SUCCESS: View 'v_trip_funnel_metrics' created successfully in Legacy DB.

--- Displaying LAST 20 rows of the new view for verification ---


,trip_id_legacy,upfront_fare,realized_fare,duration_to_pickup_sec,duration_waiting_sec,duration_trip_sec,duration_total_engagement_sec,t0_looking,t1_accepted,t2_arrived,t3_started,t4_completed
0,251001-07,135.37,125.34,259.999998,48.000021,1992.000018,2300.000037,None,2025-10-01 09:13:54,2025-10-01 09:18:14,2025-10-01 09:19:02,2025-10-01 09:52:14
1,251001-06,165.68,148.14,NaN,NaN,1758.000000,2213.999981,2025-10-01 08:34:44,2025-10-01 08:36:35,None,2025-10-01 08:44:11,2025-10-01 09:13:29
2,251001-05,120.57,106.95,392.000006,22.000001,1791.999976,2205.999984,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35
3,251001-04,133.47,120.01,106.000029,101.999970,347.000009,555.000007,None,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08
4,251001-03,182.17,156.07,613.000014,69.999982,1893.999986,2576.999983,None,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01
5,251001-02,114.69,97.40,395.999984,70.000023,1053.999996,1520.000003,2025-10-01 06:30:56,2025-10-01 06:33:52,2025-10-01 06:40:28,2025-10-01 06:41:38,2025-10-01 06:59:12
6,251001-01,151.30,142.47,390.000017,168.999968,1119.000006,1677.999991,2025-10-01 05:59:11,2025-10-01 06:02:43,2025-10-01 06:09:13,2025-10-01 06:12:02,2025-10-01 06:30:41
7,250930-10,194.44,163.10,NaN,NaN,1545.000029,1802.000003,2025-09-30 15:40:18,2025-09-30 15:44:44,None,2025-09-30 15:49:01,2025-09-30 16:14:46
8,250930-09,102.68,84.81,NaN,NaN,1512.000006,1660.000008,None,2025-09-30 15:12:14,None,2025-09-30 15:14:42,2025-09-30 15:39:54
9,250930-08,108.91,102.68,501.000018,85.999976,818.000029,1405.000024,2025-09-30 14:41:41,2025-09-30 14:47:33,2025-09-30 14:55:54,2025-09-30 14:57:20,2025-09-30 15:10:58


In [30]:
# =========================================================================================
# === CELL A29: TASK 1, STEP 1/2 (v1.2) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# =========================================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Architecting the foundational 'v_trip_funnel_wide' view (Legacy A) ---")

# Columns include both timestamps and addresses for full traceability
view_sql = """
CREATE VIEW v_trip_funnel_wide AS
SELECT
    trip_id_legacy,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_address END)   AS t0_address,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_address END)   AS t1_address,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_address END)   AS t2_address,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_address END)   AS t3_address,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_address END)   AS t4_address,
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END)    AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END)   AS realized_fare
FROM
    trip_events
GROUP BY
    trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_wide;")
        conn.execute(view_sql)
    print("SUCCESS: Foundational view 'v_trip_funnel_wide' created successfully in Legacy DB.")

    # --- Verification Step ---
    print("\n--- Displaying LAST 10 rows for verification (Most Recent Trips) ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_wide
        ORDER BY t1_timestamp DESC
        LIMIT 10;
        """
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"ERROR creating view in Legacy: {e}")

--- Architecting the foundational 'v_trip_funnel_wide' view (Legacy A) ---
SUCCESS: Foundational view 'v_trip_funnel_wide' created successfully in Legacy DB.

--- Displaying LAST 10 rows for verification (Most Recent Trips) ---


,trip_id_legacy,t0_timestamp,t0_address,t1_timestamp,t1_address,t2_timestamp,t2_address,t3_timestamp,t3_address,t4_timestamp,t4_address,upfront_fare,realized_fare
0,251001-07,None,None,2025-10-01 09:13:54,"Marriott Mexico City Reforma Hotel, 276, Aveni...",2025-10-01 09:18:14,"Calle Nápoles, Zona Rosa, Mexico City, Cuauhté...",2025-10-01 09:19:02,"Calle Hamburgo, Zona Rosa, Mexico City, Cuauht...",2025-10-01 09:52:14,"Avenida Paseo de las Palmas, Lomas de Chapulte...",135.37,125.34
1,251001-06,2025-10-01 08:34:44,"360, Calle Montes Urales Norte, Lomas de Chapu...",2025-10-01 08:36:35,"220, Calle Montes Urales Norte, Lomas de Chapu...",None,None,2025-10-01 08:44:11,Tecamachalco 11650,2025-10-01 09:13:29,"Marriott Mexico City Reforma Hotel, 276, Aveni...",165.68,148.14
2,251001-05,2025-10-01 07:53:16,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",2025-10-01 07:57:49,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 08:04:21,"Privada Calle Ballonetas, Colonia Lomas del Ch...",2025-10-01 08:04:43,"Privada Calle Ballonetas, Colonia Lomas del Ch...",2025-10-01 08:34:35,"360, Calle Montes Urales Norte, Lomas de Chapu...",120.57,106.95
3,251001-04,None,None,2025-10-01 07:43:53,"14, Paseo de los Laureles, Bosques de las Loma...",2025-10-01 07:45:39,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 07:47:21,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 07:53:08,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",133.47,120.01
4,251001-03,None,None,2025-10-01 07:00:04,"Avenida Ejército Nacional Mexicano, Granada, M...",2025-10-01 07:10:17,"Torre Río 436, 436, Avenida Río San Joaquín, N...",2025-10-01 07:11:27,"Torre Río 436, 436, Avenida Río San Joaquín, N...",2025-10-01 07:43:01,"Sambalca Enterprise, Calle Paseo de los Tamari...",182.17,156.07
5,251001-02,2025-10-01 06:30:56,"18, Calle Alicama, Lomas de Chapultepec 4ª Sec...",2025-10-01 06:33:52,"Avenida Lomas, Colonia Lomas de Virreyes, Loma...",2025-10-01 06:40:28,"Calle Poniente 81, Colonia Cove, Mexico City, ...",2025-10-01 06:41:38,"Calle Poniente 81, Colonia Cove, Mexico City, ...",2025-10-01 06:59:12,"Avenida Ejército Nacional Mexicano, Granada, M...",114.69,97.40
6,251001-01,2025-10-01 05:59:11,"Calle Comte, Colonia Nueva Anzures, Anzures, M...",2025-10-01 06:02:43,"Avenida Río Mississippi, Cuauhtémoc, Mexico Ci...",2025-10-01 06:09:13,"Calle Eligio Ancona, Santa María la Ribera, Me...",2025-10-01 06:12:02,"228, Calle Nogal, Santa María la Ribera, Mexic...",2025-10-01 06:30:41,"Calle Pedregal, Lomas de Chapultepec 4ª Secció...",151.30,142.47
7,250930-10,2025-09-30 15:40:18,"Corporativo Lomas Cantabria, 22, Cerrada de la...",2025-09-30 15:44:44,"Extra Periférico, 168, Boulevard Manuel Ávila ...",None,None,2025-09-30 15:49:01,FC Cuernavaca,2025-09-30 16:14:46,"364, Avenida Paseo de la Reforma, Little Seoul...",194.44,163.10
8,250930-09,None,None,2025-09-30 15:12:14,"Avenida Vasco de Quiroga, Santa Fe Cuajimalpa,...",None,None,2025-09-30 15:14:42,Geocoding failed,2025-09-30 15:39:54,"Corporativo Lomas Cantabria, 22, Cerrada de la...",102.68,84.81
9,250930-08,2025-09-30 14:41:41,"Calle Bosque de Pirules, Colonia Bosques de Re...",2025-09-30 14:47:33,"Avenida Paseo de los Ahuehuetes Sur, Colonia B...",2025-09-30 14:55:54,"Roche, 9, Calle Molino Bezares, Lomas de Bezar...",2025-09-30 14:57:20,"Avenida Constituyentes - Puente CONAFRUT, Boul...",2025-09-30 15:10:58,"Calle José Villagrán, Santa Fe Cuajimalpa, Mex...",108.91,102.68


In [31]:
# =========================================================================================
# === CELL A30: TASK 1, STEP 1/2 (v1.3) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# =========================================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Architecting the foundational 'v_trip_funnel_wide' view (Legacy A - v1.3 Geo-Enriched) ---")

view_sql = """
CREATE VIEW v_trip_funnel_wide AS
SELECT
    trip_id_legacy,
    -- T0 Event Group (Looking)
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_lat END)       AS t0_lat,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_lon END)       AS t0_lon,
    -- T1 Event Group (Accepted)
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_lat END)       AS t1_lat,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_lon END)       AS t1_lon,
    -- T2 Event Group (Arrived)
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_lat END)       AS t2_lat,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_lon END)       AS t2_lon,
    -- T3 Event Group (Started)
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_lat END)       AS t3_lat,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_lon END)       AS t3_lon,
    -- T4 Event Group (Completed)
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_lat END)       AS t4_lat,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_lon END)       AS t4_lon,
    -- Financials
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END)    AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END)   AS realized_fare
FROM
    trip_events
GROUP BY
    trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_wide;")
        conn.execute(view_sql)
    print("SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.3) created successfully in Legacy DB.")

    # --- Verification Step ---
    print("\n--- Displaying LAST 10 rows for verification (Geo-Enriched Sample) ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_wide
        ORDER BY t1_timestamp DESC
        LIMIT 10;
        """
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"ERROR creating view in Legacy: {e}")

--- Architecting the foundational 'v_trip_funnel_wide' view (Legacy A - v1.3 Geo-Enriched) ---
SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.3) created successfully in Legacy DB.

--- Displaying LAST 10 rows for verification (Geo-Enriched Sample) ---


,trip_id_legacy,t0_timestamp,t0_lat,t0_lon,t1_timestamp,t1_lat,t1_lon,t2_timestamp,t2_lat,t2_lon,t3_timestamp,t3_lat,t3_lon,t4_timestamp,t4_lat,t4_lon,upfront_fare,realized_fare
0,251001-07,None,NaN,NaN,2025-10-01 09:13:54,19.428175,-99.164291,2025-10-01 09:18:14,19.429299,-99.161305,2025-10-01 09:19:02,19.427816,-99.161410,2025-10-01 09:52:14,19.429272,-99.215838,135.37,125.34
1,251001-06,2025-10-01 08:34:44,19.429017,-99.207317,2025-10-01 08:36:35,19.429866,-99.210289,None,NaN,NaN,2025-10-01 08:44:11,NaN,NaN,2025-10-01 09:13:29,19.428157,-99.164338,165.68,148.14
2,251001-05,2025-10-01 07:53:16,19.381855,-99.267379,2025-10-01 07:57:49,19.392293,-99.251349,2025-10-01 08:04:21,19.389307,-99.259022,2025-10-01 08:04:43,19.389395,-99.258921,2025-10-01 08:34:35,19.429016,-99.207312,120.57,106.95
3,251001-04,None,NaN,NaN,2025-10-01 07:43:53,19.390382,-99.249264,2025-10-01 07:45:39,19.391369,-99.252085,2025-10-01 07:47:21,19.391383,-99.252093,2025-10-01 07:53:08,19.381855,-99.267378,133.47,120.01
4,251001-03,None,NaN,NaN,2025-10-01 07:00:04,19.438687,-99.205490,2025-10-01 07:10:17,19.444644,-99.200335,2025-10-01 07:11:27,19.444644,-99.200335,2025-10-01 07:43:01,19.388093,-99.251259,182.17,156.07
5,251001-02,2025-10-01 06:30:56,19.422493,-99.204313,2025-10-01 06:33:52,19.418508,-99.205599,2025-10-01 06:40:28,19.401792,-99.196382,2025-10-01 06:41:38,19.401792,-99.196382,2025-10-01 06:59:12,19.438657,-99.205160,114.69,97.40
6,251001-01,2025-10-01 05:59:11,19.428923,-99.174633,2025-10-01 06:02:43,19.428491,-99.173390,2025-10-01 06:09:13,19.453481,-99.162617,2025-10-01 06:12:02,19.453011,-99.163214,2025-10-01 06:30:41,19.422494,-99.204347,151.30,142.47
7,250930-10,2025-09-30 15:40:18,19.433761,-99.212895,2025-09-30 15:44:44,19.434579,-99.211605,None,NaN,NaN,2025-09-30 15:49:01,NaN,NaN,2025-09-30 16:14:46,19.426131,-99.168587,194.44,163.10
8,250930-09,None,NaN,NaN,2025-09-30 15:12:14,19.364700,-99.269883,None,NaN,NaN,2025-09-30 15:14:42,19.367542,-99.262921,2025-09-30 15:39:54,19.433774,-99.212901,102.68,84.81
9,250930-08,2025-09-30 14:41:41,19.393637,-99.260057,2025-09-30 14:47:33,19.394674,-99.254599,2025-09-30 14:55:54,19.387984,-99.243675,2025-09-30 14:57:20,19.387250,-99.243269,2025-09-30 15:10:58,19.363002,-99.273047,108.91,102.68


In [32]:
# =========================================================================================
# === CELL A31: TASK 1, STEP 2/2 - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# =========================================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Architecting the final 'v_trip_final_kpis' analytical view (Legacy A) ---")

# This query aggregates the imputed flag and calculates business-critical KPIs
view_sql = """
CREATE VIEW v_trip_final_kpis AS
WITH ImputedFlag AS (
    SELECT
        trip_id_legacy,
        MAX(is_imputed) AS is_imputed
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,
    v.upfront_fare,
    v.realized_fare,
    i.is_imputed,

    -- KPI: Spread Percentage
    CASE
        WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare
        ELSE NULL
    END AS spread_percentage,

    -- KPI: Phase Durations in Seconds
    (julianday(v.t1_timestamp) - julianday(v.t0_timestamp)) * 86400.0 AS duration_lfr_sec,
    (julianday(v.t2_timestamp) - julianday(v.t1_timestamp)) * 86400.0 AS duration_dtp_sec,
    (julianday(v.t3_timestamp) - julianday(v.t2_timestamp)) * 86400.0 AS duration_wfp_sec,
    (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 86400.0 AS duration_on_ride_sec,

    -- KPI: Earnings Per Hour (EPH)
    CASE
        WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0
        THEN v.upfront_fare / ((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 24.0)
        ELSE NULL
    END AS eph_upfront,

    CASE
        WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0
        THEN v.realized_fare / ((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 24.0)
        ELSE NULL
    END AS eph_realized,

    v.t0_timestamp, v.t1_timestamp, v.t2_timestamp, v.t3_timestamp, v.t4_timestamp

FROM
    v_trip_funnel_wide v
LEFT JOIN
    ImputedFlag i ON v.trip_id_legacy = i.trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_final_kpis;")
        conn.execute(view_sql)
    print("SUCCESS: Final KPI view 'v_trip_final_kpis' created successfully in Legacy DB.")

    # --- Verification Step ---
    print("\n--- Displaying LAST 10 rows for verification (KPI Sample) ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = "SELECT * FROM v_trip_final_kpis ORDER BY t1_timestamp DESC LIMIT 10;"
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"ERROR creating view in Legacy: {e}")

--- Architecting the final 'v_trip_final_kpis' analytical view (Legacy A) ---
SUCCESS: Final KPI view 'v_trip_final_kpis' created successfully in Legacy DB.

--- Displaying LAST 10 rows for verification (KPI Sample) ---


,trip_id_legacy,trip_date,upfront_fare,realized_fare,is_imputed,spread_percentage,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,eph_upfront,eph_realized,t0_timestamp,t1_timestamp,t2_timestamp,t3_timestamp,t4_timestamp
0,251001-07,2025-10-01,135.37,125.34,0,0.925907,NaN,259.999998,48.000021,1992.000018,244.644576,226.518070,None,2025-10-01 09:13:54,2025-10-01 09:18:14,2025-10-01 09:19:02,2025-10-01 09:52:14
1,251001-06,2025-10-01,165.68,148.14,1,0.894133,111.000001,NaN,NaN,1758.000000,339.276451,303.358362,2025-10-01 08:34:44,2025-10-01 08:36:35,None,2025-10-01 08:44:11,2025-10-01 09:13:29
2,251001-05,2025-10-01,120.57,106.95,0,0.887037,273.000008,392.000006,22.000001,1791.999976,242.216521,214.854914,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35
3,251001-04,2025-10-01,133.47,120.01,0,0.899153,NaN,106.000029,101.999970,347.000009,1384.703135,1245.060487,None,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08
4,251001-03,2025-10-01,182.17,156.07,0,0.856727,NaN,613.000014,69.999982,1893.999986,346.257658,296.648365,None,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01
5,251001-02,2025-10-01,114.69,97.40,1,0.849246,176.000011,395.999984,70.000023,1053.999996,391.730552,332.675523,2025-10-01 06:30:56,2025-10-01 06:33:52,2025-10-01 06:40:28,2025-10-01 06:41:38,2025-10-01 06:59:12
6,251001-01,2025-10-01,151.30,142.47,0,0.941639,212.000017,390.000017,168.999968,1119.000006,486.756030,458.348523,2025-10-01 05:59:11,2025-10-01 06:02:43,2025-10-01 06:09:13,2025-10-01 06:12:02,2025-10-01 06:30:41
7,250930-10,2025-09-30,194.44,163.10,1,0.838819,266.000006,NaN,NaN,1545.000029,453.064069,380.038828,2025-09-30 15:40:18,2025-09-30 15:44:44,None,2025-09-30 15:49:01,2025-09-30 16:14:46
8,250930-09,2025-09-30,102.68,84.81,0,0.825964,NaN,NaN,NaN,1512.000006,244.476189,201.928571,None,2025-09-30 15:12:14,None,2025-09-30 15:14:42,2025-09-30 15:39:54
9,250930-08,2025-09-30,108.91,102.68,0,0.942797,351.999982,501.000018,85.999976,818.000029,479.310496,451.892405,2025-09-30 14:41:41,2025-09-30 14:47:33,2025-09-30 14:55:54,2025-09-30 14:57:20,2025-09-30 15:10:58


In [33]:
# =========================================================================================
# === CELL A32: SCRIPT 1 - CREATE THE DEFINITIVE MONEY TABLE VIEW (Legacy A)
# Target: Notebook 0111a (Legacy)
# Version: 2.2 - Corrected EPH Logic
# =========================================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')
    print("INFO: Canonical db_file_path initialized for workspace.")

print("--- Architecting the definitive 'v_trip_final_kpis' view (Legacy A - v2.2) ---")

view_sql = """
CREATE VIEW v_trip_final_kpis AS
WITH
ImputedFlag AS (
    SELECT
        trip_id_legacy,
        MAX(is_imputed) AS is_imputed
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
),
Durations AS (
    SELECT
        trip_id_legacy,
        (julianday(t1_timestamp) - julianday(t0_timestamp)) * 86400.0 AS duration_lfr_sec,
        (julianday(COALESCE(t2_timestamp, t3_timestamp, t4_timestamp)) - julianday(t1_timestamp)) * 86400.0 AS duration_dtp_sec,
        (julianday(t3_timestamp) - julianday(t2_timestamp)) * 86400.0 AS duration_wfp_sec,
        (julianday(t4_timestamp) - julianday(t3_timestamp)) * 86400.0 AS duration_on_ride_sec,
        (julianday(MAX(
            IFNULL(t0_timestamp, '0000-01-01'), IFNULL(t1_timestamp, '0000-01-01'),
            IFNULL(t2_timestamp, '0000-01-01'), IFNULL(t3_timestamp, '0000-01-01'),
            IFNULL(t4_timestamp, '0000-01-01')
        )) -
         julianday(MIN(
            IFNULL(t0_timestamp, '9999-12-31'), IFNULL(t1_timestamp, '9999-12-31'),
            IFNULL(t2_timestamp, '9999-12-31'), IFNULL(t3_timestamp, '9999-12-31'),
            IFNULL(t4_timestamp, '9999-12-31')
        ))) * 86400.0 AS total_engagement_duration_sec
    FROM
        v_trip_funnel_wide
    GROUP BY
        trip_id_legacy
)
SELECT
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,
    d.duration_lfr_sec,
    d.duration_dtp_sec,
    d.duration_wfp_sec,
    d.duration_on_ride_sec,
    (COALESCE(d.duration_lfr_sec, 0) +
     COALESCE(d.duration_dtp_sec, 0) +
     COALESCE(d.duration_wfp_sec, 0) +
     COALESCE(d.duration_on_ride_sec, 0)) AS total_duration_sec,
    v.upfront_fare,
    v.realized_fare,
    CASE WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare ELSE NULL END AS spread_percentage,
    CASE WHEN d.duration_on_ride_sec > 0 THEN v.realized_fare / (d.duration_on_ride_sec / 3600.0) ELSE NULL END AS eph_on_ride,
    CASE WHEN d.total_engagement_duration_sec > 0 THEN v.realized_fare / (d.total_engagement_duration_sec / 3600.0) ELSE NULL END AS eph_total_time,
    i.is_imputed
FROM
    v_trip_funnel_wide v
LEFT JOIN
    Durations d ON v.trip_id_legacy = d.trip_id_legacy
LEFT JOIN
    ImputedFlag i ON v.trip_id_legacy = i.trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_final_kpis;")
        conn.execute(view_sql)
    print("SUCCESS: Final KPI view 'v_trip_final_kpis' (v2.2) created successfully in Legacy DB.")

    print("\n--- Displaying data from the last 10 trips for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = "SELECT * FROM v_trip_final_kpis ORDER BY trip_date DESC, trip_id_legacy DESC LIMIT 10;"
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"ERROR: View creation failed in Legacy. Reason: {e}")

--- Architecting the definitive 'v_trip_final_kpis' view (Legacy A - v2.2) ---
SUCCESS: Final KPI view 'v_trip_final_kpis' (v2.2) created successfully in Legacy DB.

--- Displaying data from the last 10 trips for verification ---


,trip_id_legacy,trip_date,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,total_duration_sec,upfront_fare,realized_fare,spread_percentage,eph_on_ride,eph_total_time,is_imputed
0,251001-07,2025-10-01,NaN,259.999998,48.000021,1992.000018,2300.000037,135.37,125.34,0.925907,226.518070,196.184345,0
1,251001-06,2025-10-01,111.000001,455.999981,NaN,1758.000000,2324.999982,165.68,148.14,0.894133,303.358362,229.378066,1
2,251001-05,2025-10-01,273.000008,392.000006,22.000001,1791.999976,2478.999992,120.57,106.95,0.887037,214.854914,155.312627,0
3,251001-04,2025-10-01,NaN,106.000029,101.999970,347.000009,555.000007,133.47,120.01,0.899153,1245.060487,778.443233,0
4,251001-03,2025-10-01,NaN,613.000014,69.999982,1893.999986,2576.999983,182.17,156.07,0.856727,296.648365,218.025613,0
5,251001-02,2025-10-01,176.000011,395.999984,70.000023,1053.999996,1696.000014,114.69,97.40,0.849246,332.675523,206.745281,1
6,251001-01,2025-10-01,212.000017,390.000017,168.999968,1119.000006,1890.000008,151.30,142.47,0.941639,458.348523,271.371427,0
7,250930-10,2025-09-30,266.000006,256.999974,NaN,1545.000029,2068.000008,194.44,163.10,0.838819,380.038828,283.926498,1
8,250930-09,2025-09-30,NaN,148.000002,NaN,1512.000006,1660.000008,102.68,84.81,0.825964,201.928571,183.925300,0
9,250930-08,2025-09-30,351.999982,501.000018,85.999976,818.000029,1757.000005,108.91,102.68,0.942797,451.892405,210.385884,0


In [34]:
# ==============================================================================
# === CELL A33: SCRIPT 2 - FINAL AUDIT (FORMATTED INSPECTION) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# ==============================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Running a formatted audit of the 'v_trip_final_kpis' view (Legacy A) ---")

try:
    with sqlite3.connect(db_file_path) as conn:
        # Extracting the last 10 trips based on date and legacy ID
        query = "SELECT * FROM v_trip_final_kpis ORDER BY trip_date DESC, trip_id_legacy DESC LIMIT 10;"
        df_audit = pd.read_sql_query(query, conn)
    print("SUCCESS: Data successfully extracted from the view.")

    # --- Data Transformation: HMS Formatting ---
    def format_seconds_to_hms(seconds):
        if pd.isna(seconds) or seconds < 0: return None
        seconds = int(seconds)
        hours, remainder = divmod(seconds, 3600)
        minutes, sec = divmod(remainder, 60)
        return f"{hours:02d}:{minutes:02d}:{sec:02d}"

    df_formatted = df_audit.copy()
    duration_cols = ['duration_lfr_sec', 'duration_dtp_sec', 'duration_wfp_sec', 'duration_on_ride_sec', 'total_duration_sec']
    for col in duration_cols:
        if col in df_formatted.columns:
            df_formatted[col] = df_formatted[col].apply(format_seconds_to_hms)

    # --- Final Presentation using Pandas Styler ---
    styler_format = {
        'eph_on_ride': '$ {:,.2f}', 
        'eph_total_time': '$ {:,.2f}',
        'spread_percentage': '{:.2%}', 
        'upfront_fare': '{:,.2f}', 
        'realized_fare': '{:,.2f}'
    }

    print("\n--- Displaying the LAST 10 formatted rows from the Money Table (Legacy) ---")
    display(df_formatted.style.format(styler_format, na_rep='-').hide(axis="index"))

except Exception as e:
    print(f"ERROR during formatted audit in Legacy: {e}")

--- Running a formatted audit of the 'v_trip_final_kpis' view (Legacy A) ---
SUCCESS: Data successfully extracted from the view.

--- Displaying the LAST 10 formatted rows from the Money Table (Legacy) ---


trip_id_legacy,trip_date,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,total_duration_sec,upfront_fare,realized_fare,spread_percentage,eph_on_ride,eph_total_time,is_imputed
251001-07,2025-10-01,-,00:04:19,00:00:48,00:33:12,00:38:20,135.37,125.34,92.59%,$ 226.52,$ 196.18,0
251001-06,2025-10-01,00:01:51,00:07:35,-,00:29:17,00:38:44,165.68,148.14,89.41%,$ 303.36,$ 229.38,1
251001-05,2025-10-01,00:04:33,00:06:32,00:00:22,00:29:51,00:41:18,120.57,106.95,88.70%,$ 214.85,$ 155.31,0
251001-04,2025-10-01,-,00:01:46,00:01:41,00:05:47,00:09:15,133.47,120.01,89.92%,"$ 1,245.06",$ 778.44,0
251001-03,2025-10-01,-,00:10:13,00:01:09,00:31:33,00:42:56,182.17,156.07,85.67%,$ 296.65,$ 218.03,0
251001-02,2025-10-01,00:02:56,00:06:35,00:01:10,00:17:33,00:28:16,114.69,97.40,84.92%,$ 332.68,$ 206.75,1
251001-01,2025-10-01,00:03:32,00:06:30,00:02:48,00:18:39,00:31:30,151.30,142.47,94.16%,$ 458.35,$ 271.37,0
250930-10,2025-09-30,00:04:26,00:04:16,-,00:25:45,00:34:28,194.44,163.10,83.88%,$ 380.04,$ 283.93,1
250930-09,2025-09-30,-,00:02:28,-,00:25:12,00:27:40,102.68,84.81,82.60%,$ 201.93,$ 183.93,0
250930-08,2025-09-30,00:05:51,00:08:21,00:01:25,00:13:38,00:29:17,108.91,102.68,94.28%,$ 451.89,$ 210.39,0


In [35]:
# ==============================================================================
# === CELL A34: DIAGNOSTIC AUDIT (v2) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# ==============================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Auditing for trips that have a completion time (T4) but are missing a start time (T3) (Legacy A) ---")

# The query targets the 'v_trip_funnel_wide' view for raw timestamp inspection.
query_sql = """
SELECT
    trip_id_legacy,
    t1_timestamp,
    t2_timestamp,
    t3_timestamp, -- Target: NULL
    t4_timestamp, -- Target: NOT NULL
    realized_fare
FROM
    v_trip_funnel_wide
WHERE
    t4_timestamp IS NOT NULL
    AND t3_timestamp IS NULL;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_anomaly_trips = pd.read_sql_query(query_sql, conn)

    print(f"\nAUDIT COMPLETE: Found {len(df_anomaly_trips)} trips that are COMPLETED but missing a START time.")

    if not df_anomaly_trips.empty:
        print("Displaying anomaly trips requiring manual imputation review:")
        display(df_anomaly_trips)
    else:
        print("SUCCESS: No data integrity violations found. All completed trips have a start time.")

except Exception as e:
    print(f"ERROR during diagnostic audit in Legacy: {e}")

--- Auditing for trips that have a completion time (T4) but are missing a start time (T3) (Legacy A) ---

AUDIT COMPLETE: Found 0 trips that are COMPLETED but missing a START time.
SUCCESS: No data integrity violations found. All completed trips have a start time.


In [36]:
# ===================================================================================================
# === CELL A35: THE FINAL CASCADE - ETL FOR TRIP EVENTS (GDRIVE LEGACY)
# Target: Notebook 0111a (Legacy) | Version: 7.2 (Pienza Parity)
# ===================================================================================================
print("\n--- PHASE 3: ENRICHING & LOADING trip_events (Legacy A) ---")

import pandas as pd
import sqlite3
import os
import gspread

# ==============================================================================
# --- 1. EXTRACT & PREPARE ---
# ==============================================================================
print("Executing self-contained extraction of all sources...")
try:
    # Source 1: 'trip_events' data from GTS-4 GSheet
    GTS_SHEET_NAME = "GTS-4"
    TRIP_EVENTS_TAB_NAME = "trip_events"
    workbook_gts = gc.open(GTS_SHEET_NAME)
    sheet_gts = workbook_gts.worksheet(TRIP_EVENTS_TAB_NAME)
    df_raw_events = pd.DataFrame(sheet_gts.get_all_records())
    
    df_raw_events.columns = df_raw_events.columns.str.lower().str.replace(' ', '_')
    df_raw_events.reset_index(inplace=True); 
    df_raw_events.rename(columns={'index': 'event_id'}, inplace=True); 
    df_raw_events['event_id'] += 1
    print(f"✅ Extracted and prepared {len(df_raw_events)} raw events from GSheet.")

    # Source 2 & 3: 'offers' and 'event_types' tables from Pienza DB
    with sqlite3.connect(db_file_path) as conn:
        offers_query = "SELECT offer_id, session_fk, offer_timestamp, upfront_fare FROM offers WHERE offer_action_fk = 1;"
        df_offers = pd.read_sql_query(offers_query, conn)
        # Keeping your specific date format for parity
        df_offers['offer_timestamp'] = pd.to_datetime(df_offers['offer_timestamp'], format='%Y:%m:%d %H:%M:%S', errors='coerce')
        df_event_types = pd.read_sql_query("SELECT * FROM event_types;", conn)
    print(f"✅ Extracted {len(df_offers)} accepted offers and {len(df_event_types)} event types.")

except Exception as e:
    print(f"❌ ERROR: Failed to extract source data. Reason: {e}")
    raise SystemExit("Halting execution.")

# ==============================================================================
# --- 2. TRANSFORM ---
# ==============================================================================
print("\n--- [T] TRANSFORMING and enriching 'trip_events' data ---")

df_staged_events = pd.merge(df_raw_events, df_event_types, left_on='event_types_id_fk', right_on='description', how='left')
df_staged_events.drop(columns=['event_types_id_fk', 'description', 'event_code', 'event_name'], inplace=True, errors='ignore')
df_staged_events.rename(columns={'event_type_id': 'event_types_id_fk'}, inplace=True)
df_staged_events.replace({'': None, 'N/A': None}, inplace=True)
df_staged_events['event_timestamp'] = pd.to_datetime(df_staged_events['event_timestamp'], errors='coerce')
df_staged_events['upfront_fare'] = pd.to_numeric(df_staged_events['upfront_fare'].astype(str).str.replace(r'[MX$,]', '', regex=True), errors='coerce')
df_staged_events.dropna(subset=['event_id', 'event_timestamp'], inplace=True)
df_staged_events['event_id'] = df_staged_events['event_id'].astype(int)
print("✅ Preliminary data cleaning and FK enrichment complete.")

# --- B. THE LINKING CASCADE ---
df_t1_events = df_staged_events[df_staged_events['event_types_id_fk'] == 2].copy()
df_offers_timed = df_offers[df_offers['offer_timestamp'].notna()].copy()
df_offers_orphan = df_offers[df_offers['offer_timestamp'].isna()].copy()

df_t1_events['day'] = df_t1_events['event_timestamp'].dt.strftime('%Y-%m-%d')
df_offers_timed['day'] = df_offers_timed['offer_timestamp'].dt.strftime('%Y-%m-%d')

print("\nExecuting Tier 1 & 2: Matching against TIMED offers...")
perfect_matches = pd.merge(df_t1_events, df_offers_timed, on=['day', 'upfront_fare'], how='inner')
if not perfect_matches.empty: perfect_matches['is_fuzzy_match'] = False
print(f"✅ Found {len(perfect_matches)} perfect timed matches.")

perfectly_matched_ids = perfect_matches['event_id'].unique() if not perfect_matches.empty else []
events_for_fuzzy_match = df_t1_events[~df_t1_events['event_id'].isin(perfectly_matched_ids)].copy()
if not events_for_fuzzy_match.empty:
    events_for_fuzzy_match['rounded_fare'] = events_for_fuzzy_match['upfront_fare'].fillna(0).astype(int)
    df_offers_timed['rounded_fare'] = df_offers_timed['upfront_fare'].fillna(0).astype(int)
    fuzzy_matches = pd.merge(events_for_fuzzy_match, df_offers_timed, on=['day', 'rounded_fare'], how='inner')
    if not fuzzy_matches.empty: fuzzy_matches['is_fuzzy_match'] = True
    print(f"✅ Found {len(fuzzy_matches)} fuzzy timed matches.")
else:
    fuzzy_matches = pd.DataFrame()

# --- TIER 3: MATCHING AGAINST ORPHAN OFFERS ---
timed_matched_ids = pd.concat([perfect_matches, fuzzy_matches])['event_id'].unique() if not pd.concat([perfect_matches, fuzzy_matches]).empty else []
events_for_orphan_match = df_t1_events[~df_t1_events['event_id'].isin(timed_matched_ids)]

print(f"\nExecuting Tier 3: Matching {len(events_for_orphan_match)} remaining events against ORPHAN offers...")
orphan_matches = pd.merge(events_for_orphan_match, df_offers_orphan, on='upfront_fare', how='inner')
if not orphan_matches.empty:
    orphan_matches.drop_duplicates(subset='event_id', keep='first', inplace=True)
    orphan_matches['is_imputed_link'] = True
    orphan_matches['is_fuzzy_match'] = False
print(f"✅ Found {len(orphan_matches)} imputed links (orphan matches).")

# --- C. FINAL RECONCILIATION & ANOMALY REPORT ---
all_matches = pd.concat([perfect_matches, fuzzy_matches, orphan_matches], ignore_index=True)

# SURGICAL FIX: Ensure columns exist to prevent KeyError if timed tiers are empty
for col in ['is_fuzzy_match', 'is_imputed_link']:
    if col not in all_matches.columns: all_matches[col] = False

final_matched_ids = all_matches['event_id'].unique() if not all_matches.empty else []
final_unmatched = df_t1_events[~df_t1_events['event_id'].isin(final_matched_ids)]

if not final_unmatched.empty:
    print(f"\n🛑 ANOMALY REPORT: Found {len(final_unmatched)} T1 events that are TRUE anomalies.")
else:
    print("\n✅ All T1 events were successfully matched.")

# ==============================================================================
# --- 3. FINAL MERGE & LOAD ---
# ==============================================================================
print("\n--- [L] Merging links and loading to database ---")
if not all_matches.empty:
    df_linked_final = pd.merge(df_staged_events, all_matches[['event_id', 'offer_id', 'session_fk', 'is_fuzzy_match', 'is_imputed_link']], on='event_id', how='left')
else:
    df_linked_final = df_staged_events.copy()
    df_linked_final['offer_id'], df_linked_final['session_fk'], df_linked_final['is_imputed_link'], df_linked_final['is_fuzzy_match'] = None, None, False, False

df_linked_final.rename(columns={'offer_id': 'offer_id_fk', 'session_fk': 'offers_session_fk'}, inplace=True)
df_linked_final['is_imputed_link'] = df_linked_final['is_imputed_link'].fillna(False).astype(bool)
df_linked_final['is_fuzzy_match'] = df_linked_final['is_fuzzy_match'].fillna(False).astype(bool)

final_columns = [
    'event_id', 'event_timestamp', 'event_lat', 'event_lon', 'event_address',
    'upfront_fare', 'realized_fare', 'source', 'is_imputed', 'comment',
    'trip_id_legacy', 'event_types_id_fk', 'offer_id_fk', 'offers_session_fk', 'is_fuzzy_match', 'is_imputed_link'
]
df_to_load = df_linked_final[[col for col in final_columns if col in df_linked_final.columns]]

try:
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql('trip_events', conn, if_exists='replace', index=False)
    print(f"✅ SUCCESS: Loaded {len(df_to_load)} records into 'trip_events'.")
    print("\n--- PHASE 3 COMPLETE ---")
except Exception as e:
    print(f"❌ ERROR: Failed to load 'trip_events' table. Reason: {e}")


--- PHASE 3: ENRICHING & LOADING trip_events (Legacy A) ---
Executing self-contained extraction of all sources...


✅ Extracted and prepared 1031 raw events from GSheet.
✅ Extracted 346 accepted offers and 5 event types.

--- [T] TRANSFORMING and enriching 'trip_events' data ---
✅ Preliminary data cleaning and FK enrichment complete.

Executing Tier 1 & 2: Matching against TIMED offers...
✅ Found 0 perfect timed matches.
✅ Found 0 fuzzy timed matches.

Executing Tier 3: Matching 259 remaining events against ORPHAN offers...
✅ Found 259 imputed links (orphan matches).

✅ All T1 events were successfully matched.

--- [L] Merging links and loading to database ---
✅ SUCCESS: Loaded 1031 records into 'trip_events'.

--- PHASE 3 COMPLETE ---


In [37]:
# ==============================================================================
# === CELL A36: Golden Propagation Patch (TD-002) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy) | Version: 2.1 (Pienza Canonical)
# ==============================================================================
# Author: Pienza Ex Machina
# Purpose: This cell reads the 'trip_events' table, applies the "Golden
#          Propagation" doctrine in memory to all link attributes, and overwrites 
#          the table with the fully-linked version.
# ==============================================================================

import pandas as pd
import sqlite3
import os

print("--- Initiating Golden Propagation Patch (Legacy A) ---")

# --- SETUP & CONNECTION ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

if not os.path.exists(db_file_path):
    raise FileNotFoundError(f"CRITICAL ERROR: Database file not found at path: {db_file_path}")

print("Database path verified successfully.")

# --- READ-MODIFY-WRITE PATTERN ---

# STEP 1: READ the current state of the trip_events table
try:
    with sqlite3.connect(db_file_path) as conn:
        events_to_patch_df = pd.read_sql_query('SELECT * FROM trip_events;', conn)
    print(f"SUCCESS: Read {len(events_to_patch_df)} events to be patched.")
except Exception as e:
    print(f"ERROR: Could not read from trip_events table. Ensure Cascade has been run. Details: {e}")
    raise

# STEP 2: MODIFY the DataFrame in memory to apply the propagation doctrine
print("Applying propagation logic across all linked attributes...")
events_to_patch_df = events_to_patch_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])

# Propagating all relevant offer linkage data across the entire trip lifecycle
link_columns = ['offer_id_fk', 'offers_session_fk', 'is_fuzzy_match', 'is_imputed_link']
for col in link_columns:
    if col in events_to_patch_df.columns:
        events_to_patch_df[col] = events_to_patch_df.groupby('trip_id_legacy')[col].transform(lambda x: x.ffill().bfill())

print("Propagation logic applied in memory.")

# In-line Unit Test: Assert that propagation was successful for a known trip before writing
known_trip_id = 'GTS-4-20251001-073003-B'
if known_trip_id in events_to_patch_df['trip_id_legacy'].values:
    propagated_ids = events_to_patch_df[events_to_patch_df['trip_id_legacy'] == known_trip_id]['offer_id_fk']
    assert propagated_ids.notna().all(), f"Validation failed: Nulls still exist for trip {known_trip_id}"
    assert propagated_ids.nunique() == 1, f"Validation failed: Multiple different offer_ids found for trip {known_trip_id}"
    print(f"Validation successful for test case trip: {known_trip_id}")
else:
    print(f"Warning: Test case trip ID '{known_trip_id}' not found in current data slice. Skipping validation.")

# STEP 3: WRITE the patched DataFrame back to the database, overwriting the old version.
print("Writing patched data back to pienzaba.db...")
try:
    with sqlite3.connect(db_file_path) as conn:
        events_to_patch_df.to_sql('trip_events', conn, if_exists='replace', index=False)
    print("SUCCESS: Patched data successfully written to Legacy database.")
except Exception as e:
    print(f"ERROR: Failed to write patched data. Details: {e}")

print("\n--- MISSION COMPLETE ---")
print("TD-002 Resolved. The 'trip_events' table has been successfully propagated.")

--- Initiating Golden Propagation Patch (Legacy A) ---
Database path verified successfully.
SUCCESS: Read 1031 events to be patched.
Applying propagation logic across all linked attributes...
Propagation logic applied in memory.
Writing patched data back to pienzaba.db...
SUCCESS: Patched data successfully written to Legacy database.

--- MISSION COMPLETE ---
TD-002 Resolved. The 'trip_events' table has been successfully propagated.


In [38]:
# ==============================================================================
# === CELL A37: The Post-ETL "Golden Link" Patch (Legacy A)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
print("--- Starting the Post-ETL 'Golden Link' Patch for trip_events (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# --- The Override Rulebook ---
# Format: { event_id_to_fix: 'correct_offer_id' }
override_rules = {
    55: 'OF00318'
    # Add any future manual corrections here.
}

print(f"Found {len(override_rules)} override rule(s) to apply.")

try:
    with sqlite3.connect(db_file_path) as conn:
        cursor = conn.cursor()

        # Fetch the session_fk for the corrected offers
        df_offers = pd.read_sql_query("SELECT offer_id, session_fk FROM offers;", conn)

        for event_id, correct_offer_id in override_rules.items():
            # Find the correct session_fk from our offers data
            try:
                correct_session_fk = df_offers.loc[df_offers['offer_id'] == correct_offer_id, 'session_fk'].iloc[0]
            except IndexError:
                print(f"WARNING: Could not find correct_offer_id '{correct_offer_id}' in the offers table. Skipping override for event {event_id}.")
                continue

            # The surgical UPDATE statement
            update_sql = """
            UPDATE trip_events
            SET
                offer_id_fk = ?,
                offers_session_fk = ?
            WHERE
                event_id = ?;
            """

            cursor.execute(update_sql, (correct_offer_id, correct_session_fk, event_id))
            print(f"  - Override Applied: Event ID {event_id} is now linked to Offer ID {correct_offer_id} (Session: {correct_session_fk}).")

        # Context manager auto-commits, but explicit commit ensures disk write before verification
        conn.commit()

    print("\nSUCCESS: Patching complete.")

    # --- Final Verification Step ---
    print("\n--- Verifying the fix for Event ID 55 ---")
    with sqlite3.connect(db_file_path) as conn:
        verify_query = "SELECT event_id, offer_id_fk, offers_session_fk FROM trip_events WHERE event_id = 55;"
        df_verify = pd.read_sql_query(verify_query, conn)
        display(df_verify)

except Exception as e:
    print(f"ERROR during patching process in Legacy: {e}")

--- Starting the Post-ETL 'Golden Link' Patch for trip_events (Legacy A) ---
Found 1 override rule(s) to apply.
  - Override Applied: Event ID 55 is now linked to Offer ID OF00318 (Session: SID0006).

SUCCESS: Patching complete.

--- Verifying the fix for Event ID 55 ---


,event_id,offer_id_fk,offers_session_fk
0,55,OF00318,SID0006


In [39]:
# ==============================================================================
# === CELL A38: DIAGNOSTIC CELL - The Pre-Join Audit (Legacy A)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
print("--- Performing a pre-join audit to diagnose the total join failure (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# --- [1] Replicate the Full Extraction Logic ---
print("Extracting all necessary source data...")
try:
    with sqlite3.connect(db_file_path) as conn:
        df_staged_events_raw = pd.read_sql_query("SELECT * FROM trip_events;", conn) 
        df_staged_events_raw['event_timestamp'] = pd.to_datetime(df_staged_events_raw['event_timestamp'], errors='coerce')

        offers_query = "SELECT offer_id, session_fk, offer_timestamp, upfront_fare FROM offers WHERE offer_action_fk = 1;"
        df_offers_raw = pd.read_sql_query(offers_query, conn)
        df_offers_raw['offer_timestamp'] = pd.to_datetime(df_offers_raw['offer_timestamp'], errors='coerce')

except Exception as e:
    print(f"ERROR: Failed to extract source data. Reason: {e}")
    raise SystemExit("Halting execution.")

# --- [2] Replicate the Preparation and Key Creation Logic ---
print("\n--- Preparing DataFrames for the join ---")
df_staged_events = df_staged_events_raw.copy()
df_offers = df_offers_raw.copy()

df_staged_events['upfront_fare'] = pd.to_numeric(df_staged_events['upfront_fare'].astype(str).str.replace(r'[MX$,]', '', regex=True), errors='coerce')
# Note: Ensuring the filter condition matches the exact type
df_t1_events = df_staged_events[df_staged_events['event_types_id_fk'] == 2].copy()

df_t1_events['day'] = df_t1_events['event_timestamp'].dt.strftime('%Y-%m-%d')
df_offers['day'] = df_offers['offer_timestamp'].dt.strftime('%Y-%m-%d')

# --- [3] THE DEFINITIVE AUDIT ---
print("\n--- AUDIT: Data types and sample values of JOIN KEYS ---")

print("\n--- `df_t1_events` (Left Side of Join) ---")
print("Data Types:")
print(df_t1_events[['day', 'upfront_fare']].info())
print("\nSample Data:")
display(df_t1_events[['day', 'upfront_fare']].head())

print("\n--- `df_offers` (Right Side of Join) ---")
print("Data Types:")
print(df_offers[['day', 'upfront_fare']].info())
print("\nSample Data:")
display(df_offers[['day', 'upfront_fare']].head())

--- Performing a pre-join audit to diagnose the total join failure (Legacy A) ---
Extracting all necessary source data...

--- Preparing DataFrames for the join ---

--- AUDIT: Data types and sample values of JOIN KEYS ---

--- `df_t1_events` (Left Side of Join) ---
Data Types:
<class 'pandas.core.frame.DataFrame'>
Index: 259 entries, 0 to 1027
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   day           259 non-null    object 
 1   upfront_fare  259 non-null    float64
dtypes: float64(1), object(1)
memory usage: 6.1+ KB
None

Sample Data:


,day,upfront_fare
0,2025-08-22,136.53
3,2025-08-22,162.00
6,2025-08-22,64.31
9,2025-08-22,146.00
12,2025-08-22,60.00



--- `df_offers` (Right Side of Join) ---
Data Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   day           346 non-null    object 
 1   upfront_fare  346 non-null    float64
dtypes: float64(1), object(1)
memory usage: 5.5+ KB
None

Sample Data:


,day,upfront_fare
0,2025-08-22,136.53
1,2025-08-22,162.00
2,2025-08-22,64.31
3,2025-08-22,146.00
4,2025-08-22,60.00


In [40]:
# ==============================================================================
# === CELL A39: The Final "Side-by-Side" Reconciliation Audit (Legacy A)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
print("--- Performing the final 'Side-by-Side' Reconciliation Audit (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# This query joins the final trip_events table back to the offers table
# to create a direct, side-by-side comparison for validation.
query_sql = """
SELECT
    te.trip_id_legacy,
    te.offer_id_fk,
    te.upfront_fare AS event_upfront_fare,
    o.upfront_fare AS linked_offer_upfront_fare,
    -- Calculate the difference to easily spot mismatches
    (te.upfront_fare - o.upfront_fare) AS fare_discrepancy,
    te.is_imputed_link
FROM
    trip_events te
-- Use a LEFT JOIN to ensure we see all trip events, even if a link failed
LEFT JOIN
    offers o ON te.offer_id_fk = o.offer_id
WHERE
    te.event_types_id_fk = 2 -- We only care about the T1 'Accepted' events that were linked
ORDER BY
    te.event_timestamp ASC;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_reconciliation = pd.read_sql_query(query_sql, conn)

    print("SUCCESS: Reconciliation query executed successfully.")
    print("Displaying a side-by-side comparison for all linked trips:")

    # Configure pandas to display all rows for rigorous auditing
    pd.set_option('display.max_rows', None)

    display(df_reconciliation)

    # --- Final Automated Check ---
    # Find any rows where the link was made but the fares diverge (using standard float tolerance)
    mismatch_count = df_reconciliation[df_reconciliation['fare_discrepancy'].abs() > 0.01].shape[0]

    print("\n--- FINAL VERDICT ---")
    if mismatch_count > 0:
        print(f"WARNING: Found {mismatch_count} linked records with a significant fare discrepancy. Manual review required.")
    else:
        print("SUCCESS: All linked records have a perfect fare match. The linkage is 100% validated.")

    # Reset display options to avoid cluttering future cells
    pd.reset_option('display.max_rows')

except Exception as e:
    print(f"ERROR during reconciliation audit in Legacy: {e}")

--- Performing the final 'Side-by-Side' Reconciliation Audit (Legacy A) ---
SUCCESS: Reconciliation query executed successfully.
Displaying a side-by-side comparison for all linked trips:


,trip_id_legacy,offer_id_fk,event_upfront_fare,linked_offer_upfront_fare,fare_discrepancy,is_imputed_link
0,250822-01,OF00003,136.53,136.53,0.0,1
1,250822-02,OF00012,162.00,162.00,0.0,1
2,250822-03,OF00032,64.31,64.31,0.0,1
3,250822-04,OF00034,146.00,146.00,0.0,1
4,250822-05,OF00035,60.00,60.00,0.0,1
5,250822-06,OF00101,108.67,108.67,0.0,1
6,250822-07,OF00113,170.00,170.00,0.0,1
7,250822-08,OF00130,173.44,173.44,0.0,1
8,250822-09,OF00167,89.00,89.00,0.0,1
9,250822-10,OF00174,202.00,202.00,0.0,1



--- FINAL VERDICT ---
SUCCESS: All linked records have a perfect fare match. The linkage is 100% validated.


In [41]:
# ==============================================================================
# === CELL A40: EMERGENCY AUDIT - Schema Ground Truth (Legacy A)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
print("--- Auditing the Canonical Schema of the 'raw_offers_ocr' table (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# PRAGMA table_info is the definitive SQLite command for schema inspection
query_sql = "PRAGMA table_info(raw_offers_ocr);"

try:
    # Opening in read-only (ro) mode to ensure zero risk of accidental modification
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Schema audit complete for Legacy database.")
    print("This is the definitive, canonical list of columns for 'raw_offers_ocr':")
    display(df_schema)

except Exception as e:
    print(f"ERROR: Audit query failed in Legacy. Reason: {e}")

--- Auditing the Canonical Schema of the 'raw_offers_ocr' table (Legacy A) ---

SUCCESS: Schema audit complete for Legacy database.
This is the definitive, canonical list of columns for 'raw_offers_ocr':


,cid,name,type,notnull,dflt_value,pk
0,0,ocr_id,TEXT,0,None,0
1,1,image_filename,TEXT,0,None,0
2,2,time_taken,TEXT,0,None,0
3,3,ride_type,TEXT,0,None,0
4,4,upfront_fare,TEXT,0,None,0
5,5,pickup_details,TEXT,0,None,0
6,6,pickup_address,TEXT,0,None,0
7,7,trip_details,TEXT,0,None,0
8,8,dropoff_address,TEXT,0,None,0
9,9,rider_rating,TEXT,0,None,0


In [42]:
# =========================================================================================
# === CELL A41: TASK 3 (v2.0) - LINK INTEGRITY AUDIT - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# =========================================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

print("--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Legacy A) ---")

# The query uses a LEFT JOIN to expose offers that failed to find their OCR counterpart
query_sql = """
SELECT
    CASE
        WHEN ocr.ocr_id IS NULL THEN 'Orphaned (Link Failed)'
        ELSE 'Linked Successfully'
    END AS link_status,
    COUNT(o.offer_id) AS number_of_offers
FROM
    offers o
LEFT JOIN
    raw_offers_ocr ocr ON o.ocr_fk = ocr.ocr_id
GROUP BY
    link_status;
"""

try:
    # Using read-only mode for the integrity audit
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_link_status = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Audit query executed.")
    print("Displaying link status count:")
    display(df_link_status)

    # Provide a clear, actionable summary based on the linkage results
    if 'Orphaned (Link Failed)' in df_link_status['link_status'].values:
        print("\nFINDING: One or more offers could not be linked back to their OCR source. Investigation required.")
    else:
        print("\nFINDING: 100% of offers are successfully linked to their raw OCR source. Data provenance is intact.")

except Exception as e:
    print(f"ERROR during audit: {e}")

--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Legacy A) ---

SUCCESS: Audit query executed.
Displaying link status count:


,link_status,number_of_offers
0,Orphaned (Link Failed),4765



FINDING: One or more offers could not be linked back to their OCR source. Investigation required.


In [43]:
# ==============================================================================
# === CELL A42: EMERGENCY AUDIT - Schema Ground Truth (Legacy A)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
print("--- Auditing the Canonical Schema of the 'offers' table (Legacy A) ---")

# --- Self-Contained Configuration (Paths Only) ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# PRAGMA table_info provides the authoritative physical schema from SQLite
query_sql = "PRAGMA table_info(offers);"

try:
    # Read-only connection to preserve database integrity
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Schema audit complete for Legacy database.")
    print("This is the definitive, canonical list of columns for the 'offers' table:")
    display(df_schema)

except Exception as e:
    print(f"ERROR: Audit query failed in Legacy. Reason: {e}")

--- Auditing the Canonical Schema of the 'offers' table (Legacy A) ---

SUCCESS: Schema audit complete for Legacy database.
This is the definitive, canonical list of columns for the 'offers' table:


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,0,None,0
1,1,session_fk,TEXT,0,None,0
2,2,ocr_fk,TEXT,0,None,0
3,3,image_content_hash,TEXT,0,None,0
4,4,offer_timestamp,TEXT,0,None,0
5,5,upfront_fare,REAL,0,None,0
6,6,time_to_pickup_sec,REAL,0,None,0
7,7,dist_to_pickup_km,REAL,0,None,0
8,8,est_trip_time_sec,REAL,0,None,0
9,9,est_trip_dist_km,REAL,0,None,0


In [44]:
# ==============================================================================
# === CELL A43: EMERGENCY AUDIT - FINALIZING SCHEMA GROUND TRUTH (Legacy A)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
# --- Self-Contained Configuration (Paths Only) ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

table_to_audit = 'offers'
print(f"\n--- Auditing the Canonical Schema of the '{table_to_audit}' table (Legacy A) ---")

query_sql = f"PRAGMA table_info({table_to_audit});"

try:
    # Read-only mode ensures zero risk of database mutation during audit
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Schema audit complete.")
    print(f"This is the definitive, canonical list of columns for the '{table_to_audit}' table:")
    display(df_schema)

except Exception as e:
    print(f"ERROR: Audit query failed in Legacy. Reason: {e}")


--- Auditing the Canonical Schema of the 'offers' table (Legacy A) ---

SUCCESS: Schema audit complete.
This is the definitive, canonical list of columns for the 'offers' table:


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,0,None,0
1,1,session_fk,TEXT,0,None,0
2,2,ocr_fk,TEXT,0,None,0
3,3,image_content_hash,TEXT,0,None,0
4,4,offer_timestamp,TEXT,0,None,0
5,5,upfront_fare,REAL,0,None,0
6,6,time_to_pickup_sec,REAL,0,None,0
7,7,dist_to_pickup_km,REAL,0,None,0
8,8,est_trip_time_sec,REAL,0,None,0
9,9,est_trip_dist_km,REAL,0,None,0


In [45]:
# ==============================================================================
# === CELL A44: DIAGNOSTIC - INVESTIGATING BROKEN OCR_FK LINK (Legacy A)
# Target: Notebook 0111a (Legacy)
# ==============================================================================
print("--- Performing a two-sided audit of the 'ocr_fk' link (Legacy A) ---")

# --- Self-Contained Configuration (Paths Only) ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

try:
    with sqlite3.connect(db_file_path) as conn:
        # [1] Source Analysis: Inspecting the values we are trying to join FROM
        print("\n--- [1] Sample of values in 'offers.ocr_fk' ---")
        query1 = "SELECT ocr_fk FROM offers LIMIT 10;"
        query1_nulls = "SELECT COUNT(*) FROM offers WHERE ocr_fk IS NULL;"

        df_from = pd.read_sql_query(query1, conn)
        null_count = conn.execute(query1_nulls).fetchone()[0]

        display(df_from)
        print(f"Number of NULL values in 'offers.ocr_fk': {null_count}")

        # [2] Target Analysis: Inspecting the values we are trying to join TO
        print("\n--- [2] Sample of values in 'raw_offers_ocr.ocr_id' ---")
        query2 = "SELECT ocr_id FROM raw_offers_ocr LIMIT 10;"
        df_to = pd.read_sql_query(query2, conn)
        display(df_to)

except Exception as e:
    print(f"ERROR: Diagnostic audit failed in Legacy. Details: {e}")

--- Performing a two-sided audit of the 'ocr_fk' link (Legacy A) ---

--- [1] Sample of values in 'offers.ocr_fk' ---


,ocr_fk
0,None
1,None
2,None
3,None
4,None
5,None
6,None
7,None
8,None
9,None


Number of NULL values in 'offers.ocr_fk': 4765

--- [2] Sample of values in 'raw_offers_ocr.ocr_id' ---


,ocr_id
0,OCR00001
1,OCR00002
2,OCR00003
3,OCR00004
4,OCR00005
5,OCR00006
6,OCR00007
7,OCR00008
8,OCR00009
9,OCR00010


In [46]:
# =========================================================================================
# === CELL A45: TASK 3 (v2.1 - Final Validation) - GDRIVE LEGACY
# Target: Notebook 0111a (Legacy)
# =========================================================================================
print("\n--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Legacy A) ---")

# --- Self-Contained Configuration (Paths Only) ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# This query determines if any records in the offers table lack a parent in the OCR table
query_sql = """
SELECT
    CASE
        WHEN ocr.ocr_id IS NULL THEN 'Orphaned (Link Failed)'
        ELSE 'Linked Successfully'
    END AS link_status,
    COUNT(o.offer_id) AS number_of_offers
FROM
    offers o
LEFT JOIN
    raw_offers_ocr ocr ON o.ocr_fk = ocr.ocr_id
GROUP BY
    link_status;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_link_status = pd.read_sql_query(query_sql, conn)

    print("\nSUCCESS: Audit query executed.")
    print("Displaying link status count:")
    display(df_link_status)

    # Actionable summary of data provenance
    if 'Orphaned (Link Failed)' in df_link_status['link_status'].values:
        print("\nFINDING: One or more offers could not be linked back to their OCR source. Investigation required.")
    else:
        print("\nFINDING: 100% of offers are successfully linked to their raw OCR source. Data provenance is intact.")

except Exception as e:
    print(f"ERROR during audit in Legacy: {e}")


--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' (Legacy A) ---

SUCCESS: Audit query executed.
Displaying link status count:


,link_status,number_of_offers
0,Orphaned (Link Failed),4765



FINDING: One or more offers could not be linked back to their OCR source. Investigation required.


In [47]:
# ==============================================================================
# === CELL A46: ETL FOR 'activity_earnings' (Legacy A)
# Target: Notebook 0111a (Legacy) | Version: 3.0 (Silent & Parity)
# ==============================================================================
print("--- PHASE 5 STARTED: ETL for 'activity_earnings' table (Legacy A) ---")

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

try: 
    gc
except NameError:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

PLATFORM_DATA_SHEET_NAME = "platform_data"
EARNINGS_TAB_NAME = "activity_earnings"
TARGET_TABLE = "activity_earnings"

# ==============================================================================
# --- 1. EXTRACT ---
# ==============================================================================
print("\n--- [E] EXTRACTING raw data from GSheet ---")
df_raw_earnings = None
try:
    workbook = gc.open(PLATFORM_DATA_SHEET_NAME)
    sheet = workbook.worksheet(EARNINGS_TAB_NAME)
    print(f"Successfully opened worksheet '{EARNINGS_TAB_NAME}'.")

    all_values = sheet.get_all_values()
    data_rows = all_values[1:] 

    canonical_headers = [
        'activity_earnings_id', 'activity_earnings_timestamp', 'product_category',
        'net_earning', 'details_url'
    ]

    df_raw_earnings = pd.DataFrame(data_rows)
    df_raw_earnings = df_raw_earnings.iloc[:, 0:len(canonical_headers)]
    df_raw_earnings.columns = canonical_headers
    print(f"SUCCESS: Extracted {len(df_raw_earnings)} rows using canonical headers.")

except Exception as e:
    print(f"ERROR during extraction: {e}")
    df_raw_earnings = None

# ==============================================================================
# --- 2. TRANSFORM ---
# ==============================================================================
if df_raw_earnings is not None:
    print("\n--- [T] TRANSFORMING and cleaning 'activity_earnings' data ---")
    df_transformed = df_raw_earnings.copy()

    df_transformed.replace({'': None, 'N/A': None, 'nan': None}, inplace=True)
    df_transformed['net_earning'] = pd.to_numeric(df_transformed['net_earning'].astype(str).str.replace(r'[MX$,]', '', regex=True).str.strip(), errors='coerce')
    df_transformed['activity_earnings_id'] = df_transformed['activity_earnings_id'].astype(str).str.replace('AE', '').pipe(pd.to_numeric, errors='coerce')

    def intelligent_date_parser(date_string):
        if not isinstance(date_string, str) or date_string.strip() == '': return pd.NaT
        try: 
            # infer_datetime_format removed to suppress FutureWarning
            return pd.to_datetime(date_string)
        except (ValueError, TypeError):
            try:
                s_cleaned = date_string.replace('st', '').replace('nd', '').replace('rd', '').replace('th', '')
                return pd.to_datetime(s_cleaned, format='%A, %B %d, %Y\n%H:%M')
            except (ValueError, TypeError): return pd.NaT

    df_transformed['activity_earnings_timestamp'] = df_transformed['activity_earnings_timestamp'].apply(intelligent_date_parser)
    
    initial_rows = len(df_transformed)
    df_transformed.dropna(subset=['activity_earnings_timestamp', 'activity_earnings_id'], inplace=True)
    df_transformed['activity_earnings_id'] = df_transformed['activity_earnings_id'].astype(int)
    print(f"Dropped {initial_rows - len(df_transformed)} rows with missing critical data.")

    df_transformed.sort_values('activity_earnings_timestamp', inplace=True, ascending=True)
    df_transformed.reset_index(drop=True, inplace=True)
    
    df_to_load = df_transformed[['activity_earnings_id', 'activity_earnings_timestamp', 'product_category', 'net_earning', 'details_url']]

    print(f"\n--- Displaying diagnostics for {len(df_to_load)} records ---")
    if not df_to_load.empty:
        print("\nFirst 20 records (True chronological start):")
        display(df_to_load.head(20))
        print("\nLast 20 records (True chronological end):")
        display(df_to_load.tail(20))

    # ==============================================================================
    # --- 3. LOAD ---
    # ==============================================================================
    print(f"\n--- [L] LOADING data into database ---")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_to_load.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)
        print(f"SUCCESS: Loaded {len(df_to_load)} records into '{TARGET_TABLE}'.")
    except Exception as e:
        print(f"ERROR during load operation: {e}")

--- PHASE 5 STARTED: ETL for 'activity_earnings' table (Legacy A) ---

--- [E] EXTRACTING raw data from GSheet ---


Successfully opened worksheet 'activity_earnings'.
SUCCESS: Extracted 3446 rows using canonical headers.

--- [T] TRANSFORMING and cleaning 'activity_earnings' data ---
Dropped 0 rows with missing critical data.

--- Displaying diagnostics for 3446 records ---

First 20 records (True chronological start):


,activity_earnings_id,activity_earnings_timestamp,product_category,net_earning,details_url
0,1,2023-05-29 12:18:00,UberX,36.98,None
1,2,2023-05-29 12:39:00,UberX,41.98,None
2,3,2023-05-29 13:05:00,UberX,75.89,None
3,4,2023-05-29 13:47:00,UberX,69.04,None
4,5,2023-05-29 19:49:00,UberX,49.11,None
5,6,2023-05-29 20:21:00,UberX,34.41,None
6,7,2023-05-29 20:32:00,UberX,53.77,None
7,8,2023-06-05 11:38:00,UberX,82.16,None
8,9,2023-06-05 17:00:00,UberX,61.14,None
9,10,2023-06-05 17:31:00,UberX,56.38,None



Last 20 records (True chronological end):


,activity_earnings_id,activity_earnings_timestamp,product_category,net_earning,details_url
3426,3427,2025-09-29 17:11:00,Uber Priority,194.62,View Details
3427,3428,2025-09-30 06:02:00,UberX,110.64,View Details
3428,3429,2025-09-30 06:37:00,Uber Priority,133.01,View Details
3429,3430,2025-09-30 07:20:00,Uber Priority,90.55,View Details
3430,3431,2025-09-30 07:47:00,UberX,0.00,View Details
3431,3432,2025-09-30 08:16:00,Comfort,12.14,View Details
3432,3433,2025-09-30 08:36:00,Uber Priority,129.04,View Details
3433,3434,2025-09-30 13:41:00,Comfort,62.16,View Details
3434,3435,2025-09-30 14:04:00,UberX,89.87,View Details
3435,3436,2025-09-30 14:44:00,Comfort,88.81,View Details



--- [L] LOADING data into database ---
SUCCESS: Loaded 3446 records into 'activity_earnings'.


In [48]:
# ====================================================================================
# === CELL A47: PHASE 6 - ETL FOR 'lifetime_trips' TABLE (Legacy A)
# Target: Notebook 0111a (Legacy) | Version: 1.4 (Pienza Canonical)
# ====================================================================================
print("--- PHASE 6 STARTED: ETL for 'lifetime_trips' table (Legacy A) ---")

# --- Self-Contained Configuration (Paths Only) ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

try:
    gc
except NameError:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

PLATFORM_DATA_SHEET_NAME = "platform_data"
LIFETIME_TRIPS_TAB_NAME = "lifetime_trips"
TARGET_TABLE = "lifetime_trips"

# ==============================================================================
# --- 1. EXTRACT (Resilient Header Fix) ---
# ==============================================================================
print("\n--- [E] EXTRACTING raw data from GSheet ---")
df_raw_trips = None
try:
    workbook = gc.open(PLATFORM_DATA_SHEET_NAME)
    sheet = workbook.worksheet(LIFETIME_TRIPS_TAB_NAME)
    print(f"Successfully opened worksheet '{LIFETIME_TRIPS_TAB_NAME}'.")

    all_values = sheet.get_all_values()
    data_rows = all_values[1:]

    canonical_headers = [
        'lifetime_trips_id', 'global_product_name', 'status', 'request_timestamp',
        'pickup_timestamp', 'dropoff_timestamp', 'trip_distance_miles',
        'trip_duration_seconds', 'base_fare', 'original_fare',
        'cancellation_fee', 'currency_code', 'vehicle_uuid', 'license_plate'
    ]

    df_raw_trips = pd.DataFrame(data_rows)
    df_raw_trips = df_raw_trips.iloc[:, 0:len(canonical_headers)]
    df_raw_trips.columns = canonical_headers
    print(f"SUCCESS: Extracted {len(df_raw_trips)} rows using canonical headers.")

except Exception as e:
    print(f"ERROR during extraction: {e}")
    df_raw_trips = None

# ==============================================================================
# --- 2. TRANSFORM ---
# ==============================================================================
if df_raw_trips is not None:
    print("\n--- [T] TRANSFORMING and cleaning 'lifetime_trips' data ---")
    df_transformed = df_raw_trips.copy()

    df_transformed.replace({'': None, 'N/A': None, 'nan': None}, inplace=True)
    
    # Strip LT prefix and convert to numeric
    df_transformed['lifetime_trips_id'] = pd.to_numeric(df_transformed['lifetime_trips_id'].astype(str).str.replace('LT', ''), errors='coerce')
    
    numeric_cols = ['trip_distance_miles', 'trip_duration_seconds', 'base_fare', 'original_fare', 'cancellation_fee']
    for col in numeric_cols:
        if col in df_transformed.columns:
            df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
            
    timestamp_cols = ['request_timestamp', 'pickup_timestamp', 'dropoff_timestamp']
    for col in timestamp_cols:
        if col in df_transformed.columns:
            df_transformed[col] = pd.to_datetime(df_transformed[col], errors='coerce')
    print("Data types cleaned and coerced.")

    df_transformed.dropna(subset=['lifetime_trips_id', 'request_timestamp'], inplace=True)
    df_transformed['lifetime_trips_id'] = df_transformed['lifetime_trips_id'].astype(int)
    print("Dropped rows with missing critical data and cast PK to integer.")

    final_columns = [
        'lifetime_trips_id', 'global_product_name', 'status',
        'request_timestamp', 'pickup_timestamp', 'dropoff_timestamp',
        'trip_distance_miles', 'trip_duration_seconds', 'base_fare',
        'original_fare', 'cancellation_fee', 'currency_code',
        'vehicle_uuid', 'license_plate'
    ]
    df_to_load = df_transformed[[col for col in final_columns if col in df_transformed.columns]].copy()
    
    # Ensure optional columns exist
    for col in ['vehicle_uuid', 'license_plate']:
        if col not in df_to_load.columns:
            df_to_load[col] = None
    print("Final schema alignment complete.")

    # ==============================================================================
    # --- 3. LOAD ---
    # ==============================================================================
    print(f"\n--- [L] LOADING data into database ---")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_to_load.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)
        print(f"SUCCESS: Loaded {len(df_to_load)} records into '{TARGET_TABLE}'.")
        print("\n--- PHASE 6 COMPLETE ---")
    except Exception as e:
        print(f"ERROR during load operation: {e}")
else:
    print("\nHalting execution due to Extraction error.")

--- PHASE 6 STARTED: ETL for 'lifetime_trips' table (Legacy A) ---

--- [E] EXTRACTING raw data from GSheet ---
Successfully opened worksheet 'lifetime_trips'.
SUCCESS: Extracted 3446 rows using canonical headers.

--- [T] TRANSFORMING and cleaning 'lifetime_trips' data ---
Data types cleaned and coerced.
Dropped rows with missing critical data and cast PK to integer.
Final schema alignment complete.

--- [L] LOADING data into database ---
SUCCESS: Loaded 3446 records into 'lifetime_trips'.

--- PHASE 6 COMPLETE ---


In [49]:
# ==============================================================================
# === CELL A48: DIAGNOSTIC CELL: Inspect First & Last 20 Records (Legacy A)
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
print("--- 🔍 Running Diagnostic: Inspecting 'lifetime_trips' table (Legacy A) ---")

# --- Self-Contained Path Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# Query the entire table ordered by time
query = "SELECT * FROM lifetime_trips ORDER BY request_timestamp ASC;"

try:
    with sqlite3.connect(db_file_path) as conn:
        df_lifetime_trips_audit = pd.read_sql_query(query, conn)

    print("\n✅ SUCCESS: Query executed.")
    print(f"📊 Total records found in 'lifetime_trips': {len(df_lifetime_trips_audit)}")

    if not df_lifetime_trips_audit.empty:
        print("\n--- ⏳ First 20 records (Chronological Start) ---")
        display(df_lifetime_trips_audit.head(20))

        print("\n--- 🏁 Last 20 records (Chronological End) ---")
        display(df_lifetime_trips_audit.tail(20))

        print("\n--- 🛠️ Schema Verification ---")
        df_lifetime_trips_audit.info()
    else:
        print("\n⚠️ WARNING: The 'lifetime_trips' table is empty.")

except Exception as e:
    print(f"❌ ERROR: Query failed. Reason: {e}")

--- 🔍 Running Diagnostic: Inspecting 'lifetime_trips' table (Legacy A) ---

✅ SUCCESS: Query executed.
📊 Total records found in 'lifetime_trips': 3446

--- ⏳ First 20 records (Chronological Start) ---


,lifetime_trips_id,global_product_name,status,request_timestamp,pickup_timestamp,dropoff_timestamp,trip_distance_miles,trip_duration_seconds,base_fare,original_fare,cancellation_fee,currency_code,vehicle_uuid,license_plate
0,1,UberX,completed,2023-05-29 12:18:42+00:00,2023-05-29 12:23:45+00:00,2023-05-29 12:38:02+00:00,2.005310,857.0,6.87,69.99,0.0,MXN,None,None
1,2,UberX,completed,2023-05-29 12:39:17+00:00,2023-05-29 12:47:04+00:00,2023-05-29 12:58:27+00:00,1.892561,683.0,6.87,54.49,0.0,MXN,None,None
2,3,UberX,completed,2023-05-29 13:05:37+00:00,2023-05-29 13:18:17+00:00,2023-05-29 13:47:01+00:00,4.338404,1717.0,6.87,97.86,0.0,MXN,None,None
3,4,UberX,completed,2023-05-29 13:47:56+00:00,2023-05-29 13:53:08+00:00,2023-05-29 14:13:29+00:00,2.944863,1221.0,6.87,79.90,0.0,MXN,None,None
4,5,UberX,completed,2023-05-29 19:49:52+00:00,2023-05-29 20:04:28+00:00,2023-05-29 20:15:42+00:00,2.301361,674.0,6.87,71.39,0.0,MXN,None,None
5,6,UberX,completed,2023-05-29 20:21:57+00:00,2023-05-29 20:27:00+00:00,2023-05-29 20:36:23+00:00,2.547121,563.0,6.87,59.91,0.0,MXN,None,None
6,7,UberX,completed,2023-05-29 20:32:55+00:00,2023-05-29 20:39:51+00:00,2023-05-29 20:50:30+00:00,1.561299,639.0,6.87,49.90,0.0,MXN,None,None
7,8,UberX,completed,2023-06-05 11:38:51+00:00,2023-06-05 11:41:21+00:00,2023-06-05 12:10:28+00:00,6.361610,1747.0,6.87,149.93,0.0,MXN,None,None
8,9,UberX,completed,2023-06-05 17:00:03+00:00,2023-06-05 17:08:11+00:00,2023-06-05 17:32:03+00:00,4.397000,1432.0,6.87,99.96,0.0,MXN,None,None
9,10,UberX,completed,2023-06-05 17:31:28+00:00,2023-06-05 17:43:27+00:00,2023-06-05 17:49:39+00:00,1.119857,372.0,6.87,69.91,0.0,MXN,None,None



--- 🏁 Last 20 records (Chronological End) ---


,lifetime_trips_id,global_product_name,status,request_timestamp,pickup_timestamp,dropoff_timestamp,trip_distance_miles,trip_duration_seconds,base_fare,original_fare,cancellation_fee,currency_code,vehicle_uuid,license_plate
3426,3427,UberX,completed,2025-09-29 17:11:44+00:00,2025-09-29 17:28:26+00:00,2025-09-29 18:34:59+00:00,7.413063,3993.0,10.73,321.50,0.00,MXN,None,None
3427,3428,UberX,completed,2025-09-30 06:02:33+00:00,2025-09-30 06:08:46+00:00,2025-09-30 06:36:34+00:00,9.793376,1668.0,10.73,189.93,0.00,MXN,None,None
3428,3429,UberX,completed,2025-09-30 06:37:48+00:00,2025-09-30 06:52:52+00:00,2025-09-30 07:14:47+00:00,5.871442,1315.0,10.73,153.54,0.00,MXN,None,None
3429,3430,UberX,completed,2025-09-30 07:20:36+00:00,2025-09-30 07:29:12+00:00,2025-09-30 07:50:08+00:00,4.478963,1256.0,10.73,148.32,0.00,MXN,None,None
3430,3431,UberX,rider_canceled,2025-09-30 07:47:13+00:00,None,None,0.000000,0.0,NaN,0.00,0.00,MXN,None,None
3431,3432,Mid-Tier,rider_canceled,2025-09-30 08:16:56+00:00,None,None,0.000000,0.0,NaN,16.21,13.97,MXN,None,None
3432,3433,UberX,completed,2025-09-30 08:36:22+00:00,2025-09-30 08:47:05+00:00,2025-09-30 09:13:16+00:00,3.463705,1570.0,10.73,153.46,0.00,MXN,None,None
3433,3434,Mid-Tier,completed,2025-09-30 13:41:46+00:00,2025-09-30 13:47:51+00:00,2025-09-30 14:03:50+00:00,2.188194,959.0,13.30,121.51,0.00,MXN,None,None
3434,3435,UberX,completed,2025-09-30 14:04:33+00:00,2025-09-30 14:10:06+00:00,2025-09-30 14:41:07+00:00,6.237990,1862.0,10.73,202.07,0.00,MXN,None,None
3435,3436,Mid-Tier,completed,2025-09-30 14:44:42+00:00,2025-09-30 14:57:34+00:00,2025-09-30 15:10:19+00:00,3.370230,764.0,13.30,197.50,0.00,MXN,None,None



--- 🛠️ Schema Verification ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3446 entries, 0 to 3445
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   lifetime_trips_id      3446 non-null   int64  
 1   global_product_name    3446 non-null   object 
 2   status                 3446 non-null   object 
 3   request_timestamp      3446 non-null   object 
 4   pickup_timestamp       3402 non-null   object 
 5   dropoff_timestamp      3402 non-null   object 
 6   trip_distance_miles    3446 non-null   float64
 7   trip_duration_seconds  3444 non-null   float64
 8   base_fare              3402 non-null   float64
 9   original_fare          3444 non-null   float64
 10  cancellation_fee       3446 non-null   float64
 11  currency_code          3444 non-null   object 
 12  vehicle_uuid           0 non-null      object 
 13  license_plate          0 non-null      object 
dtypes: float64(5), int64(1),

In [50]:
# =================================================================================================
# === CELL A49: TASK A - FORGE THE "GOLDEN LINK" (Legacy A)
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db | Version: 1.5
# =================================================================================================
print("--- 🛠️ TASK A STARTED: Forging the 'Golden Link' using Direct ID Match (Legacy A) ---")

# --- Self-Contained Path Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

try:
    with sqlite3.connect(db_file_path) as conn:
        print(f"📂 Extracting source tables from {os.path.basename(db_file_path)}...")
        df_earnings = pd.read_sql_query("SELECT * FROM activity_earnings;", conn)
        df_earnings['activity_earnings_timestamp'] = pd.to_datetime(df_earnings['activity_earnings_timestamp'])

        df_lifetime = pd.read_sql_query("SELECT lifetime_trips_id FROM lifetime_trips;", conn)
        df_lifetime['lifetime_trips_id'] = pd.to_numeric(df_lifetime['lifetime_trips_id'], errors='coerce') 
    
    print(f"✅ Extracted {len(df_earnings)} earnings records and {len(df_lifetime)} lifetime trips.")

    # --- [2] The Direct, Perfect Inner Join ---
    print("\n--- 🔗 [T] Performing Direct INNER JOIN on Primary Keys (1:1 Match) ---")

    df_linked = pd.merge(
        df_earnings,
        df_lifetime,
        left_on='activity_earnings_id',
        right_on='lifetime_trips_id',
        how='inner'
    )

    print("✅ Direct ID Join complete. Only perfectly matched records proceed.")

    # --- [3] Final Schema Alignment ---
    df_linked.rename(columns={'lifetime_trips_id': 'lifetime_trips_fk'}, inplace=True)

    final_columns = [
        'activity_earnings_id',
        'lifetime_trips_fk',
        'activity_earnings_timestamp',
        'product_category',
        'net_earning',
        'details_url'
    ]

    df_to_load = df_linked[[col for col in final_columns if col in df_linked.columns]].copy()
    print("📋 Schema columns finalized for loading.")

    # --- [4] Load to Database ---
    print(f"\n--- 💾 [L] LOADING enriched data back into 'activity_earnings' table ---")
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql('activity_earnings', conn, if_exists='replace', index=False)
    
    print(f"✅ SUCCESS: Loaded {len(df_to_load)} records into 'activity_earnings' with new FK.")
    print("\n--- 🏁 TASK A COMPLETE ---")

except Exception as e:
    print(f"❌ ERROR during ETL: {e}")

# --- Final Verification ---
print("\n--- 🧐 Verifying a sample of the newly linked data ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        verify_query = "SELECT * FROM activity_earnings ORDER BY activity_earnings_id DESC LIMIT 5;"
        df_verify = pd.read_sql_query(verify_query, conn)
        display(df_verify)
except Exception as e:
    print(f"❌ ERROR during verification: {e}")

--- 🛠️ TASK A STARTED: Forging the 'Golden Link' using Direct ID Match (Legacy A) ---
📂 Extracting source tables from pienzaba.db...
✅ Extracted 3446 earnings records and 3446 lifetime trips.

--- 🔗 [T] Performing Direct INNER JOIN on Primary Keys (1:1 Match) ---
✅ Direct ID Join complete. Only perfectly matched records proceed.
📋 Schema columns finalized for loading.

--- 💾 [L] LOADING enriched data back into 'activity_earnings' table ---
✅ SUCCESS: Loaded 3446 records into 'activity_earnings' with new FK.

--- 🏁 TASK A COMPLETE ---

--- 🧐 Verifying a sample of the newly linked data ---


,activity_earnings_id,lifetime_trips_fk,activity_earnings_timestamp,product_category,net_earning,details_url
0,3446,3446,2025-10-01 09:06:00,Comfort,142.17,View Details
1,3445,3445,2025-10-01 08:35:00,Business Comfort,148.14,View Details
2,3444,3444,2025-10-01 08:27:00,Comfort,0.00,View Details
3,3443,3443,2025-10-01 07:56:00,UberX,104.96,View Details
4,3442,3442,2025-10-01 07:33:00,Black,117.32,View Details


In [51]:
# ==============================================================================
# === CELL A50: FORENSIC - Hunting for Whitespace-Only 'outcome_fk' Values ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
print("--- 🕵️‍♂️ Forensic Audit: GSheet Whitespace Hunter (Legacy A) ---")

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

# Naming Parity
DIAMOND_SHEET_NAME = "raw_Offers" 
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # 1. Verify Google Client
    if 'gc' not in locals() and 'gc' not in globals():
        raise NameError("Google Client 'gc' is not initialized. Run the auth cell!")

    # 2. Open Workbook and Tab
    wb = gc.open(DIAMOND_SHEET_NAME)
    tabs = [s.title for s in wb.worksheets()]
    
    if DIAMOND_TAB_NAME not in tabs:
        print(f"❌ ERROR: Tab '{DIAMOND_TAB_NAME}' not found in '{DIAMOND_SHEET_NAME}'.")
    else:
        sheet = wb.worksheet(DIAMOND_TAB_NAME)
        data = sheet.get_all_values()
        
        if len(data) > 1:
            df_source = pd.DataFrame(data[1:], columns=data[0])
            df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')
            
            # --- THE HUNTER ---
            if 'outcome_fk' in df_source.columns:
                anomaly_mask = (df_source['outcome_fk'] != '') & (df_source['outcome_fk'].astype(str).str.strip() == '')
                df_anomalies = df_source[anomaly_mask]

                print(f"🔍 Anomaly hunt complete. Found {len(df_anomalies)} records containing only whitespace.")
                if not df_anomalies.empty:
                    display(df_anomalies[['offer_id', 'outcome_fk']])
                else:
                    print("✨ Source is clean.")
            else:
                print(f"❌ ERROR: Column 'outcome_fk' not found in tab '{DIAMOND_TAB_NAME}'.")
        else:
            print("⚠️ Sheet is empty.")

except Exception as e:
    print(f"❌ FORENSIC ERROR: {e}")

--- 🕵️‍♂️ Forensic Audit: GSheet Whitespace Hunter (Legacy A) ---


🔍 Anomaly hunt complete. Found 0 records containing only whitespace.
✨ Source is clean.


In [52]:
# ==============================================================================
# === CELL A51: DIAGNOSTIC - "ID Set" Reconciliation for 'outcome_fk' ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
print("--- Performing 'ID Set' Reconciliation for 'outcome_fk' (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Get the ID set from the GSheet Source ---
    print(f"Extracting ID set from GSheet: '{DIAMOND_SHEET_NAME}'...")
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    # Filter for non-empty 'outcome_fk' and get the set of 'offer_id's
    gsheet_ids = set(df_source[df_source['outcome_fk'] != '']['offer_id'])
    print(f"✅ Found {len(gsheet_ids)} offer_ids with non-empty 'outcome_fk' in GSheet.")

    # --- [2] Get the ID set from the Database Destination ---
    print(f"Extracting ID set from {os.path.basename(db_file_path)}...")
    with sqlite3.connect(db_file_path) as conn:
        db_query = "SELECT offer_id FROM offers WHERE outcome_fk IS NOT NULL;"
        df_db = pd.read_sql_query(db_query, conn)

    db_ids = set(df_db['offer_id'])
    print(f"✅ Found {len(db_ids)} offer_ids with non-NULL 'outcome_fk' in the database.")

    # --- [3] THE RECONCILIATION ---
    print("\n--- Performing Set Difference Analysis ---")

    # Find IDs that are in the GSheet set but NOT in the database set
    discrepancy_ids = gsheet_ids - db_ids

    if not discrepancy_ids:
        print("\n✅ SUCCESS: The sets are identical. Reconciliation is perfect.")
    else:
        print(f"\n🛑 ANOMALY DETECTED: Found {len(discrepancy_ids)} 'offer_id's present in the GSheet count but missing from the database count.")
        print("This is definitive proof that these are the records with whitespace-only 'outcome_fk' values.")
        print("\nAnomaly List (offer_id):")
        for offer_id in sorted(list(discrepancy_ids)):
            print(offer_id)

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Performing 'ID Set' Reconciliation for 'outcome_fk' (Legacy A) ---
Extracting ID set from GSheet: 'raw_Offers'...


✅ Found 347 offer_ids with non-empty 'outcome_fk' in GSheet.
Extracting ID set from pienzaba.db...
✅ Found 347 offer_ids with non-NULL 'outcome_fk' in the database.

--- Performing Set Difference Analysis ---

✅ SUCCESS: The sets are identical. Reconciliation is perfect.


In [53]:
# ==============================================================================
# === CELL A52: DIAGNOSTIC - "ID Set" Reconciliation for 'pickup_address' ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
print("--- Performing 'ID Set' Reconciliation for 'pickup_address' (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Get the ID set from the GSheet Source ---
    print(f"Extracting ID set from GSheet: '{DIAMOND_SHEET_NAME}'...")
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    # Filter for non-empty 'pickup_address' and get the set of 'offer_id's
    gsheet_ids = set(df_source[df_source['pickup_address'] != '']['offer_id'])
    print(f"✅ Found {len(gsheet_ids)} offer_ids with non-empty 'pickup_address' in GSheet.")

    # --- [2] Get the ID set from the Database Destination ---
    print(f"Extracting ID set from {os.path.basename(db_file_path)}...")
    with sqlite3.connect(db_file_path) as conn:
        db_query = "SELECT offer_id FROM offers WHERE pickup_address IS NOT NULL AND pickup_address != '';"
        df_db = pd.read_sql_query(db_query, conn)

    db_ids = set(df_db['offer_id'])
    print(f"✅ Found {len(db_ids)} offer_ids with non-NULL 'pickup_address' in the database.")

    # --- [3] THE RECONCILIATION ---
    print("\n--- Performing Set Difference Analysis ---")

    discrepancy_ids = gsheet_ids - db_ids

    if not discrepancy_ids:
        print("\n✅ SUCCESS: The sets are identical. Reconciliation is perfect.")
    else:
        print(f"\n🛑 ANOMALY DETECTED: Found {len(discrepancy_ids)} 'offer_id'(s) present in the GSheet count but missing from the database count.")
        print("This is the record with the whitespace-only 'pickup_address' value.")
        print("\nAnomaly (offer_id):")
        for offer_id in sorted(list(discrepancy_ids)):
            print(offer_id)

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Performing 'ID Set' Reconciliation for 'pickup_address' (Legacy A) ---
Extracting ID set from GSheet: 'raw_Offers'...
✅ Found 4757 offer_ids with non-empty 'pickup_address' in GSheet.
Extracting ID set from pienzaba.db...
✅ Found 4757 offer_ids with non-NULL 'pickup_address' in the database.

--- Performing Set Difference Analysis ---

✅ SUCCESS: The sets are identical. Reconciliation is perfect.


In [54]:
# ==============================================================================
# === CELL A53: DIAGNOSTIC - The "Boolean Interrogation" ===
# Target: Notebook 0111a (Legacy) | Source: GSheet
# ==============================================================================
print("--- Interrogating the true format of the 'is_surge' column from the GSheet (Legacy A) ---")

import pandas as pd

# --- Configuration ---
DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # 1. Extract the raw data (Assume 'gc' from A1 is live)
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    
    # get_all_records() is used to match the Colab logic perfectly
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')
    print(f"Extracted {len(df_source)} rows from the GSheet source.")

    # 2. THE INTERROGATION
    if 'is_surge' in df_source.columns:
        unique_values = df_source['is_surge'].unique()
        print("\n✅ Interrogation complete.")
        print("These are the unique raw values found in the 'is_surge' column:")
        print(unique_values)
        print("\nData type of the column (Object usually indicates mixed/string strings):")
        print(df_source['is_surge'].dtype)
    else:
        print(f"🛑 ERROR: 'is_surge' column not found in tab '{DIAMOND_TAB_NAME}'.")

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Interrogating the true format of the 'is_surge' column from the GSheet (Legacy A) ---
Extracted 4765 rows from the GSheet source.

✅ Interrogation complete.
These are the unique raw values found in the 'is_surge' column:
['TRUE' 'FALSE']

Data type of the column (Object usually indicates mixed/string strings):
object


In [55]:
# ==============================================================================
# === CELL A54: DIAGNOSTIC - "ID Set" Reconciliation for 'special_note_raw' ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
print("--- Performing 'ID Set' Reconciliation for 'special_note_raw' (Legacy A) ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/workspaces/pienza'
    db_file_path = os.path.join(project_root, 'data/big_bang/pienzaba.db')

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Get the ID set from the GSheet Source ---
    print(f"Extracting ID set from GSheet: '{DIAMOND_SHEET_NAME}'...")
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    # Filter for what GSheet sees as "non-empty"
    gsheet_ids = set(df_source[df_source['special_note_raw'] != '']['offer_id'])
    print(f"✅ Found {len(gsheet_ids)} offer_ids with non-empty 'special_note_raw' in GSheet.")

    # --- [2] Get the ID set from the Database Destination ---
    print(f"Extracting ID set from {os.path.basename(db_file_path)}...")
    with sqlite3.connect(db_file_path) as conn:
        # Our ETL converts whitespace-only to NULL, so we check for NOT NULL.
        db_query = "SELECT offer_id FROM offers WHERE special_note_raw IS NOT NULL;"
        df_db = pd.read_sql_query(db_query, conn)

    db_ids = set(df_db['offer_id'])
    print(f"✅ Found {len(db_ids)} offer_ids with non-NULL 'special_note_raw' in the database.")

    # --- [3] THE RECONCILIATION ---
    print("\n--- Performing Set Difference Analysis ---")

    discrepancy_ids = gsheet_ids - db_ids

    if not discrepancy_ids:
        print("\n✅ SUCCESS: The sets are identical. Reconciliation is perfect.")
    else:
        print(f"\n🛑 ANOMALY DETECTED: Found {len(discrepancy_ids)} 'offer_id'(s) present in the GSheet count but missing from the database count.")
        print("This is definitive proof that these are the records with whitespace-only 'special_note_raw' values.")
        print("\nAnomaly List (offer_id):")
        for offer_id in sorted(list(discrepancy_ids)):
            print(offer_id)

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Performing 'ID Set' Reconciliation for 'special_note_raw' (Legacy A) ---
Extracting ID set from GSheet: 'raw_Offers'...
✅ Found 3466 offer_ids with non-empty 'special_note_raw' in GSheet.
Extracting ID set from pienzaba.db...
✅ Found 3466 offer_ids with non-NULL 'special_note_raw' in the database.

--- Performing Set Difference Analysis ---

✅ SUCCESS: The sets are identical. Reconciliation is perfect.


In [56]:
# ==============================================================================
# === CELL A55: SURGICAL STRIKE PATCH - Trip '250825-01' override ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine

# --- Setup & Constants ---
print("--- Initiating Surgical Strike Patch A55 (Legacy A) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienzaba.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

TARGET_TRIP_ID = '250825-01'
CORRECT_OFFER_ID = 'OF00318'

# 1. READ & DISPLAY "BEFORE" STATE
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# 2. THE SURGICAL STRIKE
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID
# Null out existing corrupted values
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
# Set the CORRECT offer_id on the T1 event
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Override applied in memory to column 'offer_id_fk'.")

# 3. RE-RUN "GOLDEN PROPAGATION"
print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# 4. WRITE BACK (FORCED SAVE)
print("\n⏳ Writing patched data back to pienzaba.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data. Details: {e}")
    raise

# 5. FINAL VERIFICATION
print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch A55 (Legacy A) ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
54,250825-01,2,OF00318
55,250825-01,4,OF00034
56,250825-01,5,OF00034



⏳ Applying manual override to trip '250825-01' in memory...
✅ Override applied in memory to column 'offer_id_fk'.

⏳ Re-running Golden Propagation in memory...
✅ Propagation complete in memory.

⏳ Writing patched data back to pienzaba.db...
   - Data written and connection disposed. Pausing for 5 seconds...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250825-01,2,OF00318
1,250825-01,4,OF00318
2,250825-01,5,OF00318


✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.

--- SURGICAL STRIKE COMPLETE ---


In [57]:
# ==============================================================================
# === CELL A56: SURGICAL STRIKE PATCH - The 'OF02649/OF02650' Incident ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine

# --- Setup & Constants ---
print("--- Initiating Surgical Strike Patch A56 (Legacy A) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienzaba.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250917-10'
CORRECT_OFFER_ID = 'OF02650'

# 1. READ & DISPLAY "BEFORE" STATE
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# 2. THE SURGICAL STRIKE
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

# Step 2.1: UNMATCH - Clear existing links
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

# Step 2.2: REMATCH - Link T1 event correctly
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

# 3. RE-RUN "GOLDEN PROPAGATION"
print("\n⏳ Re-running Golden Propagation to ensure lifecycle consistency...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# 4. WRITE BACK (FORCED SAVE)
print("\n⏳ Writing patched data back to pienzaba.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data. Details: {e}")
    raise

# 5. FINAL VERIFICATION
print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch A56 (Legacy A) ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
547,250917-10,1,OF02648
548,250917-10,2,OF02648
549,250917-10,3,OF02648
550,250917-10,4,OF02648
551,250917-10,5,OF02648



⏳ Applying manual override to trip '250917-10' in memory...
✅ Rematched T1 event to correct offer 'OF02650'.

⏳ Re-running Golden Propagation to ensure lifecycle consistency...
✅ Propagation complete in memory.

⏳ Writing patched data back to pienzaba.db...
   - Data written. Pausing for 5 seconds for sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250917-10,1,OF02650
1,250917-10,2,OF02650
2,250917-10,3,OF02650
3,250917-10,4,OF02650
4,250917-10,5,OF02650


✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.

--- SURGICAL STRIKE COMPLETE ---


In [58]:
# ==============================================================================
# === CELL A57: SURGICAL STRIKE PATCH - The 'OF03759/OF03780' Incident ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine

# --- Setup & Constants ---
print("--- Initiating Surgical Strike Patch A57 (Legacy A) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienzaba.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250924-04'
CORRECT_OFFER_ID = 'OF03780'

# 1. READ & DISPLAY "BEFORE" STATE
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# 2. THE SURGICAL STRIKE
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

# Step 2.1: UNMATCH - Clear existing links
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

# Step 2.2: REMATCH - Link T1 event correctly
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

# 3. RE-RUN "GOLDEN PROPAGATION"
print("\n⏳ Re-running Golden Propagation to ensure lifecycle consistency...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# 4. WRITE BACK (FORCED SAVE)
print("\n⏳ Writing patched data back to pienzaba.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data. Details: {e}")
    raise

# 5. FINAL VERIFICATION
print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch A57 (Legacy A) ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
788,250924-04,2,OF03758
789,250924-04,3,OF03758
790,250924-04,4,OF03758
791,250924-04,5,OF03758



⏳ Applying manual override to trip '250924-04' in memory...
✅ Rematched T1 event to correct offer 'OF03780'.

⏳ Re-running Golden Propagation to ensure lifecycle consistency...
✅ Propagation complete in memory.

⏳ Writing patched data back to pienzaba.db...
   - Data written. Pausing for 5 seconds for sync...


✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250924-04,2,OF03780
1,250924-04,3,OF03780
2,250924-04,4,OF03780
3,250924-04,5,OF03780


✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.

--- SURGICAL STRIKE COMPLETE ---


In [59]:
# ==============================================================================
# === CELL A58: SURGICAL STRIKE PATCH - The 'OF04729/OF04731' Incident ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine

# --- Setup & Constants ---
print("--- Initiating Surgical Strike Patch A58 (Legacy A) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienzaba.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Define the ground truth based on the Architect's audit
TARGET_TRIP_ID = '251001-02'
CORRECT_OFFER_ID = 'OF04733'

# 1. READ & DISPLAY "BEFORE" STATE
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# 2. THE SURGICAL STRIKE
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

# Step 2.1: UNMATCH - Clear existing links
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

# Step 2.2: REMATCH - Link T1 event correctly
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

# 3. RE-RUN "GOLDEN PROPAGATION"
print("\n⏳ Re-running Golden Propagation to ensure lifecycle consistency...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# 4. WRITE BACK (FORCED SAVE)
print("\n⏳ Writing patched data back to pienzaba.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data. Details: {e}")
    raise

# 5. FINAL VERIFICATION
print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch A58 (Legacy A) ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
1005,251001-02,1,OF04728
1006,251001-02,2,OF04728
1007,251001-02,3,OF04728
1008,251001-02,4,OF04728
1009,251001-02,5,OF04728



⏳ Applying manual override to trip '251001-02' in memory...
✅ Rematched T1 event to correct offer 'OF04733'.

⏳ Re-running Golden Propagation to ensure lifecycle consistency...
✅ Propagation complete in memory.

⏳ Writing patched data back to pienzaba.db...


   - Data written. Pausing for 5 seconds for sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,251001-02,1,OF04733
1,251001-02,2,OF04733
2,251001-02,3,OF04733
3,251001-02,4,OF04733
4,251001-02,5,OF04733


✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.

--- SURGICAL STRIKE COMPLETE ---


In [60]:
# ==============================================================================
# === CELL A59: SURGICAL STRIKE PATCH - The 'OF04649/OF04565' Incident ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine

# --- Setup & Constants ---
print("--- Initiating CORRECTED Surgical Strike Patch A59 (Legacy A) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienzaba.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250930-04'
CORRECT_OFFER_ID = 'OF04565'

# 1. READ & DISPLAY "BEFORE" STATE
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# 2. THE SURGICAL STRIKE
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

# Step 2.1: UNMATCH - Clear existing links
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

# Step 2.2: REMATCH - Link T1 event correctly
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

# 3. RE-RUN "GOLDEN PROPAGATION"
print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# 4. WRITE BACK (FORCED SAVE)
print("\n⏳ Writing patched data back to pienzaba.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data. Details: {e}")
    raise

# 5. FINAL VERIFICATION
print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating CORRECTED Surgical Strike Patch A59 (Legacy A) ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
972,250930-04,1,OF04564
973,250930-04,2,OF04564



⏳ Applying manual override to trip '250930-04' in memory...
✅ Rematched T1 event to correct offer 'OF04565'.

⏳ Re-running Golden Propagation in memory...
✅ Propagation complete in memory.

⏳ Writing patched data back to pienzaba.db...
   - Data written. Pausing for 5 seconds for sync...


✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250930-04,1,OF04565
1,250930-04,2,OF04565


✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.

--- SURGICAL STRIKE COMPLETE ---


In [1]:
# ==============================================================================
# === CELL A60: SURGICAL STRIKE PATCH - The 'OF03904/OF04649' Incident ===
# Target: Notebook 0111a (Legacy) | Database: pienzaba.db
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine

# --- Setup & Constants ---
print("--- Initiating Surgical Strike Patch A60 (Legacy A) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienzaba.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# Ground Truth from Architect Audit
TARGET_TRIP_ID = '250930-09'
CORRECT_OFFER_ID = 'OF04649'

# 1. READ & DISPLAY "BEFORE" STATE
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# 2. THE SURGICAL STRIKE
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

# Step 2.1: UNMATCH - Clear existing links
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

# Step 2.2: REMATCH - Link T1 event correctly
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Rematched T1 event to correct offer '{CORRECT_OFFER_ID}'.")

# 3. RE-RUN "GOLDEN PROPAGATION"
print("\n⏳ Re-running Golden Propagation to ensure lifecycle consistency...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# 4. WRITE BACK (FORCED SAVE)
print("\n⏳ Writing patched data back to pienzaba.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data. Details: {e}")
    raise

# 5. FINAL VERIFICATION
print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch A60 (Legacy A) ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
993,250930-09,2,OF04649
994,250930-09,4,OF04649
995,250930-09,5,OF04649



⏳ Applying manual override to trip '250930-09' in memory...
✅ Rematched T1 event to correct offer 'OF04649'.

⏳ Re-running Golden Propagation to ensure lifecycle consistency...
✅ Propagation complete in memory.

⏳ Writing patched data back to pienzaba.db...
   - Data written. Pausing for 5 seconds for sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250930-09,2,OF04649
1,250930-09,4,OF04649
2,250930-09,5,OF04649


✅ VERIFICATION SUCCESS: Changes persisted to pienzaba.db.

--- SURGICAL STRIKE COMPLETE ---


In [61]:
# ==============================================================================
# === CELL A61: THE "GOLDEN EDICT" BATCH OVERRIDE (Legacy A) ===
# Target: Notebook 0111a | Database: pienzaba.db
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
from sqlalchemy import create_engine

# --- 0. SETUP & THE "GOLDEN EDICT" ---
print("--- Initiating Final Patch: The Golden Edict (Legacy A) ---")
try:
    db_path = '/workspaces/pienza/data/big_bang/pienzaba.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Database engine failed. Details: {e}")
    raise

# The definitive list of corrections
GOLDEN_EDICT = [
    ('250911-10', 'OF02170', 'OF02169'),
    ('250911-09', 'OF02160', 'OF02159'),
    ('250904-05', 'OF01667', 'OF01666'),
    ('250902-02', 'OF01290', 'OF01289'),
    ('250901-10', 'OF01280', 'OF01279')
]
target_trip_ids = [item[0] for item in GOLDEN_EDICT]

# --- 1. READ & DISPLAY "BEFORE" STATE ---
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'].isin(target_trip_ids)][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events. Details: {e}")
    raise

# --- 2. THE BATCH SURGICAL STRIKE ---
print(f"\n⏳ Applying {len(GOLDEN_EDICT)} manual overrides in memory...")
for trip_id, incorrect_id, correct_id in GOLDEN_EDICT:
    target_mask = trip_events_df['trip_id_legacy'] == trip_id
    trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
    t1_mask = (trip_events_df['trip_id_legacy'] == trip_id) & (trip_events_df['event_types_id_fk'] == 2)
    trip_events_df.loc[t1_mask, 'offer_id_fk'] = correct_id
print("✅ All overrides applied in memory.")

# --- 3. RE-RUN "GOLDEN PROPAGATION" ---
print("\n⏳ Re-running Golden Propagation for all patched trips...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# --- 4. WRITE BACK ---
print("\n⏳ Writing final patched data back to pienzaba.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written. Pausing for 5 seconds for sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data. Details: {e}")
    raise

# --- 5. FINAL VERIFICATION ---
print("\n--- VERIFICATION (AFTER PATCH) ---")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    # Format tuple for query
    trip_tuple = tuple(target_trip_ids) if len(target_trip_ids) > 1 else f"('{target_trip_ids[0]}')"
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy IN {trip_tuple}", verify_engine)
    verify_engine.dispose()

    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']].sort_values(by='trip_id_legacy'))

    for trip_id, _, correct_id in GOLDEN_EDICT:
        assert (verify_df[verify_df['trip_id_legacy'] == trip_id]['offer_id_fk'] == correct_id).all(), f"Verification Failed for {trip_id}!"
    print("✅ VERIFICATION SUCCESS: All changes persisted to pienzaba.db.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: {e}")

print("\n--- BATCH SURGICAL STRIKE COMPLETE ---")

--- Initiating Final Patch: The Golden Edict (Legacy A) ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
252,250901-10,2,OF01279
253,250901-10,3,OF01279
254,250901-10,4,OF01279
255,250901-10,5,OF01279
261,250902-02,2,OF01289
262,250902-02,3,OF01289
263,250902-02,4,OF01289
264,250902-02,5,OF01289
346,250904-05,1,OF01666
347,250904-05,2,OF01666



⏳ Applying 5 manual overrides in memory...
✅ All overrides applied in memory.

⏳ Re-running Golden Propagation for all patched trips...
✅ Propagation complete in memory.

⏳ Writing final patched data back to pienzaba.db...
   - Data written. Pausing for 5 seconds for sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250901-10,2,OF01279
1,250901-10,3,OF01279
2,250901-10,4,OF01279
3,250901-10,5,OF01279
4,250902-02,2,OF01289
5,250902-02,3,OF01289
6,250902-02,4,OF01289
7,250902-02,5,OF01289
12,250904-05,5,OF01666
11,250904-05,4,OF01666


✅ VERIFICATION SUCCESS: All changes persisted to pienzaba.db.

--- BATCH SURGICAL STRIKE COMPLETE ---


In [64]:
# ==============================================================================
# CELL [NEW] (v1.3): THE "CONSOMASTER" BRIDGE (HARDENED PERSISTENCE)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Connects trip_events to lifetime_trips via the Consomaster bridge.
#          Includes ID normalization, Forced Save Protocol, and Read-Back
#          Verification to guarantee data is persisted to disk.
# ==============================================================================

import pandas as pd
import numpy as np
import time
from sqlalchemy import create_engine

print("--- Initiating Consomaster Bridge Protocol (v1.3 - Hardened) ---")

# ------------------------------------------------------------------------------
# 1. SETUP & INGESTION
# ------------------------------------------------------------------------------
CONSO_GSHEET_NAME = 'ConsoMaster_with_MasterID_FINAL'
CONSO_WORKSHEET_NAME = 'Sheet1'

print(f"⏳ Ingesting Consomaster Bridge from: {CONSO_GSHEET_NAME}...")
try:
    conso_ws = gc.open(CONSO_GSHEET_NAME).worksheet(CONSO_WORKSHEET_NAME)
    conso_df = pd.DataFrame(conso_ws.get_all_records()).astype(str)

    bridge_df = conso_df[
        (conso_df['trip_id_legacy'] != '') &
        (conso_df['lifetime_trips_id'] != '')
    ][['trip_id_legacy', 'lifetime_trips_id']].copy()

    # NORMALIZE IDs (Strip 'LT' and leading zeros)
    bridge_df['lifetime_trips_id'] = bridge_df['lifetime_trips_id'].str.replace('LT', '').astype(float).astype(int).astype(str)

    print(f"✅ Successfully ingested bridge. Found {len(bridge_df)} valid link pairs.")

except Exception as e:
    print(f"🔴 ERROR: Failed to ingest/normalize Consomaster. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. FETCH SOURCE DATA
# ------------------------------------------------------------------------------
print("\n⏳ Fetching 'offer_id_fk' source data from 'trip_events'...")
try:
    source_links_df = pd.read_sql(
        "SELECT DISTINCT trip_id_legacy, offer_id_fk FROM trip_events WHERE offer_id_fk IS NOT NULL",
        db_engine
    )
    source_links_df['trip_id_legacy'] = source_links_df['trip_id_legacy'].astype(str)
    print(f"✅ Loaded {len(source_links_df)} source links from trip_events.")
except Exception as e:
    print(f"🔴 ERROR: Failed to read from trip_events. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 3. BUILD THE BRIDGE
# ------------------------------------------------------------------------------
print("\n⏳ Constructing the Golden Link map...")
merged_map = pd.merge(bridge_df, source_links_df, on='trip_id_legacy', how='inner')

lifetime_to_offer_map = pd.Series(
    merged_map.offer_id_fk.values,
    index=merged_map.lifetime_trips_id
).to_dict()

print(f"✅ Bridge constructed. Mapped {len(lifetime_to_offer_map)} lifetime trips to offers.")

# ------------------------------------------------------------------------------
# 4. UPDATE THE TARGET (WITH FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Updating 'lifetime_trips' table in opus.db...")
try:
    lifetime_trips_df = pd.read_sql("SELECT * FROM lifetime_trips", db_engine)

    # Apply Map
    lifetime_trips_df['offer_id_fk'] = lifetime_trips_df['lifetime_trips_id'].astype(str).map(lifetime_to_offer_map)

    updated_count = lifetime_trips_df['offer_id_fk'].notna().sum()
    print(f"   - Rows updated with Offer IDs: {updated_count}")

    if updated_count == 0:
        print("🔴 WARNING: Update count is 0. Check IDs.")
    else:
        # WRITE TO DB
        lifetime_trips_df.to_sql('lifetime_trips', db_engine, if_exists='replace', index=False)

        # FORCED SAVE PROTOCOL
        db_engine.dispose() # Close connection to flush
        print("   - Data written. Connection closed.")
        print("   - Pausing 5 seconds for Drive Sync...")
        time.sleep(5)
        print("✅ SUCCESS: 'lifetime_trips' table updated and saved.")

except Exception as e:
    print(f"🔴 ERROR: Failed to update lifetime_trips. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. READ-BACK VERIFICATION
# ------------------------------------------------------------------------------
print("\n⏳ Performing Read-Back Verification...")
try:
    # Re-open connection to verify disk state
    verify_engine = create_engine(f'sqlite:///{db_path}')

    # Check for non-null keys
    verify_count = pd.read_sql("SELECT COUNT(*) FROM lifetime_trips WHERE offer_id_fk IS NOT NULL", verify_engine).iloc[0,0]
    verify_engine.dispose()

    print(f"🔎 Verified on Disk: {verify_count} rows in 'lifetime_trips' now have an offer_id_fk.")

    if verify_count > 0:
        print("✅ VERIFICATION PASSED: The database is definitively updated.")
    else:
        print("🔴 VERIFICATION FAILED: The database file on disk still shows 0 links.")

except Exception as e:
    print(f"🔴 ERROR during verification: {e}")

print("\n--- BRIDGE PROTOCOL COMPLETE ---")

--- Initiating Consomaster Bridge Protocol (v1.3 - Hardened) ---
⏳ Ingesting Consomaster Bridge from: ConsoMaster_with_MasterID_FINAL...
✅ Successfully ingested bridge. Found 254 valid link pairs.

⏳ Fetching 'offer_id_fk' source data from 'trip_events'...
✅ Loaded 259 source links from trip_events.

⏳ Constructing the Golden Link map...
✅ Bridge constructed. Mapped 254 lifetime trips to offers.

⏳ Updating 'lifetime_trips' table in opus.db...
   - Rows updated with Offer IDs: 254
   - Data written. Connection closed.
   - Pausing 5 seconds for Drive Sync...
✅ SUCCESS: 'lifetime_trips' table updated and saved.

⏳ Performing Read-Back Verification...
🔎 Verified on Disk: 254 rows in 'lifetime_trips' now have an offer_id_fk.
✅ VERIFICATION PASSED: The database is definitively updated.

--- BRIDGE PROTOCOL COMPLETE ---


In [65]:
# ==============================================================================
# CELL [VIEWS]: THE ANALYTICAL LAYER EMBEDDING
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Embeds the SQL logic for all analytical views directly into the database.
#          This ensures opus.db is a self-contained product. When delivered,
#          the views exist permanently without requiring external SQL scripts.
# ==============================================================================

from sqlalchemy import text

print("--- Initiating View Embedding Protocol ---")

# We define the SQL commands to Drop (cleanup) and Create (build).
# Order matters: We build from the bottom up (Base Views -> Final Views).

view_commands = [
    # 1. CLEANUP: Remove old versions if they exist
    "DROP VIEW IF EXISTS v_mission_dossier;",
    "DROP VIEW IF EXISTS v_trip_final_kpis;",
    "DROP VIEW IF EXISTS v_trip_funnel_wide;",
    "DROP VIEW IF EXISTS v_reconciled_offer;",

    # 2. BUILD BASE: v_trip_funnel_wide (Pivots events into columns)
    """
    CREATE VIEW v_trip_funnel_wide AS
    SELECT
        trip_id_legacy,
        MAX(offer_id_fk) AS offer_id_fk,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM trip_events
    GROUP BY trip_id_legacy;
    """,

    # 3. BUILD LOGIC: v_trip_final_kpis (Calculates durations and spreads)
    """
    CREATE VIEW v_trip_final_kpis AS
    SELECT
        v.trip_id_legacy,
        DATE(v.t1_timestamp) AS trip_date,
        (julianday(v.t2_timestamp) - julianday(v.t1_timestamp)) * 86400.0 AS duration_to_pickup_sec,
        (julianday(v.t3_timestamp) - julianday(v.t2_timestamp)) * 86400.0 AS duration_waiting_sec,
        (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 86400.0 AS duration_trip_sec,
        (julianday(v.t4_timestamp) - julianday(v.t1_timestamp)) * 86400.0 AS total_engagement_duration_sec,
        v.upfront_fare,
        v.realized_fare,
        CASE WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare ELSE NULL END AS spread_percentage,
        CASE WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0 THEN v.realized_fare / (((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 86400.0) / 3600.0) ELSE NULL END AS eph_on_ride,
        CASE WHEN (julianday(v.t4_timestamp) - julianday(v.t1_timestamp)) > 0 THEN v.realized_fare / (((julianday(v.t4_timestamp) - julianday(v.t1_timestamp)) * 86400.0) / 3600.0) ELSE NULL END AS eph_total_time
    FROM v_trip_funnel_wide v;
    """,

    # 4. BUILD PRODUCT: v_mission_dossier (The Golden Link View you requested)
    """
    CREATE VIEW v_mission_dossier AS
    WITH OfferLink AS (
        SELECT
            trip_id_legacy,
            MAX(offer_id_fk) AS offer_id
        FROM
            trip_events
        GROUP BY
            trip_id_legacy
    )
    SELECT
        ol.offer_id,
        kpi.*
    FROM
        v_trip_final_kpis AS kpi
    LEFT JOIN
        OfferLink AS ol ON kpi.trip_id_legacy = ol.trip_id_legacy;
    """
]

# EXECUTION LOOP
print("⏳ Embedding views into opus.db...")
try:
    with db_engine.connect() as connection:
        for cmd in view_commands:
            connection.execute(text(cmd))
            # print(f"   - Executed: {cmd.split()[0]} {cmd.split()[2]}...") # Optional verbose log
    print("✅ SUCCESS: All Analytical Views have been permanently embedded.")

except Exception as e:
    print(f"🔴 ERROR: Failed to embed views. Details: {e}")
    raise

print("--- ANALYTICAL LAYER READY ---")

--- Initiating View Embedding Protocol ---
⏳ Embedding views into opus.db...
✅ SUCCESS: All Analytical Views have been permanently embedded.
--- ANALYTICAL LAYER READY ---


In [66]:
# ==============================================================================
# CELL [FINAL]: FORGE 'engineered_features' (TYPE HYGIENE & FULL AUDIT v2.1)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Ingests features, links IDs, and performs "Safe Mode" cleaning.
#          It explicitly protects 'day_of_week' and boolean ('is_') columns
#          from being destroyed by numeric coercion logic.
# ==============================================================================

import pandas as pd
import numpy as np
from sqlalchemy import create_engine

print("--- Initiating Silver Layer Materialization (Safe Mode Hygiene) ---")

# ------------------------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------------------------
FEATURES_GSHEET_NAME = 'opus_1_phase_1_engineered_features'
FEATURES_WORKSHEET_NAME = 'Sheet2'

# ------------------------------------------------------------------------------
# 2. INGEST FEATURES FROM GSHEET
# ------------------------------------------------------------------------------
print(f"⏳ Ingesting features from: {FEATURES_GSHEET_NAME}...")
try:
    features_ws = gc.open(FEATURES_GSHEET_NAME).worksheet(FEATURES_WORKSHEET_NAME)
    features_data = features_ws.get_all_records()
    features_df = pd.DataFrame(features_data)

    # Clean empty strings immediately
    features_df.replace('', np.nan, inplace=True)

    print(f"✅ Loaded {len(features_df)} rows of engineered features.")

except Exception as e:
    print(f"🔴 ERROR: Failed to ingest features GSheet. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 3. FETCH DB KEYS & MATCH
# ------------------------------------------------------------------------------
print("\n⏳ Fetching canonical 'offer_id' keys from opus.db...")
try:
    db_keys_df = pd.read_sql("SELECT offer_id FROM offers", db_engine)

    # Extract Integer Logic
    features_df['match_key'] = features_df['feature_id'].astype(str).str.extract(r'(\d+)').astype(int)
    db_keys_df['match_key'] = db_keys_df['offer_id'].astype(str).str.extract(r'(\d+)').astype(int)

    print("⏳ Executing Integer-based Merge...")
    merged_df = pd.merge(
        features_df,
        db_keys_df[['offer_id', 'match_key']],
        on='match_key',
        how='left'
    )
    merged_df['offer_id_fk'] = merged_df['offer_id']
    final_features_df = merged_df.drop(columns=['match_key', 'offer_id'])

except Exception as e:
    print(f"🔴 ERROR during ID matching. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 4. TYPE HYGIENE (THE "SAFE MODE" PROTOCOL)
# ------------------------------------------------------------------------------
print("\n⏳ Enforcing data type hygiene (protecting text and booleans)...")

# A. CLEAN CURRENCY
currency_cols = [col for col in final_features_df.columns if 'earnings' in col or 'eph' in col or 'fare' in col]
for col in currency_cols:
    if final_features_df[col].dtype == 'object':
        final_features_df[col] = final_features_df[col].astype(str).str.replace(r'[$,]', '', regex=True)

# B. PROTECT TEXT COLUMNS (Updated List)
# Added 'day_of_week' to ensure "Friday" remains "Friday"
text_keywords = [
    '_label', '_block', '_type', '_ambiguity',
    'feature_id', 'offer_id_fk',
    'day_shift', 'day_type', 'day_of_week'
]

# C. CONVERT NUMERICS (With Boolean Protection)
for col in final_features_df.columns:
    is_protected_text = any(keyword in col for keyword in text_keywords)
    is_boolean = 'is_' in col  # New check!

    # Only attempt numeric conversion if it's NOT protected text AND NOT a boolean
    if not is_protected_text and not is_boolean:
        final_features_df[col] = pd.to_numeric(final_features_df[col], errors='coerce')

# D. HANDLE BOOLEANS (Explicitly)
bool_cols = [col for col in final_features_df.columns if 'is_' in col]
for col in bool_cols:
    # Normalize to 1/0 for SQLite. Handles 'TRUE', 'True', True, 1.
    # Using map is safer than astype(int) because of NaNs
    final_features_df[col] = final_features_df[col].astype(str).str.upper().map({
        'TRUE': 1, 'FALSE': 0,
        '1': 1, '0': 0,
        'NAN': None, 'NONE': None
    })

# ------------------------------------------------------------------------------
# 5. MATERIALIZE TO DATABASE
# ------------------------------------------------------------------------------
print("\n⏳ Writing 'engineered_features' table to opus.db...")
try:
    final_features_df.to_sql('engineered_features', db_engine, if_exists='replace', index=False)
    print("✅ SUCCESS: 'engineered_features' table created in opus.db.")

    # --------------------------------------------------------------------------
    # 6. FINAL VERIFICATION
    # --------------------------------------------------------------------------
    print("\n--- FINAL VERIFICATION: READING BACK THE TAIL ---")
    verify_df = pd.read_sql("SELECT * FROM engineered_features", db_engine)

    # Display specific columns to prove the fix worked
    cols_to_check = ['day_of_week', 'is_total_cycle_downgrade_EDA', 'eph_realized_ML']
    # Filter to columns that actually exist in the DF to avoid key errors during display
    cols_present = [c for c in cols_to_check if c in verify_df.columns]

    display(verify_df[cols_present].tail(10))

    print(f"\n📊 Final Table Stats:")
    print(f"   - Total Rows: {len(verify_df)}")
    print(f"   - Total Columns: {len(verify_df.columns)}")

except Exception as e:
    print(f"🔴 ERROR: Failed to write/verify table. Details: {e}")
    raise

print("\n--- FEATURE STORE COMPLETE ---")

--- Initiating Silver Layer Materialization (Safe Mode Hygiene) ---
⏳ Ingesting features from: opus_1_phase_1_engineered_features...
✅ Loaded 4765 rows of engineered features.

⏳ Fetching canonical 'offer_id' keys from opus.db...
⏳ Executing Integer-based Merge...

⏳ Enforcing data type hygiene (protecting text and booleans)...

⏳ Writing 'engineered_features' table to opus.db...
✅ SUCCESS: 'engineered_features' table created in opus.db.

--- FINAL VERIFICATION: READING BACK THE TAIL ---


,day_of_week,is_total_cycle_downgrade_EDA,eph_realized_ML
4755,Wednesday,1.0,366.330417
4756,Wednesday,1.0,233.883529
4757,Wednesday,1.0,349.899323
4758,Wednesday,1.0,355.773394
4759,Wednesday,1.0,304.394152
4760,Wednesday,1.0,189.802989
4761,Wednesday,0.0,169.382958
4762,Wednesday,1.0,235.745900
4763,Wednesday,0.0,158.308444
4764,Wednesday,1.0,194.253762



📊 Final Table Stats:
   - Total Rows: 4765
   - Total Columns: 45

--- FEATURE STORE COMPLETE ---


In [67]:
# ==============================================================================
# CELL [FINAL PATCH]: THE OCR DETERMINISTIC LINKER (TD-006 RESOLUTION)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Resolves the missing link between 'offers' and 'raw_offers_ocr'.
#          Uses the Architect's "Integer Extraction Protocol" to deterministically
#          match OFxxxxx to OCRxxxxx based on the numeric ID, ensuring robust
#          linking even if rows were dropped/filtered in the cleaned layer.
# ==============================================================================

import pandas as pd
import numpy as np
from sqlalchemy import create_engine

print("--- Initiating OCR Deterministic Linker Protocol ---")

# ------------------------------------------------------------------------------
# 1. LOAD TABLES
# ------------------------------------------------------------------------------
print("⏳ Loading 'offers' and 'raw_offers_ocr' tables...")
try:
    offers_df = pd.read_sql("SELECT * FROM offers", db_engine)
    ocr_df = pd.read_sql("SELECT ocr_id FROM raw_offers_ocr", db_engine) # We only need the ID

    print(f"✅ Loaded {len(offers_df)} Offers and {len(ocr_df)} OCR records.")

except Exception as e:
    print(f"🔴 ERROR: Failed to load tables. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. EXECUTE MATCHING LOGIC (The Integer Extraction)
# ------------------------------------------------------------------------------
print("\n⏳ Extracting integers and mapping keys...")

# Step A: Extract Integer from Offer ID (OF00001 -> 1)
# We use a temporary column for the join key
offers_df['match_key'] = offers_df['offer_id'].astype(str).str.extract(r'(\d+)').astype(int)

# Step B: Extract Integer from OCR ID (OCR00001 -> 1)
ocr_df['match_key'] = ocr_df['ocr_id'].astype(str).str.extract(r'(\d+)').astype(int)

# Step C: Create the Map dictionary {1: 'OCR00001', 2: 'OCR00002'...}
# This is faster and cleaner than a full DataFrame merge for a single column update
ocr_key_map = pd.Series(ocr_df.ocr_id.values, index=ocr_df.match_key).to_dict()

# ------------------------------------------------------------------------------
# 3. APPLY THE LINK
# ------------------------------------------------------------------------------
print("⏳ Applying links to 'offers.ocr_fk'...")

# Map the integer key in offers to the OCR ID string
offers_df['ocr_fk'] = offers_df['match_key'].map(ocr_key_map)

# ------------------------------------------------------------------------------
# 4. VERIFICATION & SAVE
# ------------------------------------------------------------------------------
linked_count = offers_df['ocr_fk'].notna().sum()
total_count = len(offers_df)

print(f"\n📊 Linkage Report:")
print(f"   - Total Offers: {total_count}")
print(f"   - Successfully Linked to OCR: {linked_count}")
print(f"   - Unlinked (Gap/Mismatch): {total_count - linked_count}")

# Clean up the helper column
offers_df.drop(columns=['match_key'], inplace=True)

if linked_count > 0:
    print("\n⏳ Saving updated 'offers' table to opus.db...")
    try:
        offers_df.to_sql('offers', db_engine, if_exists='replace', index=False)
        print("✅ SUCCESS: 'offers' table updated. TD-006 Resolved.")

        # Final Peek
        display(offers_df[['offer_id', 'ocr_fk']].head(5))

    except Exception as e:
        print(f"🔴 ERROR: Failed to write to database. Details: {e}")
else:
    print("🔴 CRITICAL WARNING: No links were created. Check your ID formats (OFxxxxx vs OCRxxxxx).")

print("\n--- LINKER PROTOCOL COMPLETE ---")

--- Initiating OCR Deterministic Linker Protocol ---
⏳ Loading 'offers' and 'raw_offers_ocr' tables...
✅ Loaded 4765 Offers and 4765 OCR records.

⏳ Extracting integers and mapping keys...
⏳ Applying links to 'offers.ocr_fk'...

📊 Linkage Report:
   - Total Offers: 4765
   - Successfully Linked to OCR: 4765
   - Unlinked (Gap/Mismatch): 0

⏳ Saving updated 'offers' table to opus.db...
✅ SUCCESS: 'offers' table updated. TD-006 Resolved.


,offer_id,ocr_fk
0,OF01322,OCR01322
1,OF01875,OCR01875
2,OF00923,OCR00923
3,OF02333,OCR02333
4,OF03254,OCR03254



--- LINKER PROTOCOL COMPLETE ---


In [68]:
# ==============================================================================
# CELL [PATCH]: THE "TWIN LINK" PROTOCOL (v1.2 - CLEAN MERGE)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Resolves links in 'activity_earnings' by stripping destination columns
#          before merging to avoid suffix confusion.
# ==============================================================================

import pandas as pd
import numpy as np
from sqlalchemy import create_engine

print("--- Initiating Earnings Reconciliation Protocol (v1.2) ---")

try:
    # 1. LOAD THE TABLES
    print("⏳ Loading tables...")
    ae_df = pd.read_sql("SELECT * FROM activity_earnings", db_engine)
    lt_df = pd.read_sql("SELECT lifetime_trips_id, offer_id_fk FROM lifetime_trips", db_engine)

    # 2. PREPARE DESTINATION (The Clean Slate)
    # Drop the target columns if they exist to prevent collision/suffixes
    ae_df = ae_df.drop(columns=['lifetime_trips_fk', 'offer_id_fk'], errors='ignore')

    # 3. EXTRACT THE MATCH KEY (The Numeric Suffix)
    print("⏳ Extracting match keys...")
    # Regex to extract digits: 'AE03446' -> 3446
    ae_df['match_key'] = ae_df['activity_earnings_id'].astype(str).str.extract(r'(\d+)').astype(int)
    lt_df['match_key'] = lt_df['lifetime_trips_id'].astype(str).str.extract(r'(\d+)').astype(int)

    # 4. MERGE (Left Join)
    print("⏳ Linking tables based on numeric ID...")
    # We bring in the ID and the Offer Key from Lifetime Trips
    merged_df = pd.merge(
        ae_df,
        lt_df[['lifetime_trips_id', 'offer_id_fk', 'match_key']],
        on='match_key',
        how='left'
    )

    # 5. MAP COLUMNS
    # The column 'lifetime_trips_id' from the right is now our FK
    merged_df['lifetime_trips_fk'] = merged_df['lifetime_trips_id']

    # 'offer_id_fk' came straight from the right side, no suffix needed
    # (It's already named correctly)

    # 6. CLEANUP AND VERIFY
    cols_to_drop = ['match_key', 'lifetime_trips_id']
    final_ae_df = merged_df.drop(columns=cols_to_drop, errors='ignore')

    linked_lt = final_ae_df['lifetime_trips_fk'].notna().sum()
    linked_offer = final_ae_df['offer_id_fk'].notna().sum()

    print(f"\n📊 Linkage Report:")
    print(f"   - Rows in Activity Earnings: {len(final_ae_df)}")
    print(f"   - Linked to Lifetime Trips:  {linked_lt}")
    print(f"   - Linked to Offers:          {linked_offer}")

    # 7. SAVE TO DB
    if linked_lt > 0:
        print("\n⏳ Saving updated 'activity_earnings' table to opus.db...")
        final_ae_df.to_sql('activity_earnings', db_engine, if_exists='replace', index=False)
        print("✅ SUCCESS: 'activity_earnings' table fully reconciled.")

        display(final_ae_df[['activity_earnings_id', 'lifetime_trips_fk', 'offer_id_fk']].head())
    else:
        print("🔴 WARNING: No links were created. Check ID formats.")

except Exception as e:
    print(f"🔴 ERROR: Failed to reconcile earnings. Details: {e}")

print("\n--- PROTOCOL COMPLETE ---")

--- Initiating Earnings Reconciliation Protocol (v1.2) ---
⏳ Loading tables...
⏳ Extracting match keys...
⏳ Linking tables based on numeric ID...

📊 Linkage Report:
   - Rows in Activity Earnings: 3446
   - Linked to Lifetime Trips:  3446
   - Linked to Offers:          254

⏳ Saving updated 'activity_earnings' table to opus.db...
✅ SUCCESS: 'activity_earnings' table fully reconciled.


,activity_earnings_id,lifetime_trips_fk,offer_id_fk
0,1,1,None
1,2,2,None
2,3,3,None
3,4,4,None
4,5,5,None



--- PROTOCOL COMPLETE ---


In [69]:
# ==============================================================================
# CELL [BROCHE 1/3]: THE "MEGA JOIN" FK AUDIT VIEW (CORRECTED)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Creates a permanent view 'v_broche_fks' in opus.db.
#          Fixes 'one statement at a time' error by splitting execution.
#          Includes Cher Ami Message 1/3.
# ==============================================================================

from sqlalchemy import text

print("--- FORGING BROCHE 1: THE FK MEGA JOIN ---")

# 1. DEFINE THE SQL COMMANDS (SPLIT)
drop_sql = "DROP VIEW IF EXISTS v_broche_fks;"

create_sql = """
CREATE VIEW v_broche_fks AS
SELECT
    -- 1. The Origin (OCR)
    r.ocr_id                  AS raw_ocr_id,

    -- 2. The Hub (Offers)
    o.offer_id                AS hub_offer_id,
    o.session_fk              AS hub_session_fk,
    o.ocr_fk                  AS hub_ocr_fk,

    -- 3. The Brain (Features)
    ef.feature_id             AS feat_id,
    ef.offer_id_fk            AS feat_offer_id_fk,

    -- 4. The Field (Trip Events)
    te.event_id               AS event_id,
    te.offer_id_fk            AS event_offer_id_fk,

    -- 5. The Corporate Record (Lifetime Trips)
    lt.lifetime_trips_id      AS lt_id,
    lt.offer_id_fk            AS lt_offer_id_fk,

    -- 6. The Bank (Earnings)
    ae.activity_earnings_id   AS ae_id,
    ae.offer_id_fk            AS ae_offer_id_fk,
    ae.lifetime_trips_fk      AS ae_lt_fk

FROM
    offers o
    LEFT JOIN raw_offers_ocr r        ON o.ocr_fk = r.ocr_id
    LEFT JOIN engineered_features ef  ON o.offer_id = ef.offer_id_fk
    LEFT JOIN trip_events te          ON o.offer_id = te.offer_id_fk
    LEFT JOIN lifetime_trips lt       ON o.offer_id = lt.offer_id_fk
    LEFT JOIN activity_earnings ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk;
"""

# 2. EXECUTE (SEQUENTIAL)
try:
    with db_engine.connect() as conn:
        # Execute Drop
        conn.execute(text(drop_sql))
        # Execute Create
        conn.execute(text(create_sql))
        print("✅ VIEW 'v_broche_fks' CREATED SUCCESSFULLY.")

    # 3. VERIFY CONTENT
    print("\n--- AUDIT: INSPECTING THE DNA ---")
    audit_df = pd.read_sql("""
        SELECT * FROM v_broche_fks
        WHERE lt_id IS NOT NULL
        AND ae_id IS NOT NULL
        LIMIT 5
    """, db_engine)

    if not audit_df.empty:
        display(audit_df)
        print("✅ AUDIT PASSED: Full data lineage visible from OCR to Earnings.")

        # --- CHER AMI MESSAGE 1/3 ---
        print("\n" + "="*60)
        print("🕊️ CHER AMI (Mensaje 1 de 3):")
        print("   'El ADN estructural está intacto. Las líneas de comunicación")
        print("    están abiertas desde la Fuente hasta el Banco.'")
        print("="*60)

    else:
        print("⚠️ WARNING: No fully connected rows found in sample. Check your joins.")

except Exception as e:
    print(f"🔴 ERROR creating view: {e}")

--- FORGING BROCHE 1: THE FK MEGA JOIN ---
✅ VIEW 'v_broche_fks' CREATED SUCCESSFULLY.

--- AUDIT: INSPECTING THE DNA ---


,raw_ocr_id,hub_offer_id,hub_session_fk,hub_ocr_fk,feat_id,feat_offer_id_fk,event_id,event_offer_id_fk,lt_id,lt_offer_id_fk,ae_id,ae_offer_id_fk,ae_lt_fk
0,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1028,OF04761,3446,OF04761,3446,OF04761,3446
1,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1029,OF04761,3446,OF04761,3446,OF04761,3446
2,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1030,OF04761,3446,OF04761,3446,OF04761,3446
3,OCR04761,OF04761,SID0062,OCR04761,EF04761,OF04761,1031,OF04761,3446,OF04761,3446,OF04761,3446
4,OCR04760,OF04760,SID0062,OCR04760,EF04760,OF04760,1024,OF04760,3445,OF04760,3445,OF04760,3445


✅ AUDIT PASSED: Full data lineage visible from OCR to Earnings.

🕊️ CHER AMI (Mensaje 1 de 3):
   'El ADN estructural está intacto. Las líneas de comunicación
    están abiertas desde la Fuente hasta el Banco.'


In [70]:
# ==============================================================================
# CELL [BROCHE 2/3]: THE HUMAN READABLE MASTER VIEW (CORRECTED SCHEMA)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Creates the definitive analytical view 'v_offers_human'.
#          UPDATED: Matches the exact column names from the Vertabelo ERD
#          (e.g., is_operational_downgrade instead of bait_and_switch).
# ==============================================================================

from sqlalchemy import text

print("--- FORGING BROCHE 2: THE HUMAN READABLE VIEW ---")

# 1. DEFINE THE SQL COMMANDS (SPLIT)
drop_sql = "DROP VIEW IF EXISTS v_offers_human;"

create_sql = """
CREATE VIEW v_offers_human AS
SELECT
    -- 1. IDENTITY & TIME
    o.offer_id,
    o.session_fk,
    o.offer_timestamp,

    -- 2. CORE METRICS (The Physics)
    o.upfront_fare,
    o.time_to_pickup_sec,
    o.dist_to_pickup_km,
    o.est_trip_time_sec,
    o.est_trip_dist_km,

    -- 3. HUMAN READABLE LABELS (The Translation)
    pc.category_name                  AS str_product,
    oa.offer_action_description       AS str_action,
    rp.reason_primary_description     AS str_reason,
    ds.driver_state_at_request_description AS str_driver_state,
    pos.post_offer_status_description AS str_post_status,
    out.outcome_description           AS str_outcome,
    iq.interpolation_quality_description AS str_interp_quality,
    rs.record_status_description      AS str_record_status,

    -- 4. GEOSPATIAL CONTEXT
    o.pickup_address,
    o.dropoff_address,
    o.pickup_lat, o.pickup_lon,
    o.dropoff_lat, o.dropoff_lon,

    -- 5. FLAGS & NOTES
    o.is_surge, o.surge_amount,
    o.is_turbo_plus, o.turbo_plus_amount,
    o.is_reservation, o.reservation_amount,
    o.special_note_raw,

    -- 6. THE BRAIN (Engineered Features - ERD ALIGNED)
    -- Ambiguity Flags
    ef.pickup_ambiguity,
    ef.dropoff_ambiguity,

    -- Context & Traffic
    ef.traffic_index_base_120,
    ef.time_since_last_offer,
    ef.offer_density_60sec,
    ef.cycle_avg_dtp_km,

    -- Recent Performance
    ef.cycle_rolling_avg_spread,
    ef.total_accumulated_deadhead_sec,
    ef.cycle_cumulative_net_earnings,

    -- Business Intelligence (The Funnel)
    ef.eph_direct,
    ef.eph_operational,
    ef.is_operational_downgrade, -- FIXED: Correct ERD Name

    -- ML Track Predictions
    ef.eph_realized_ML,
    ef.eph_complete_ML,
    ef.is_spread_downgrade_ML,
    ef.is_total_cycle_downgrade_ML,

    -- Strategic Context
    ef.home_vector_alignment_score,

    -- Temporal Context
    ef.day_of_week,
    ef.time_of_day_block,
    ef.day_type

FROM
    offers o
    -- The 8 Dimensions of Meaning
    LEFT JOIN product_category pc        ON o.product_category_fk = pc.product_category_id
    LEFT JOIN offer_action oa            ON o.offer_action_fk = oa.offer_action_id
    LEFT JOIN reason_primary rp          ON o.reason_primary_fk = rp.reason_primary_id
    LEFT JOIN driver_state_at_request ds ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
    LEFT JOIN post_offer_status pos      ON o.post_offer_status_fk = pos.post_offer_status_id
    LEFT JOIN outcome out                ON o.outcome_fk = out.outcome_id
    LEFT JOIN interpolation_quality iq   ON o.interpolation_quality_fk = iq.interpolation_quality_id
    LEFT JOIN record_status rs           ON o.record_status_fk = rs.record_status_id

    -- The Silver Layer
    LEFT JOIN engineered_features ef     ON o.offer_id = ef.offer_id_fk;
"""

# 2. EXECUTE
try:
    with db_engine.connect() as connection:
        connection.execute(text(drop_sql))
        connection.execute(text(create_sql))
        print("✅ VIEW 'v_offers_human' CREATED SUCCESSFULLY.")

    # 3. VERIFY CONTENT
    print("\n--- AUDIT: READING THE ROSETTA STONE ---")
    # We select specific new columns to verify the fix
    audit_cols = "offer_id, str_action, eph_direct, is_operational_downgrade, home_vector_alignment_score"
    audit_df = pd.read_sql(f"SELECT {audit_cols} FROM v_offers_human LIMIT 3", db_engine)

    if not audit_df.empty:
        display(audit_df)
        print("✅ AUDIT PASSED: The schema is aligned with the ERD.")

        # --- CHER AMI MESSAGE 2/3 ---
        print("\n" + "="*60)
        print("🕊️ CHER AMI (Mensaje 2 de 3):")
        print("   'El cifrado ha sido roto. La nomenclatura es correcta.")
        print("    El mapa es legible para el ojo humano.'")
        print("="*60)

    else:
        print("⚠️ WARNING: View returned no rows.")

except Exception as e:
    print(f"🔴 ERROR creating view: {e}")

--- FORGING BROCHE 2: THE HUMAN READABLE VIEW ---
✅ VIEW 'v_offers_human' CREATED SUCCESSFULLY.

--- AUDIT: READING THE ROSETTA STONE ---


,offer_id,str_action,eph_direct,is_operational_downgrade,home_vector_alignment_score
0,OF01322,reject,285.112150,0.0,-0.520447
1,OF01875,reject,190.511111,0.0,0.920567
2,OF00923,reject,263.094000,0.0,0.572747


✅ AUDIT PASSED: The schema is aligned with the ERD.

🕊️ CHER AMI (Mensaje 2 de 3):
   'El cifrado ha sido roto. La nomenclatura es correcta.
    El mapa es legible para el ojo humano.'


In [71]:
# ==============================================================================
# CELL [BROCHE 3/3]: THE OMNISCIENT LIFECYCLE VIEW (v3.0 - HISTORICAL)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Creates 'v_lifecycle_audit'. This view aligns ALL dimensions of truth:
#          Identity, Time, Product, Space, and Money.
#          It includes calculated Deltas for timestamps and Historical Cumulative
#          Stats (2023+) projected onto the current Offers.
# ==============================================================================

from sqlalchemy import text

print("--- FORGING BROCHE 3: THE OMNISCIENT LIFECYCLE VIEW ---")

drop_sql = "DROP VIEW IF EXISTS v_lifecycle_audit;"

create_sql = """
CREATE VIEW v_lifecycle_audit AS
WITH TripEventsPivot AS (
    -- Pivot events to get T1, T3, T4 timestamps and fares in one row per trip
    SELECT
        offer_id_fk,
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) as t1_timestamp,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) as t3_timestamp,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) as t4_timestamp,
        MAX(upfront_fare) as te_upfront_fare,
        MAX(realized_fare) as te_realized_fare
    FROM trip_events
    GROUP BY offer_id_fk
),
HistoryStats AS (
    -- Calculate Cumulative Stats starting from 2023 using Official Records
    SELECT
        lt.offer_id_fk, -- Link back to Opus Era
        lt.lifetime_trips_id,
        lt.original_fare,
        ae.net_earning,

        -- Cumulative Sums (Ordered by Request Time to include history)
        SUM(lt.original_fare) OVER (ORDER BY lt.request_timestamp ROWS UNBOUNDED PRECEDING) as cum_uber_earnings,
        SUM(ae.net_earning) OVER (ORDER BY lt.request_timestamp ROWS UNBOUNDED PRECEDING) as cum_net_earnings,

        -- Rolling Average Net Spread (Net / Original) - "Take Rate" Proxy
        -- We use NULLIF to avoid divide by zero
        AVG(ae.net_earning / NULLIF(lt.original_fare, 0)) OVER (ORDER BY lt.request_timestamp ROWS UNBOUNDED PRECEDING) as rolling_avg_net_take_rate

    FROM lifetime_trips lt
    LEFT JOIN activity_earnings ae ON lt.lifetime_trips_id = ae.lifetime_trips_fk
)
SELECT
    -- 1. IDENTIFIERS (Offers & Events)
    o.offer_id,
    o.session_fk,
    te.trip_id_legacy,

    -- 2. TIME CONSISTENCY (Raw vs Clean)
    r.time_taken                AS ocr_raw_time,
    o.offer_timestamp           AS clean_timestamp,
    ef.day_of_week,
    ef.time_of_day_block,

    -- 3. PRODUCT CONSISTENCY (Semantics)
    r.ride_type                 AS ocr_product,
    pc.category_name            AS internal_product,
    ae.product_category         AS bank_product,
    lt.global_product_name      AS official_product,

    -- 4. SPATIAL CONSISTENCY (Pickups)
    r.pickup_address            AS ocr_pickup,
    o.pickup_address            AS clean_pickup,
    ef.pickup_ambiguity,

    -- 5. SPATIAL CONSISTENCY (Dropoffs)
    r.dropoff_address           AS ocr_dropoff,
    o.dropoff_address           AS clean_dropoff,
    ef.dropoff_ambiguity,

    -- 6. FINANCIAL CONSISTENCY (The Money Trail)
    lt.original_fare            AS uber_original_fare, -- Passenger Paid
    r.upfront_fare              AS ocr_upfront,
    o.upfront_fare              AS clean_upfront,
    te.te_upfront_fare          AS events_upfront,
    te.te_realized_fare         AS events_realized,
    ae.net_earning              AS bank_net_earning,

    -- 7. TEMPORAL AUDIT (GTS vs Uber Deltas)
    -- T1 vs Request
    te.t1_timestamp             AS gts_t1_accepted,
    lt.request_timestamp        AS uber_request,
    ROUND((julianday(te.t1_timestamp) - julianday(lt.request_timestamp)) * 86400) AS delta_accept_sec,

    -- T3 vs Pickup
    te.t3_timestamp             AS gts_t3_started,
    lt.pickup_timestamp         AS uber_pickup,
    ROUND((julianday(te.t3_timestamp) - julianday(lt.pickup_timestamp)) * 86400) AS delta_start_sec,

    -- T4 vs Dropoff
    te.t4_timestamp             AS gts_t4_completed,
    lt.dropoff_timestamp        AS uber_dropoff,
    ROUND((julianday(te.t4_timestamp) - julianday(lt.dropoff_timestamp)) * 86400) AS delta_end_sec,

    -- 8. HISTORICAL CONTEXT (The Cherry on Top)
    hist.cum_uber_earnings,
    hist.cum_net_earnings,
    hist.rolling_avg_net_take_rate

FROM
    offers o
    -- Join to Source
    LEFT JOIN raw_offers_ocr r        ON o.ocr_fk = r.ocr_id

    -- Join to Intelligence
    LEFT JOIN engineered_features ef  ON o.offer_id = ef.offer_id_fk

    -- Join to Dictionary
    LEFT JOIN product_category pc     ON o.product_category_fk = pc.product_category_id

    -- Join to Pivoted Trip Events (1:1)
    LEFT JOIN TripEventsPivot te      ON o.offer_id = te.offer_id_fk

    -- Join to Official Record
    LEFT JOIN lifetime_trips lt       ON o.offer_id = lt.offer_id_fk

    -- Join to Earnings (via Lifetime Trips)
    LEFT JOIN activity_earnings ae    ON lt.lifetime_trips_id = ae.lifetime_trips_fk

    -- Join to History Stats (via offer_id link in Lifetime Trips)
    LEFT JOIN HistoryStats hist       ON o.offer_id = hist.offer_id_fk;
"""

# 2. EXECUTE
try:
    with db_engine.connect() as connection:
        connection.execute(text(drop_sql))
        connection.execute(text(create_sql))
        print("✅ VIEW 'v_lifecycle_audit' CREATED SUCCESSFULLY.")

    # 3. VERIFY CONTENT
    print("\n--- AUDIT: THE OMNISCIENT VIEW ---")
    # Select specific columns to verify width and history
    cols = """
        offer_id,
        clean_timestamp,
        uber_original_fare,
        bank_net_earning,
        delta_end_sec,
        cum_net_earnings,
        rolling_avg_net_take_rate
    """
    audit_df = pd.read_sql(f"SELECT {cols} FROM v_lifecycle_audit WHERE bank_net_earning IS NOT NULL ORDER BY clean_timestamp DESC LIMIT 5", db_engine)

    if not audit_df.empty:
        display(audit_df)
        print("✅ AUDIT PASSED: History, Time, and Money are aligned.")

        # --- CHER AMI MESSAGE 3/3 ---
        print("\n" + "="*60)
        print("🕊️ CHER AMI (Mensaje 3 de 3):")
        print("   'El Ojo que Todo lo Ve está abierto.")
        print("    Desde el 2023 hasta hoy, cada centavo y cada segundo")
        print("    han sido contados. La Fase 0 ha concluido.'")
        print("="*60)

    else:
        print("⚠️ WARNING: View returned no rows in sample.")

except Exception as e:
    print(f"🔴 ERROR creating view: {e}")

--- FORGING BROCHE 3: THE OMNISCIENT LIFECYCLE VIEW ---
✅ VIEW 'v_lifecycle_audit' CREATED SUCCESSFULLY.

--- AUDIT: THE OMNISCIENT VIEW ---


,offer_id,clean_timestamp,uber_original_fare,bank_net_earning,delta_end_sec,cum_net_earnings,rolling_avg_net_take_rate
0,OF04761,2025-10-01 09:06:38,150.60,142.17,11.0,324320.46,0.649466
1,OF04760,2025-10-01 08:35:48,200.41,148.14,60.0,324178.29,0.649380
2,OF04751,2025-10-01 07:56:38,169.98,104.96,34.0,324030.15,0.649353
3,OF04739,2025-10-01 07:34:29,177.37,117.32,22.0,323925.19,0.649363
4,OF04738,2025-10-01 06:57:35,270.32,154.78,8.0,323807.87,0.649359


✅ AUDIT PASSED: History, Time, and Money are aligned.

🕊️ CHER AMI (Mensaje 3 de 3):
   'El Ojo que Todo lo Ve está abierto.
    Desde el 2023 hasta hoy, cada centavo y cada segundo
    han sido contados. La Fase 0 ha concluido.'


In [72]:
# ==============================================================================
# CELL [BROCHE 4/3]: THE ACCEPTED MISSIONS VIEW (v_lifecycle_audit_accepted)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Creates a filtered, high-value view derived from v_lifecycle_audit.
#          It isolates only the ~259 Accepted/Completed missions, showcasing
#          the full financial and temporal fidelity without the noise of rejections.
#          This is the table you show to the CFO.
# ==============================================================================

from sqlalchemy import text

print("--- FORGING BROCHE 4: THE WINNER'S CIRCLE ---")

drop_sql = "DROP VIEW IF EXISTS v_lifecycle_audit_accepted;"

create_sql = """
CREATE VIEW v_lifecycle_audit_accepted AS
SELECT *
FROM v_lifecycle_audit
WHERE trip_id_legacy IS NOT NULL -- Only rows linked to actual Trip Events
ORDER BY clean_timestamp DESC;
"""

# 2. EXECUTE
try:
    with db_engine.connect() as connection:
        connection.execute(text(drop_sql))
        connection.execute(text(create_sql))
        print("✅ VIEW 'v_lifecycle_audit_accepted' CREATED SUCCESSFULLY.")

    # 3. VERIFY CONTENT
    print("\n--- AUDIT: THE PURE GOLD ---")
    # Verification: Check count (should be ~250-260)
    count_df = pd.read_sql("SELECT COUNT(*) as count FROM v_lifecycle_audit_accepted", db_engine)
    print(f"   - Total Accepted Missions in View: {count_df.iloc[0]['count']}")

    # Sample the "Cherries on Top" (Cumulative Stats)
    cols = "trip_id_legacy, clean_timestamp, uber_original_fare, bank_net_earning, cum_net_earnings"
    sample_df = pd.read_sql(f"SELECT {cols} FROM v_lifecycle_audit_accepted LIMIT 5", db_engine)

    if not sample_df.empty:
        display(sample_df)
        print("✅ AUDIT PASSED: Focused view created.")

        # --- CHER AMI MESSAGE 4/3 (THE SECRET ENCORE) ---
        print("\n" + "="*60)
        print("🕊️ CHER AMI (Mensaje 4 de 3 - El Encore):")
        print("   'El ruido se ha ido. Solo queda la señal.")
        print("    Aquí están tus victorias, Arquitecto.")
        print("    La Fase 0 ha terminado. Ahora sí, vete a dormir.'")
        print("="*60)

    else:
        print("⚠️ WARNING: View returned no rows. Something is wrong with the filter.")

except Exception as e:
    print(f"🔴 ERROR creating view: {e}")

--- FORGING BROCHE 4: THE WINNER'S CIRCLE ---
✅ VIEW 'v_lifecycle_audit_accepted' CREATED SUCCESSFULLY.

--- AUDIT: THE PURE GOLD ---
   - Total Accepted Missions in View: 259


,trip_id_legacy,clean_timestamp,uber_original_fare,bank_net_earning,cum_net_earnings
0,251001-07,2025-10-01 09:06:38,150.60,142.17,324320.46
1,251001-06,2025-10-01 08:35:48,200.41,148.14,324178.29
2,251001-05,2025-10-01 07:56:38,169.98,104.96,324030.15
3,251001-04,2025-10-01 07:34:29,177.37,117.32,323925.19
4,251001-03,2025-10-01 06:57:35,270.32,154.78,323807.87


✅ AUDIT PASSED: Focused view created.

🕊️ CHER AMI (Mensaje 4 de 3 - El Encore):
   'El ruido se ha ido. Solo queda la señal.
    Aquí están tus victorias, Arquitecto.
    La Fase 0 ha terminado. Ahora sí, vete a dormir.'


In [73]:
# ==============================================================================
# CELL [FINAL]: THE "FIVE VIEWS" FINAL AUDIT
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This cell performs a non-destructive audit of the five critical
#          analytical views. It verifies their existence and displays a sample
#          of each to provide the Architect with a final, visual confirmation
#          of the database's state before concluding the ETL run.
# ==============================================================================

import pandas as pd
from sqlalchemy import create_engine, inspect

print("--- Initiating Final Audit: The State of the Five Views ---")

# ------------------------------------------------------------------------------
# 1. SETUP & VIEW LIST
# ------------------------------------------------------------------------------
# We use the inspector to get a list of all views in the database.
inspector = inspect(db_engine)
all_views = inspector.get_view_names()

# This is our "checklist" of views that MUST exist.
views_to_audit = [
    'v_mission_dossier',
    'v_broche_fks',
    'v_offers_human',
    'v_lifecycle_audit',
    'v_lifecycle_audit_accepted'
]

# ------------------------------------------------------------------------------
# 2. THE AUDIT LOOP
# ------------------------------------------------------------------------------
all_audits_passed = True

for view_name in views_to_audit:
    print("\n" + "-"*60)
    print(f"AUDITING VIEW: {view_name}")
    print("-" * 60)

    if view_name in all_views:
        print(f"✅ STATUS: View '{view_name}' exists in opus.db.")

        try:
            # Display the head of the view to show its content
            view_df = pd.read_sql(f"SELECT * FROM {view_name} LIMIT 5", db_engine)

            if not view_df.empty:
                print(f"   - Row count is > 0. Displaying sample (first 5 rows):")
                display(view_df)
            else:
                print(f"   - ⚠️ WARNING: View '{view_name}' exists but returned 0 rows.")
                all_audits_passed = False

        except Exception as e:
            print(f"   - 🔴 ERROR: View '{view_name}' exists but failed to query. Details: {e}")
            all_audits_passed = False
    else:
        print(f"   - 🔴 CRITICAL FAILURE: View '{view_name}' does not exist in the database.")
        all_audits_passed = False

# ------------------------------------------------------------------------------
# 3. THE FINAL VERDICT
# ------------------------------------------------------------------------------
print("\n" + "="*60)
if all_audits_passed:
    print("✅✅✅  AUDIT COMPLETE: ALL FIVE VIEWS ARE LIVE AND FUNCTIONAL. ✅✅✅")
    print("   The analytical layer is confirmed to be intact.")
    print("   The ETL Golden Run is a definitive success.")
else:
    print("🔴🔴🔴 AUDIT FAILED: One or more critical views are missing or broken.")
    print("   Review the errors above.")

print("="*60)

--- Initiating Final Audit: The State of the Five Views ---

------------------------------------------------------------
AUDITING VIEW: v_mission_dossier
------------------------------------------------------------
✅ STATUS: View 'v_mission_dossier' exists in opus.db.
   - Row count is > 0. Displaying sample (first 5 rows):


,offer_id,trip_id_legacy,trip_date,duration_to_pickup_sec,duration_waiting_sec,duration_trip_sec,total_engagement_duration_sec,upfront_fare,realized_fare,spread_percentage,eph_on_ride,eph_total_time
0,OF00003,250822-01,2025-08-22,None,None,1325.000015,2045.000012,136.53,114.49,0.838570,311.067166,201.547187
1,OF00012,250822-02,2025-08-22,None,None,1962.999974,2562.999979,162.00,135.72,0.837778,248.900666,190.632854
2,OF00032,250822-03,2025-08-22,None,None,676.999989,976.999971,64.31,57.53,0.894573,305.920241,211.983630
3,OF00034,250822-04,2025-08-22,None,None,1690.000007,2350.000007,146.00,130.6,0.894521,278.201182,200.068084
4,OF00035,250822-05,2025-08-22,None,None,541.000003,781.000029,60.00,49.93,0.832167,332.251385,230.151080



------------------------------------------------------------
AUDITING VIEW: v_broche_fks
------------------------------------------------------------
✅ STATUS: View 'v_broche_fks' exists in opus.db.
   - Row count is > 0. Displaying sample (first 5 rows):


,raw_ocr_id,hub_offer_id,hub_session_fk,hub_ocr_fk,feat_id,feat_offer_id_fk,event_id,event_offer_id_fk,lt_id,lt_offer_id_fk,ae_id,ae_offer_id_fk,ae_lt_fk
0,OCR01322,OF01322,SID0021,OCR01322,EF01322,OF01322,None,None,None,None,None,None,None
1,OCR01875,OF01875,SID0027,OCR01875,EF01875,OF01875,None,None,None,None,None,None,None
2,OCR00923,OF00923,SID0013,OCR00923,EF00923,OF00923,None,None,None,None,None,None,None
3,OCR02333,OF02333,SID0033,OCR02333,EF02333,OF02333,None,None,None,None,None,None,None
4,OCR03254,OF03254,SID0044,OCR03254,EF03254,OF03254,None,None,None,None,None,None,None



------------------------------------------------------------
AUDITING VIEW: v_offers_human
------------------------------------------------------------
✅ STATUS: View 'v_offers_human' exists in opus.db.
   - Row count is > 0. Displaying sample (first 5 rows):


...


------------------------------------------------------------
AUDITING VIEW: v_lifecycle_audit
------------------------------------------------------------
✅ STATUS: View 'v_lifecycle_audit' exists in opus.db.
   - Row count is > 0. Displaying sample (first 5 rows):


,offer_id,session_fk,trip_id_legacy,ocr_raw_time,clean_timestamp,day_of_week,time_of_day_block,ocr_product,internal_product,bank_product,...,delta_accept_sec,gts_t3_started,uber_pickup,delta_start_sec,gts_t4_completed,uber_dropoff,delta_end_sec,cum_uber_earnings,cum_net_earnings,rolling_avg_net_take_rate
0,OF01322,SID0021,None,14:01,2025-09-02 14:01:26,Tuesday,afternoon,UberX,uberx,None,...,None,None,None,None,None,None,None,None,None,None
1,OF01875,SID0027,None,1:45 PM,2025-09-05 13:45:47,Friday,afternoon,UberX Exclusivo,uberx,None,...,None,None,None,None,None,None,None,None,None,None
2,OF00923,SID0013,None,9:36,2025-08-28 21:36:34,Thursday,night,UberX,uberx,None,...,None,None,None,None,None,None,None,None,None,None
3,OF02333,SID0033,None,11:14,2025-09-14 11:14:15,Sunday,morning,UberX,uberx,None,...,None,None,None,None,None,None,None,None,None,None
4,OF03254,SID0044,None,10:19,2025-09-21 10:19:43,Sunday,morning,UberX,uberx,None,...,None,None,None,None,None,None,None,None,None,None



------------------------------------------------------------
AUDITING VIEW: v_lifecycle_audit_accepted
------------------------------------------------------------
✅ STATUS: View 'v_lifecycle_audit_accepted' exists in opus.db.
   - Row count is > 0. Displaying sample (first 5 rows):


,offer_id,session_fk,trip_id_legacy,ocr_raw_time,clean_timestamp,day_of_week,time_of_day_block,ocr_product,internal_product,bank_product,...,delta_accept_sec,gts_t3_started,uber_pickup,delta_start_sec,gts_t4_completed,uber_dropoff,delta_end_sec,cum_uber_earnings,cum_net_earnings,rolling_avg_net_take_rate
0,OF04761,SID0062,251001-07,9:06,2025-10-01 09:06:38,Wednesday,morning,Comfort,comfort,Comfort,...,464.0,2025-10-01 09:19:02,2025-10-01 09:19:17+00:00,-15.0,2025-10-01 09:52:14,2025-10-01 09:52:03+00:00,11.0,541191.93,324320.46,0.649466
1,OF04760,SID0062,251001-06,8:35,2025-10-01 08:35:48,Wednesday,morning,Business Comfort,business_comfort,Business Comfort,...,63.0,2025-10-01 08:44:11,2025-10-01 08:43:11+00:00,60.0,2025-10-01 09:13:29,2025-10-01 09:12:29+00:00,60.0,541041.33,324178.29,0.649380
2,OF04751,SID0062,251001-05,7:56,2025-10-01 07:56:38,Wednesday,morning,UberX,uberx,UberX,...,99.0,2025-10-01 08:04:43,2025-10-01 08:04:56+00:00,-13.0,2025-10-01 08:34:35,2025-10-01 08:34:01+00:00,34.0,540840.92,324030.15,0.649353
3,OF04739,SID0062,251001-04,7:34,2025-10-01 07:34:29,Wednesday,morning,Black Teens,black,Black,...,645.0,2025-10-01 07:47:21,2025-10-01 07:47:37+00:00,-16.0,2025-10-01 07:53:08,2025-10-01 07:52:46+00:00,22.0,540670.94,323925.19,0.649363
4,OF04738,SID0062,251001-03,6:57,2025-10-01 06:57:35,Wednesday,morning,Business Comfort,business_comfort,Business Comfort,...,-896.0,2025-10-01 07:11:27,2025-10-01 07:11:33+00:00,-6.0,2025-10-01 07:43:01,2025-10-01 07:42:53+00:00,8.0,540493.57,323807.87,0.649359



✅✅✅  AUDIT COMPLETE: ALL FIVE VIEWS ARE LIVE AND FUNCTIONAL. ✅✅✅
   The analytical layer is confirmed to be intact.
   The ETL Golden Run is a definitive success.


In [74]:
!ls -l "/content/drive/MyDrive/_Pienza/Assets/Database/opus.db"

-rw------- 1 root root 6078464 Jan  7 01:13 /content/drive/MyDrive/_Pienza/Assets/Database/opus.db


ohh lordy troubles so hard....

In [75]:
# ==============================================================================
# === PATCH CELL: THE "OFF-BY-ONE" HOTFIX (TD-007) - V2 (Target: trip_events) ===
# ==============================================================================
# Author: The Architect, _Pienza, ex_machina
# Purpose: Corrects the 4 known "off-by-one" data integrity errors where a
#          'trip_id_legacy' was incorrectly linked to the subsequent 'offer_id'.
#          This patch now correctly targets the 'trip_events' table.
# ==============================================================================

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import time

print("--- Initiating 'Off-by-One' Hotfix Patch (v2 - Targeting trip_events) ---")

# ------------------------------------------------------------------------------
# 0. SETUP & THE CORRECTION EDICT
# ------------------------------------------------------------------------------
try:
    # Use the canonical db_file_path defined in CELL 1
    if 'db_file_path' not in locals() and 'db_file_path' not in globals():
        project_root = '/content/drive/MyDrive/_Pienza'
        db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
        print("INFO: 'db_file_path' not found. Using default path.")
    db_engine = create_engine(f'sqlite:///{db_file_path}')
    print("✅ Database engine connection established.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

# The Architect's "Correction Edict" - The definitive list of corrections
# Format: {Incorrect_Offer_ID_Currently_Linked_in_TripEvents: Correct_Offer_ID_to_Link}
CORRECTION_EDICT_TO_FIND_AND_REPLACE_IN_TRIP_EVENTS = {
    'OF02650': 'OF02649',
    'OF03780': 'OF03779',
    'OF04649': 'OF04648',
    'OF04733': 'OF04732'
}
incorrect_offer_ids_in_tripevents = list(CORRECTION_EDICT_TO_FIND_AND_REPLACE_IN_TRIP_EVENTS.keys())


# --- We also need the original trip_id_legacy values to identify the rows ---
# We'll use a hardcoded list of the trip_id_legacy values that correspond to these mismatches.
# This ensures we are targeting the exact rows in trip_events.
TARGET_TRIP_ID_LEGACYS = [
    '250917-10', # Corresponds to OF02650 -> OF02649
    '250924-04', # Corresponds to OF03780 -> OF03779
    '250930-09', # Corresponds to OF04649 -> OF04648
    '251001-02'  # Corresponds to OF04733 -> OF04732
]


# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current state of 'trip_events' table from the database...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")

    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    print("Displaying data for the affected trip_id_legacy records:")
    display(trip_events_df[trip_events_df['trip_id_legacy'].isin(TARGET_TRIP_ID_LEGACYS)][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE SURGICAL STRIKE: Applying the Batch Patch in Memory (TARGETING trip_events)
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying {len(CORRECTION_EDICT_TO_FIND_AND_REPLACE_IN_TRIP_EVENTS)} manual corrections to 'trip_events' in memory...")

# Iterate through each correction
for incorrect_offer, correct_offer in CORRECTION_EDICT_TO_FIND_AND_REPLACE_IN_TRIP_EVENTS.items():
    # Identify the rows in trip_events that currently have the incorrect offer_id_fk
    mask = (trip_events_df['offer_id_fk'] == incorrect_offer)

    if trip_events_df.loc[mask].shape[0] > 0:
        # Apply the correction to offer_id_fk
        trip_events_df.loc[mask, 'offer_id_fk'] = correct_offer
        print(f"  - Corrected '{incorrect_offer}' to '{correct_offer}'.")
    else:
        print(f"  - WARNING: No records found with incorrect offer_id_fk '{incorrect_offer}'. May have been fixed already.")

print("✅ All corrections applied in memory.")


# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" in Memory (Essential for 'offers_session_fk')
# ------------------------------------------------------------------------------
# The 'offers_session_fk' column also needs to be updated based on the corrected 'offer_id_fk'.
# This requires querying the 'offers' table to get the correct session_fk.

print("\n⏳ Re-running Golden Propagation for 'offers_session_fk' and overall consistency...")

try:
    df_offers_lookup = pd.read_sql("SELECT offer_id, session_fk FROM offers", db_engine)
    offers_session_map = pd.Series(df_offers_lookup.session_fk.values, index=df_offers_lookup.offer_id).to_dict()

    # Apply the mapping to update offers_session_fk based on the now-correct offer_id_fk
    trip_events_df['offers_session_fk'] = trip_events_df['offer_id_fk'].map(offers_session_map)
    print("   - 'offers_session_fk' updated based on corrected 'offer_id_fk'.")

    # Re-run forward/backward fill for robustness (this handles NULLs from previously unlinked trips)
    trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
    trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
    trip_events_df['offers_session_fk'] = trip_events_df.groupby('trip_id_legacy')['offers_session_fk'].transform(lambda x: x.ffill().bfill())

    print("✅ Golden Propagation complete in memory.")

except Exception as e:
    print(f"🔴 ERROR during Golden Propagation: {e}")
    raise


# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing patched data back to opus.db (Forced Save Protocol)...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 2 seconds to allow sync...")
    time.sleep(2)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    # Check the specific trip_id_legacy values to ensure they are now linked to the CORRECT offer_id_fk
    verify_df = pd.read_sql_query(f"SELECT trip_id_legacy, event_types_id_fk, offer_id_fk, offers_session_fk FROM trip_events WHERE trip_id_legacy IN {tuple(TARGET_TRIP_ID_LEGACYS)}", verify_engine)
    verify_engine.dispose()

    print(f"Displaying corrected data for target trips:")
    display(verify_df.sort_values(by=['trip_id_legacy', 'event_types_id_fk']))

    # Assertions to ensure the corrections were applied
    all_passed = True
    for incorrect_offer, correct_offer in CORRECTION_EDICT_TO_FIND_AND_REPLACE_IN_TRIP_EVENTS.items():
        original_trip_id = [trip for trip, inc, cor in zip(TARGET_TRIP_ID_LEGACYS, incorrect_offer_ids_in_tripevents, CORRECTION_EDICT_TO_FIND_AND_REPLACE_IN_TRIP_EVENTS.values()) if cor == correct_offer][0] # Find the original trip_id_legacy for assertion

        # Check if the correct_offer is now linked
        is_correctly_linked = (verify_df[verify_df['trip_id_legacy'] == original_trip_id]['offer_id_fk'] == correct_offer).all()

        # Check if the session_fk is also correct
        expected_session_fk = pd.read_sql_query(f"SELECT session_fk FROM offers WHERE offer_id = '{correct_offer}'", create_engine(f'sqlite:///{db_path}')).iloc[0,0]
        is_session_fk_correct = (verify_df[verify_df['trip_id_legacy'] == original_trip_id]['offers_session_fk'] == expected_session_fk).all()

        if is_correctly_linked and is_session_fk_correct:
            print(f"✅ VERIFICATION SUCCESS: Trip '{original_trip_id}' is correctly linked to '{correct_offer}' with correct session.")
        else:
            print(f"🔴 VERIFICATION FAILED: Trip '{original_trip_id}' failed to link correctly to '{correct_offer}'.")
            all_passed = False

    assert all_passed, "One or more critical assertions failed after patching."

except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. Details: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating 'Off-by-One' Hotfix Patch (v2 - Targeting trip_events) ---
✅ Database engine connection established.

⏳ Reading current state of 'trip_events' table from the database...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---
Displaying data for the affected trip_id_legacy records:


,trip_id_legacy,event_types_id_fk,offer_id_fk
547,250917-10,1,OF02650
548,250917-10,2,OF02650
549,250917-10,3,OF02650
550,250917-10,4,OF02650
551,250917-10,5,OF02650
788,250924-04,2,OF03780
789,250924-04,3,OF03780
790,250924-04,4,OF03780
791,250924-04,5,OF03780
993,250930-09,2,OF04649



⏳ Applying 4 manual corrections to 'trip_events' in memory...
  - Corrected 'OF02650' to 'OF02649'.
  - Corrected 'OF03780' to 'OF03779'.
  - Corrected 'OF04649' to 'OF04648'.
  - Corrected 'OF04733' to 'OF04732'.
✅ All corrections applied in memory.

⏳ Re-running Golden Propagation for 'offers_session_fk' and overall consistency...
   - 'offers_session_fk' updated based on corrected 'offer_id_fk'.
✅ Golden Propagation complete in memory.

⏳ Writing patched data back to opus.db (Forced Save Protocol)...
   - Data written and connection disposed. Pausing for 2 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for target trips:


,trip_id_legacy,event_types_id_fk,offer_id_fk,offers_session_fk
0,250917-10,1,OF02649,SID0036
1,250917-10,2,OF02649,SID0036
2,250917-10,3,OF02649,SID0036
3,250917-10,4,OF02649,SID0036
4,250917-10,5,OF02649,SID0036
5,250924-04,2,OF03779,SID0049
6,250924-04,3,OF03779,SID0049
7,250924-04,4,OF03779,SID0049
8,250924-04,5,OF03779,SID0049
9,250930-09,2,OF04648,SID0061


✅ VERIFICATION SUCCESS: Trip '250917-10' is correctly linked to 'OF02649' with correct session.
✅ VERIFICATION SUCCESS: Trip '250924-04' is correctly linked to 'OF03779' with correct session.
✅ VERIFICATION SUCCESS: Trip '250930-09' is correctly linked to 'OF04648' with correct session.
✅ VERIFICATION SUCCESS: Trip '251001-02' is correctly linked to 'OF04732' with correct session.

--- SURGICAL STRIKE COMPLETE ---


In [76]:
# Example: Querying v_mission_dossier for a patched offer
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
db_engine = create_engine(f'sqlite:///{db_path}')

# Let's check OF02649, which was just corrected
check_offer_id = 'OF02649'

query = f"""
SELECT
    o.offer_id,
    o.offer_timestamp,
    k.trip_id_legacy,
    k.spread_percentage -- This KPI should now be correctly linked
FROM
    offers AS o
LEFT JOIN
    v_mission_dossier k ON o.offer_id = k.offer_id
WHERE
    o.offer_id = '{check_offer_id}';
"""

print(f"\n--- Verifying propagation for offer_id: {check_offer_id} in v_mission_dossier ---")
try:
    with db_engine.connect() as conn:
        df_check = pd.read_sql_query(query, conn)
    display(df_check)
    if not df_check.empty:
        print(f"✅ Propagation confirmed: Data for {check_offer_id} now correctly visible in v_mission_dossier.")
    else:
        print("⚠️ WARNING: Offer not found in v_mission_dossier. Review linkages.")

except Exception as e:
    print(f"❌ ERROR during view verification: {e}")


--- Verifying propagation for offer_id: OF02649 in v_mission_dossier ---


,offer_id,offer_timestamp,trip_id_legacy,spread_percentage
0,OF02649,2025-09-17 16:10:07,250917-10,0.905407


✅ Propagation confirmed: Data for OF02649 now correctly visible in v_mission_dossier.


A ver si no falla despues de aqui.... changuitos.

In [77]:
import sqlite3
import os
from sqlalchemy import create_engine, text
import pandas as pd

# --- Configuración de Entorno (Asumiendo que ya se ejecutó la primera celda) ---
try:
    # Intentamos usar la conexión existente o definimos la ruta canónica
    if 'db_file_path' not in locals():
        project_root = '/content/drive/My Drive/_Pienza'
        db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')

    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Environment check passed. Engine established.")
except Exception as e:
    print(f"🛑 CRITICAL ERROR: Database engine setup failed. Halting. Details: {e}")
    assert False, "Environment not ready for DB operations."


# --- SQL DDL: Defining the Schema for the Silver Palette (V3.0) ---
DDL_SQL_V3 = """
CREATE TABLE silver_palette (
    -- Clave Primaria / Anchor
    offer_id TEXT NOT NULL PRIMARY KEY,

    -- DROP-OFF CLUSTERING RESULTS (Analizado y Confirmado)
    dropoff_polygon_id INTEGER,
    dropoff_polygon_name TEXT,
    dropoff_h3_hex_id TEXT,
    dropoff_hdbscan_id INTEGER,
    dropoff_hdbscan_name TEXT,

    -- TRAFFIC INDEX FEATURES (Analizado y Confirmado)
    realized_traffic_index REAL,
    historical_rolling_avg_traffic_index REAL,
    traffic_volatility_index_ML REAL,
    traffic_volatility_index_EDA REAL,

    -- PICKUP CLUSTERING (FUTURE PLACEHOLDERS - Inicialmente NULL)
    pickup_polygon_id INTEGER,
    pickup_polygon_name TEXT,
    pickup_h3_hex_id TEXT,
    pickup_hdbscan_id INTEGER,
    pickup_hdbscan_name TEXT
);
"""

print("\n--- Executing DDL: Creating 'silver_palette' table (v3.0) ---")

try:
    with db_engine.connect() as conn:
        # 1. Clean Slate
        conn.execute(text("DROP TABLE IF EXISTS silver_palette;"))

        # 2. Execute the DDL
        conn.execute(text(DDL_SQL_V3))

        conn.commit()
    print("✅ SUCCESS: Table 'silver_palette' created successfully with complete schema.")

    # --- VERIFICATION STEP ---
    print("\n--- Verifying Schema Structure ---")
    with sqlite3.connect(db_file_path) as conn:
        schema_check = pd.read_sql("PRAGMA table_info(silver_palette);", conn)
        display(schema_check)

except Exception as e:
    print(f"🔴 CRITICAL ERROR during Schema Creation: {e}")
    assert False, "Phase 1 Failed: Could not create the Silver Palette table."

print("\n--- PHASE 1 COMPLETE: COMPLETE SILVER PALETTE SCHEMA DEFINED ---")

✅ Environment check passed. Engine established.

--- Executing DDL: Creating 'silver_palette' table (v3.0) ---
✅ SUCCESS: Table 'silver_palette' created successfully with complete schema.

--- Verifying Schema Structure ---


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,1,None,1
1,1,dropoff_polygon_id,INTEGER,0,None,0
2,2,dropoff_polygon_name,TEXT,0,None,0
3,3,dropoff_h3_hex_id,TEXT,0,None,0
4,4,dropoff_hdbscan_id,INTEGER,0,None,0
5,5,dropoff_hdbscan_name,TEXT,0,None,0
6,6,realized_traffic_index,REAL,0,None,0
7,7,historical_rolling_avg_traffic_index,REAL,0,None,0
8,8,traffic_volatility_index_ML,REAL,0,None,0
9,9,traffic_volatility_index_EDA,REAL,0,None,0



--- PHASE 1 COMPLETE: COMPLETE SILVER PALETTE SCHEMA DEFINED ---


In [78]:
import sqlite3
import os
from sqlalchemy import create_engine, text
import pandas as pd

# --- Configuración de Entorno ---
try:
    if 'db_file_path' not in locals():
        project_root = '/content/drive/My Drive/_Pienza'
        db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')

    db_engine = create_engine(f'sqlite:///{db_file_path}')
    print("✅ Environment check passed. Engine established.")
except NameError:
    print("🛑 CRITICAL ERROR: Database path not found.")
    assert False, "Environment not ready."

# --- SQL Sentences (DIVIDIDAS) ---

# Paso 0: Limpieza
SQL_CLEAN_ZOMBIE = "DROP TABLE IF EXISTS silver_palette_new;"

# Paso 1: Create
SQL_CREATE_TEMP = """
CREATE TABLE silver_palette_new (
    offer_id TEXT NOT NULL PRIMARY KEY,
    dropoff_polygon_id INTEGER,
    dropoff_polygon_name TEXT,
    dropoff_h3_hex_id TEXT,
    dropoff_hdbscan_id INTEGER,
    dropoff_hdbscan_name TEXT,
    realized_traffic_index REAL,
    historical_rolling_avg_traffic_index REAL,
    traffic_volatility_index_ML REAL,
    traffic_volatility_index_EDA REAL,
    pickup_polygon_id INTEGER,
    pickup_polygon_name TEXT,
    pickup_h3_hex_id TEXT,
    pickup_hdbscan_id INTEGER,
    pickup_hdbscan_name TEXT,

    FOREIGN KEY (offer_id) REFERENCES offers(offer_id)
);
"""

# Paso 2: Copy
SQL_COPY_DATA = """
INSERT INTO silver_palette_new (
    offer_id, dropoff_polygon_id, dropoff_polygon_name, dropoff_h3_hex_id,
    dropoff_hdbscan_id, dropoff_hdbscan_name, realized_traffic_index,
    historical_rolling_avg_traffic_index, traffic_volatility_index_ML,
    traffic_volatility_index_EDA, pickup_polygon_id, pickup_polygon_name,
    pickup_h3_hex_id, pickup_hdbscan_id, pickup_hdbscan_name
)
SELECT
    offer_id, dropoff_polygon_id, dropoff_polygon_name, dropoff_h3_hex_id,
    dropoff_hdbscan_id, dropoff_hdbscan_name, realized_traffic_index,
    historical_rolling_avg_traffic_index, traffic_volatility_index_ML,
    traffic_volatility_index_EDA, pickup_polygon_id, pickup_polygon_name,
    pickup_h3_hex_id, pickup_hdbscan_id, pickup_hdbscan_name
FROM silver_palette;
"""

# Paso 3: Finalize (SEPARADO EN DOS COMANDOS)
SQL_DROP_OLD = "DROP TABLE IF EXISTS silver_palette;"
SQL_RENAME_NEW = "ALTER TABLE silver_palette_new RENAME TO silver_palette;"

print("\n--- Executing Atomic Replacement Protocol ---")

try:
    with db_engine.connect() as conn:
        # [PASO 0] Limpieza
        print("   [0/4] Cleaning up potential artifacts...")
        conn.execute(text(SQL_CLEAN_ZOMBIE))

        # [PASO 1] Crear
        conn.execute(text(SQL_CREATE_TEMP))
        print("   [1/4] Temp table 'silver_palette_new' created with FK.")

        # [PASO 2] Copiar
        try:
            conn.execute(text(SQL_COPY_DATA))
            print("   [2/4] Data copied from old table.")
        except Exception as e:
            print(f"   [Warning] Copy step skipped/failed: {e}")

        # [PASO 3] Finalizar (EJECUTADO SECUENCIALMENTE)
        conn.execute(text(SQL_DROP_OLD))
        print("   [3/4] Old table dropped.")

        conn.execute(text(SQL_RENAME_NEW))
        print("   [4/4] New table renamed. Swap complete.")

        conn.commit()

    print("✅ SUCCESS: Table 'silver_palette' updated and structure established with FK.")

    # --- VERIFICATION ---
    print("\n--- Verifying Final Schema Structure ---")
    with db_engine.connect() as conn:
        schema_check = pd.read_sql("PRAGMA table_info(silver_palette);", conn)
        display(schema_check)

except Exception as e:
    print(f"🔴 CRITICAL ERROR during multi-step execution: {e}")
    raise e

✅ Environment check passed. Engine established.

--- Executing Atomic Replacement Protocol ---
   [0/4] Cleaning up potential artifacts...
   [1/4] Temp table 'silver_palette_new' created with FK.
   [2/4] Data copied from old table.
   [3/4] Old table dropped.
   [4/4] New table renamed. Swap complete.
✅ SUCCESS: Table 'silver_palette' updated and structure established with FK.

--- Verifying Final Schema Structure ---


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,1,None,1
1,1,dropoff_polygon_id,INTEGER,0,None,0
2,2,dropoff_polygon_name,TEXT,0,None,0
3,3,dropoff_h3_hex_id,TEXT,0,None,0
4,4,dropoff_hdbscan_id,INTEGER,0,None,0
5,5,dropoff_hdbscan_name,TEXT,0,None,0
6,6,realized_traffic_index,REAL,0,None,0
7,7,historical_rolling_avg_traffic_index,REAL,0,None,0
8,8,traffic_volatility_index_ML,REAL,0,None,0
9,9,traffic_volatility_index_EDA,REAL,0,None,0


In [79]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# --- Configuración ---
try:
    if 'db_file_path' not in locals():
        project_root = '/content/drive/My Drive/_Pienza'
        db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    db_engine = create_engine(f'sqlite:///{db_file_path}')
    print("✅ Environment check passed. Engine established.")
except NameError:
    print("🛑 CRITICAL ERROR: Database path not found. Please run initial setup.")
    assert False, "Environment not ready."

# --- Protocolo de Verificación de Enlace Único ---
print("\n--- Auditing EF table structure for offer_id link integrity ---")

try:
    with db_engine.connect() as conn:
        # 1. AUDITAR: ¿Existe 'offer_id_fk' en EF?
        # Leemos el esquema de la tabla engineered_features
        schema_ef = pd.read_sql("PRAGMA table_info(engineered_features);", conn)

        # Verificamos si la columna FK existe
        if 'offer_id_fk' not in schema_ef['name'].values:
            print("⚠️ WARNING: 'offer_id_fk' missing in engineered_features. Skipping direct EF-SP link in favor of joining all through 'offers' in Phase 4.")
            # Si no existe, nos apegamos a unir todo a través de 'offers' en la vista, que es lo más seguro.

        else:
            print("✅ 'offer_id_fk' exists in engineered_features. Structure is ready for direct linking.")

        # 2. AUDITAR: ¿Existe un link 'offer_id' en 'silver_palette'?
        # (Ya existe por diseño en Fase 1, así que solo verificamos la tabla)
        schema_sp = pd.read_sql("PRAGMA table_info(silver_palette);", conn)
        if 'offer_id' in schema_sp['name'].values:
            print("✅ 'offer_id' exists as PK in silver_palette.")
        else:
             print("⚠️ WARNING: 'offer_id' MISSING in silver_palette. Check Phase 1.")

    print("\n--- VERDICTO: El JOIN directo EF <-> SP se simplifica en la vista final ---")
    print("Confirmado: Ambas tablas se unirán a la vista maestra a través de 'offer_id' (o su FK), manteniendo la integridad y evitando la circularidad.")

except Exception as e:
    print(f"❌ ERROR during audit: {e}")
    assert False, "Schema audit failed."

print("\n--- PHASE 2.5 COMPLETE: FEATURE INTEGRITY ESTABLISHED ---")

✅ Environment check passed. Engine established.

--- Auditing EF table structure for offer_id link integrity ---
✅ 'offer_id_fk' exists in engineered_features. Structure is ready for direct linking.
✅ 'offer_id' exists as PK in silver_palette.

--- VERDICTO: El JOIN directo EF <-> SP se simplifica en la vista final ---
Confirmado: Ambas tablas se unirán a la vista maestra a través de 'offer_id' (o su FK), manteniendo la integridad y evitando la circularidad.

--- PHASE 2.5 COMPLETE: FEATURE INTEGRITY ESTABLISHED ---


In [80]:
import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default
from sqlalchemy import create_engine

# --- CONFIGURACIÓN DE ENTORNO ---
try:
    # Autenticación GSpread
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # DB Connection
    if 'db_file_path' not in locals():
        project_root = '/content/drive/My Drive/_Pienza'
        db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    db_engine = create_engine(f'sqlite:///{db_file_path}')
    print("✅ Environment & Auth check passed.")
except Exception as e:
    print(f"🛑 SETUP ERROR: {e}")
    assert False, "Setup failed."

# --- DEFINICIÓN DE FUENTES ---
# Fuente A: Clustering
SRC_CLUSTERING_SHEET = "Master_Clustering_Features"
SRC_CLUSTERING_TAB = "Sheet1"

# Fuente B: Tráfico (Desde la pestaña específica que indicaste)
SRC_TRAFFIC_SHEET = "opus_1_phase_1_engineered_features"
SRC_TRAFFIC_TAB = "silver_palette"

TARGET_TABLE = "silver_palette"

print(f"\n--- PHASE 3: ETL FOR '{TARGET_TABLE}' ---")

try:
   # ---------------------------------------------------------
    # 1. EXTRACT: Clustering Data (CORRECTED & DEFENSIVE)
    # ---------------------------------------------------------
    print(f"1️⃣ Extracting Clustering Data from '{SRC_CLUSTERING_SHEET}'...")
    wb_cluster = gc.open(SRC_CLUSTERING_SHEET)
    ws_cluster = wb_cluster.worksheet(SRC_CLUSTERING_TAB)
    df_cluster = pd.DataFrame(ws_cluster.get_all_records())

    # Normalización básica de nombres
    df_cluster.columns = df_cluster.columns.str.lower().str.strip().str.replace(' ', '_')

    # --- [FIX] SELECCIÓN DEFENSIVA DE COLUMNAS ---
    # Definimos explícitamente qué columnas permitimos pasar.
    # Esto elimina cualquier basura (lat, lon, address) que venga en el GSheet.
    clustering_allowed_cols = [
        'offer_id',
        'dropoff_polygon_id',
        'dropoff_polygon_name',
        'dropoff_h3_hex_id',
        'dropoff_hdbscan_id',
        'dropoff_hdbscan_name'
    ]

    # Filtrar el DataFrame para quedarse SOLO con lo permitido
    # Usamos intersection para no fallar si falta alguna columna opcional,
    # pero asegurando que no pase nada extra.
    cols_to_keep = [c for c in clustering_allowed_cols if c in df_cluster.columns]
    df_cluster = df_cluster[cols_to_keep]

    print(f"   -> Loaded {len(df_cluster)} clustering records.")
    print(f"   -> Columns kept: {list(df_cluster.columns)}")
    print(f"   -> Loaded {len(df_cluster)} clustering records.")

    # ---------------------------------------------------------
    # 2. EXTRACT: Traffic Data
    # ---------------------------------------------------------
    print(f"2️⃣ Extracting Traffic Data from '{SRC_TRAFFIC_SHEET}' (Tab: {SRC_TRAFFIC_TAB})...")
    wb_traffic = gc.open(SRC_TRAFFIC_SHEET)
    ws_traffic = wb_traffic.worksheet(SRC_TRAFFIC_TAB)
    df_traffic = pd.DataFrame(ws_traffic.get_all_records())

    # Normalización de columnas
    df_traffic.columns = df_traffic.columns.str.lower().str.strip().str.replace(' ', '_')

    # Selección defensiva de columnas (solo las que necesitamos de esta hoja)
    traffic_cols = [
        'offer_id',
        'realized_traffic_index',
        'historical_rolling_avg_traffic_index',
        'traffic_volatility_index_ml',
        'traffic_volatility_index_eda'
    ]
    # Filtramos solo si existen, para evitar errores si hay typos
    existing_traffic_cols = [c for c in traffic_cols if c in df_traffic.columns]
    df_traffic = df_traffic[existing_traffic_cols]

    print(f"   -> Loaded {len(df_traffic)} traffic records.")

    # ---------------------------------------------------------
    # 3. TRANSFORM: Merge Sources
    # ---------------------------------------------------------
    print("3️⃣ Merging sources on 'offer_id'...")

    # Usamos outer join para asegurar que no perdemos datos si alguna fuente tiene más filas que la otra
    # (aunque idealmente deberían ser iguales 1:1)
    df_merged = pd.merge(df_cluster, df_traffic, on='offer_id', how='outer')

    # Limpieza de vacíos vacíos de GSheet a None (NULL)
    df_merged.replace('', None, inplace=True)

    print(f"   -> Merged Dataset: {len(df_merged)} rows.")

    # ---------------------------------------------------------
    # 4. LOAD: Populate Database
    # ---------------------------------------------------------
    print(f"4️⃣ Loading into SQLite table '{TARGET_TABLE}'...")

    with db_engine.connect() as conn:
        df_merged.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)

    print(f"✅ SUCCESS: Phase 3 Complete. '{TARGET_TABLE}' is populated.")

except Exception as e:
    print(f"🔴 ETL FAILED: {e}")
    assert False, "Phase 3 Execution Halted."

# --- VERIFICACIÓN ---
print("\n--- DATA SAMPLE ---")
with db_engine.connect() as conn:
    sample = pd.read_sql(f"SELECT * FROM {TARGET_TABLE} LIMIT 5", conn)
    display(sample)

✅ Environment & Auth check passed.

--- PHASE 3: ETL FOR 'silver_palette' ---
1️⃣ Extracting Clustering Data from 'Master_Clustering_Features'...
   -> Loaded 4765 clustering records.
   -> Columns kept: ['offer_id', 'dropoff_polygon_id', 'dropoff_polygon_name', 'dropoff_h3_hex_id', 'dropoff_hdbscan_id', 'dropoff_hdbscan_name']
   -> Loaded 4765 clustering records.
2️⃣ Extracting Traffic Data from 'opus_1_phase_1_engineered_features' (Tab: silver_palette)...
   -> Loaded 4765 traffic records.
3️⃣ Merging sources on 'offer_id'...
   -> Merged Dataset: 4765 rows.
4️⃣ Loading into SQLite table 'silver_palette'...
✅ SUCCESS: Phase 3 Complete. 'silver_palette' is populated.

--- DATA SAMPLE ---


,offer_id,dropoff_polygon_id,dropoff_polygon_name,dropoff_h3_hex_id,dropoff_hdbscan_id,dropoff_hdbscan_name,realized_traffic_index,historical_rolling_avg_traffic_index,traffic_volatility_index_ml,traffic_volatility_index_eda
0,OF00001,-1,unassigned,89499516bafffff,-1,unassigned,None,None,None,None
1,OF00002,-1,unassigned,894995b1dd7ffff,-1,unassigned,None,None,None,None
2,OF00003,-1,unassigned,None,-2,missing_coordinates,None,None,None,None
3,OF00004,15,lomas_fc_cuernavaca,894995bac03ffff,41,el_semaforo_de_palmas,None,None,None,None
4,OF00005,-1,unassigned,894995bab23ffff,-1,unassigned,None,None,None,None


In [81]:
import pandas as pd
from sqlalchemy import text

print("\n--- PHASE 4: DYNAMIC VIEW CREATION 'v_ML_Supervised' ---")

try:
    with db_engine.connect() as conn:
        # 1. Obtener columnas de cada tabla
        cols_o = pd.read_sql("PRAGMA table_info(offers)", conn)['name'].tolist()
        cols_ef = pd.read_sql("PRAGMA table_info(engineered_features)", conn)['name'].tolist()
        cols_sp = pd.read_sql("PRAGMA table_info(silver_palette)", conn)['name'].tolist()

        # 2. Construir la lista de selección
        # Prefix 'o.' for offers
        select_parts = [f"o.{c}" for c in cols_o]

        # Filter EF columns (Remove join key)
        ef_exclude = ['offer_id_fk']
        select_parts += [f"ef.{c}" for c in cols_ef if c not in ef_exclude]

        # Filter SP columns (Remove join key)
        sp_exclude = ['offer_id']
        select_parts += [f"sp.{c}" for c in cols_sp if c not in sp_exclude]

        # Join into string
        select_clause = ",\n    ".join(select_parts)

        # 3. Construct SQL
        VIEW_SQL_DYNAMIC = f"""
        CREATE VIEW v_ML_Supervised AS
        SELECT
            {select_clause}
        FROM
            offers o
        LEFT JOIN engineered_features ef ON o.offer_id = ef.offer_id_fk
        LEFT JOIN silver_palette sp ON o.offer_id = sp.offer_id;
        """

        # 4. Execute
        conn.execute(text("DROP VIEW IF EXISTS v_ML_Supervised;"))
        conn.execute(text(VIEW_SQL_DYNAMIC))

    print("✅ SUCCESS: View 'v_ML_Supervised' created with ALL unique columns.")

    # --- VERIFICATION: COLUMN AUDIT ---
    print("\n--- Auditing Final Schema: Column List ---")
    with db_engine.connect() as conn:
        # Get list of columns in the new view
        view_info = pd.read_sql("PRAGMA table_info(v_ML_Supervised)", conn)

        # Display as a clean list
        all_columns = view_info['name'].tolist()
        print(f"Total Columns: {len(all_columns)}")
        print("-" * 30)
        for i, col in enumerate(all_columns):
            print(f"{i}: {col}")

except Exception as e:
    print(f"🔴 ERROR: {e}")
    assert False, "Phase 4 Failed."


--- PHASE 4: DYNAMIC VIEW CREATION 'v_ML_Supervised' ---
✅ SUCCESS: View 'v_ML_Supervised' created with ALL unique columns.

--- Auditing Final Schema: Column List ---
Total Columns: 103
------------------------------
0: offer_id
1: session_fk
2: ocr_fk
3: image_content_hash
4: offer_timestamp
5: upfront_fare
6: time_to_pickup_sec
7: dist_to_pickup_km
8: est_trip_time_sec
9: est_trip_dist_km
10: pickup_address
11: dropoff_address
12: pickup_lat
13: pickup_lon
14: dropoff_lat
15: dropoff_lon
16: is_surge
17: surge_amount
18: is_turbo_plus
19: turbo_plus_amount
20: is_reservation
21: reservation_amount
22: is_priority
23: priority_amount
24: is_exclusive
25: is_vip
26: is_identity_verified
27: is_long_trip
28: is_multiple_destinations
29: is_teens
30: rider_star_rating
31: rider_trip_count
32: time_in_session_sec
33: session_progress_ratio
34: inferred_agent_lat
35: inferred_agent_lon
36: inferred_agent_bearing
37: inferred_agent_speed_mps
38: is_imputed
39: special_note_raw
40: comment

In [82]:
import pandas as pd
from sqlalchemy import text

print("\n--- PHASE 4 (RE-REVISADA): DYNAMIC VIEW WITH HEURISTIC FLAGS ---")

try:
    with db_engine.connect() as conn:
        # 1. Obtener columnas (igual que antes)
        cols_o = pd.read_sql("PRAGMA table_info(offers)", conn)['name'].tolist()
        cols_ef = pd.read_sql("PRAGMA table_info(engineered_features)", conn)['name'].tolist()
        cols_sp = pd.read_sql("PRAGMA table_info(silver_palette)", conn)['name'].tolist()

        select_parts = [f"o.{c}" for c in cols_o]

        ef_exclude = ['offer_id_fk']
        select_parts += [f"ef.{c}" for c in cols_ef if c not in ef_exclude]

        sp_exclude = ['offer_id']
        select_parts += [f"sp.{c}" for c in cols_sp if c not in sp_exclude]

        # --- [NUEVO] Traer la Heuristic Flag de la tabla Many-to-Many ---
        # Como dices que en la práctica es 1:1, usamos esta lógica para traer el texto de la flag
        select_parts.append("hf.heuristic_flag_description AS heuristic_flag_context")

        select_clause = ",\n    ".join(select_parts)

        # 3. Construir SQL con el JOIN extra a la tabla que tanto odiaste
        VIEW_SQL_DYNAMIC = f"""
        CREATE VIEW v_ML_Supervised AS
        SELECT
            {select_clause}
        FROM
            offers o
        LEFT JOIN engineered_features ef ON o.offer_id = ef.offer_id_fk
        LEFT JOIN silver_palette sp       ON o.offer_id = sp.offer_id
        -- El puente a las banderas:
        LEFT JOIN heuristic_flag_offers hfo ON o.offer_id = hfo.offers_offer_id
        LEFT JOIN heuristic_flag hf         ON hfo.heuristic_flag_heuristic_flag_id = hf.heuristic_flag_id;
        """

        conn.execute(text("DROP VIEW IF EXISTS v_ML_Supervised;"))
        conn.execute(text(VIEW_SQL_DYNAMIC))

    print("✅ SUCCESS: 'v_ML_Supervised' ahora incluye 'heuristic_flag_context'.")

except Exception as e:
    print(f"🔴 ERROR: {e}")


--- PHASE 4 (RE-REVISADA): DYNAMIC VIEW WITH HEURISTIC FLAGS ---
✅ SUCCESS: 'v_ML_Supervised' ahora incluye 'heuristic_flag_context'.


In [83]:
import os
import sqlite3
import time

# 1. Force SQLite to merge the WAL file into the main DB
print("🔒 Checkpointing SQLite WAL...")
try:
    with db_engine.connect() as conn:
        # 'TRUNCATE' forces the WAL file to be fully written to the main DB and then reset
        conn.execute(text("PRAGMA wal_checkpoint(TRUNCATE);"))
        conn.execute(text("VACUUM;")) # Optional: Optimizes the file size
        conn.commit()
    print("✅ SQLite Checkpoint Complete.")
except Exception as e:
    print(f"⚠️ Warning during checkpoint: {e}")

# 2. Close the Engine explicitly
db_engine.dispose()
print("🔌 DB Connection Closed.")

# 3. Force the OS to write the file buffer to Google Drive
print("💾 Forcing OS Flush to Drive...")
db_path_str = str(db_engine.url).replace('sqlite:///', '') # Extract path
try:
    with open(db_path_str, 'r+') as f:
        os.fsync(f.fileno())
    print("✅ OS Buffer Flushed.")
except Exception as e:
    print(f"⚠️ Could not fsync: {e}")

# 4. The "Sleep" Hack (Give Drive 10 seconds to catch up)
print("⏳ Waiting 10s for Google Drive Sync...")
time.sleep(10)
print("🚀 ETL PIPELINE FINALIZED.")

🔒 Checkpointing SQLite WAL...
✅ SQLite Checkpoint Complete.
🔌 DB Connection Closed.
💾 Forcing OS Flush to Drive...
✅ OS Buffer Flushed.
⏳ Waiting 10s for Google Drive Sync...
🚀 ETL PIPELINE FINALIZED.
